# AS3179 · Nhóm 7 · Kiểm chứng ARMA và ARIMA

Đặt notebook cạnh năm CSV gốc (hoặc cạnh thư mục `data/raw/` của bài nộp).
Chọn kernel Python 3.12 rồi **Restart Kernel and Run All Cells**. Nếu tạo môi trường mới,
cài các phiên bản ở Phụ lục B bằng pip trước khi chạy. Dữ liệu xử lý, bảng và hình
sẽ được tạo lại trong thư mục đang chạy. Không cần tải dữ liệu hay cài gói mã của nhóm.

Đọc theo thứ tự **hàm và dữ liệu → khám phá → Bài 11 (§11.1–11.14) → Bài 12 (§12.1–12.14)**.
Mỗi phép kiểm chứng có mã, kết quả và diễn giải. Hạt giống mô phỏng là 3179.

In [ ]:
from types import ModuleType
from pathlib import Path
import sys
from IPython.core.magic import register_cell_magic

# Mỗi ô %%hocphan chứa nguyên mã của một mô-đun. Các import nội bộ bên dưới
# trỏ tới các ô này trong bộ nhớ của kernel, kể cả trên máy chưa cài dự án.
package = ModuleType("as3179ts")
package.__path__ = []
sys.modules[package.__name__] = package

@register_cell_magic
def hocphan(name, source):
    fullname = f"as3179ts.{name}"
    module = ModuleType(fullname)
    module.__package__ = package.__name__
    sys.modules[fullname] = module
    setattr(package, name, module)
    exec(compile(source, f"<notebook: {name}>", "exec"), module.__dict__)

In [ ]:
%%hocphan config
"""Hằng số và đường dẫn tập trung. Mọi đường dẫn tính tương đối từ thư mục gốc dự án."""
from pathlib import Path

SEED = 3179  # giống bài giảng; mọi mô phỏng dùng numpy.random.default_rng(SEED)

ROOT = Path.cwd()
DATA_DIR = ROOT / "data"
DATA_RAW = DATA_DIR / "raw" if (DATA_DIR / "raw" / "Nile.csv").is_file() else ROOT
DATA_PROCESSED = DATA_DIR / "processed"
FIG_DIR = ROOT / "outputs" / "figures"
TABLE_DIR = ROOT / "outputs" / "tables"
LECTURE_DIR = ROOT / "docs" / "lectures"

AP_TRAIN_N = 120      # AirPassengers: 1949-01..1958-12
AP_TEST_N = 24        # 1959-01..1960-12
DP_TRAIN_N = 1500     # DAX: 1500 ngày giao dịch đầu
DP_TEST_N = 360
DAX_START_TIME = 1991 + 129 / 260  # start = c(1991, 130), frequency = 260 (như time() của R)
DAX_FREQ = 260

In [ ]:
%%hocphan data
"""Đọc dữ liệu: bản gốc (data/raw) và bản đã làm sạch (data/processed)."""
import pandas as pd

from .config import DATA_PROCESSED, DATA_RAW

PROCESSED_FILES = {
    "HU": "HU_lakehuron.csv",
    "LY": "LY_lynx_log10.csv",
    "DR": "DR_dax_returns.csv",
    "NI": "NI_nile.csv",
    "AP": "AP_airpassengers.csv",
    "DP": "DP_dax_logprice.csv",
}


def load_raw(name: str) -> pd.DataFrame:
    """Đọc CSV gốc, ví dụ load_raw('LakeHuron')."""
    return pd.read_csv(DATA_RAW / f"{name}.csv")


def load_processed(code: str) -> pd.DataFrame:
    """Đọc bộ đã làm sạch theo mã ∈ {HU, LY, DR, NI, AP, DP}."""
    return pd.read_csv(DATA_PROCESSED / PROCESSED_FILES[code])

In [ ]:
%%hocphan acf_tools
"""ACF, PACF, Ljung–Box cài đặt tay theo Bài 11 (chỉ dùng numpy/scipy, không gọi statsmodels)."""
import numpy as np
from scipy import stats
from scipy.linalg import toeplitz


def sample_autocov(y, nlags: int, denom: str = "n") -> np.ndarray:
    """γ̂(h), h = 0..nlags (Bài 11, Định nghĩa 3).

    denom="n": mẫu số n (mặc định, đảm bảo Γ̂ₙ xác định không âm);
    denom="n-h": mẫu số n − h (có thể mất tính xác định dương, Bài tập 4(1)).
    """
    y = np.asarray(y, dtype=float)
    n = len(y)
    d = y - y.mean()
    s = np.array([d[h:] @ d[: n - h] for h in range(nlags + 1)])
    return s / n if denom == "n" else s / (n - np.arange(nlags + 1))


def sample_acf(y, nlags: int, denom: str = "n") -> np.ndarray:
    """ρ̂(h) = γ̂(h)/γ̂(0), h = 0..nlags (Bài 11, Định nghĩa 3)."""
    g = sample_autocov(y, nlags, denom)
    return g / g[0]


def gamma_matrix(y, size: int, denom: str = "n") -> np.ndarray:
    """Ma trận Toeplitz Γ̂ = [γ̂(i − j)] cỡ size × size (Bài 11, Mệnh đề 1(c))."""
    return toeplitz(sample_autocov(y, size - 1, denom))


def bartlett_band(n: int, rho=None, q: int = 0, z: float = 1.96) -> float:
    """Nửa độ rộng dải 95% của ρ̂(h), h > q.

    q = 0: dải iid ±z/√n (Định lý 2(a)).
    q ≥ 1: Bartlett cho MA(q), ±z·√{(1 + 2Σ_{k≤q} ρ(k)²)/n} (Định lý 2(c)); rho[k] = ρ(k), rho[0] = 1.
    """
    if q == 0:
        return z / np.sqrt(n)
    return z * np.sqrt((1 + 2 * np.sum(np.asarray(rho)[1 : q + 1] ** 2)) / n)


def pacf_durbin_levinson(rho, nlags: int) -> np.ndarray:
    """PACF α(h) = φ_hh từ ACF, h = 0..nlags, bằng đệ quy Durbin–Levinson (Bài 11, Mệnh đề 5).

    φ_hh = {ρ(h) − Σ_{j<h} φ_{h−1,j} ρ(h−j)} / {1 − Σ_{j<h} φ_{h−1,j} ρ(j)};  φ_hj = φ_{h−1,j} − φ_hh φ_{h−1,h−j}.
    rho phải chứa ρ(0) = 1 ở vị trí 0 và ít nhất nlags + 1 phần tử.
    """
    rho = np.asarray(rho, dtype=float)
    out = np.ones(nlags + 1)
    phi = np.array([])
    for h in range(1, nlags + 1):
        num = rho[h] - phi @ rho[h - 1 : 0 : -1] if h > 1 else rho[1]
        den = 1 - phi @ rho[1:h] if h > 1 else 1.0
        phi_hh = num / den
        phi = np.append(phi - phi_hh * phi[::-1], phi_hh)
        out[h] = phi_hh
    return out


def ljung_box_manual(y, H: int, model_df: int = 0) -> tuple[float, float]:
    """Q_LB(H) = n(n+2) Σ_{h=1..H} ρ̂(h)²/(n−h) ~ χ²_{H−model_df} (Bài 11, Định lý 2(b)).

    model_df = p + q cho phần dư ARMA(p, q) (Bài 11, Mục 5). Trả về (Q, p-value).
    """
    y = np.asarray(y, dtype=float)
    n = len(y)
    r = sample_acf(y, H)[1:]
    q = n * (n + 2) * np.sum(r**2 / (n - np.arange(1, H + 1)))
    return float(q), float(stats.chi2.sf(q, H - model_df))

In [ ]:
%%hocphan arma_tools
"""Công cụ AR/MA/ARMA theo Bài 11 (chỉ dùng numpy/scipy, không gọi statsmodels).

Quy ước: phi = [φ₁..φ_p], theta = [θ₁..θ_q] (không chứa hệ số 1 đứng đầu).
"""
import numpy as np
from scipy.linalg import solve, toeplitz

from .acf_tools import sample_autocov


def yule_walker_manual(y, p: int) -> tuple[np.ndarray, float]:
    """Yule–Walker: φ̂ = Γ̂ₚ⁻¹ γ̂ₚ, σ̂² = γ̂(0) − φ̂ᵀγ̂ₚ (Bài 11, Định lý 6(a)); γ̂ dùng mẫu số n.

    Trả về (phi_hat, sigma2_hat). Không hiệu chỉnh n/(n−p−1) như ar.yw của R.
    """
    g = sample_autocov(y, p)
    phi = solve(toeplitz(g[:p]), g[1 : p + 1])
    return phi, float(g[0] - phi @ g[1 : p + 1])


def ar_roots(phi) -> np.ndarray:
    """Nghiệm của φ(z) = 1 − φ₁z − … − φ_p z^p (Bài 11, Định lý 4(a))."""
    return np.roots(np.r_[-np.asarray(phi, dtype=float)[::-1], 1.0])


def ma_roots(theta) -> np.ndarray:
    """Nghiệm của θ(z) = 1 + θ₁z + … + θ_q z^q (khả nghịch ⇔ mọi |z| > 1, Định lý 5(b))."""
    return np.roots(np.r_[np.asarray(theta, dtype=float)[::-1], 1.0])


def ar_roots_modulus(phi) -> np.ndarray:
    """Môđun các nghiệm của φ(z); nhân quả ⇔ tất cả > 1."""
    return np.abs(ar_roots(phi))


def ar2_in_triangle(phi1: float, phi2: float) -> bool:
    """Tam giác nhân quả AR(2): φ₁+φ₂ < 1, φ₂−φ₁ < 1, |φ₂| < 1 (Bài 11, Mệnh đề 3)."""
    return phi1 + phi2 < 1 and phi2 - phi1 < 1 and abs(phi2) < 1


def ar2_period(phi1: float, phi2: float) -> float:
    """Chu kỳ 2π/ω của AR(2) có nghiệm phức (φ₁² + 4φ₂ < 0), cos ω = φ₁/(2r), r = √(−φ₂); NaN nếu nghiệm thực."""
    if phi1**2 + 4 * phi2 >= 0:
        return float("nan")
    return float(2 * np.pi / np.arccos(phi1 / (2 * np.sqrt(-phi2))))


def psi_weights(phi, theta, n: int) -> np.ndarray:
    """ψ₀..ψ_{n−1}, hệ số của ψ(z) = θ(z)/φ(z): ψⱼ = θⱼ + Σ_{i≤min(j,p)} φᵢψ_{j−i}, ψ₀ = 1 (Bài 11, Định lý 5(a))."""
    phi, theta = np.asarray(phi, dtype=float), np.asarray(theta, dtype=float)
    psi = np.zeros(n)
    psi[0] = 1.0
    for j in range(1, n):
        psi[j] = (theta[j - 1] if j <= len(theta) else 0.0) + sum(
            phi[i - 1] * psi[j - i] for i in range(1, min(j, len(phi)) + 1)
        )
    return psi


def theoretical_acvf(phi, theta, nlags: int, sigma2: float = 1.0, terms: int = 2000) -> np.ndarray:
    """γ(h) = σ² Σⱼ ψⱼψⱼ₊ₕ, h = 0..nlags (Bài 11, Mệnh đề 2); cắt chuỗi ở `terms` số hạng (đủ khi mô hình nhân quả)."""
    psi = psi_weights(phi, theta, terms + nlags)
    return sigma2 * np.array([psi[: terms] @ psi[h : h + terms] for h in range(nlags + 1)])


def theoretical_acf(phi, theta, nlags: int, terms: int = 2000) -> np.ndarray:
    """ρ(h) = γ(h)/γ(0), h = 0..nlags của ARMA nhân quả."""
    g = theoretical_acvf(phi, theta, nlags, 1.0, terms)
    return g / g[0]


def causal_invertible_check(phi, theta) -> dict:
    """Kiểm tra nhân quả (nghiệm φ ngoài vòng đơn vị) và khả nghịch (nghiệm θ ngoài vòng đơn vị).

    Trả về môđun nhỏ nhất của mỗi bên và khoảng cách nhỏ nhất giữa nghiệm AR và nghiệm MA
    (gần 0 ⇒ gần triệt tiêu nghiệm chung, Bài 11, Định lý 5 và Ví dụ 4.2).
    """
    ar, ma = ar_roots(phi), ma_roots(theta)
    gap = float(np.min(np.abs(ar[:, None] - ma[None, :]))) if len(ar) and len(ma) else float("nan")
    return {
        "causal": bool(len(ar) == 0 or np.all(np.abs(ar) > 1)),
        "invertible": bool(len(ma) == 0 or np.all(np.abs(ma) > 1)),
        "min_ar_modulus": float(np.abs(ar).min()) if len(ar) else float("nan"),
        "min_ma_modulus": float(np.abs(ma).min()) if len(ma) else float("nan"),
        "min_ar_ma_root_gap": gap,
    }


def ma1_ratio(theta: float) -> float:
    """ρ(1) = θ/(1 + θ²) của MA(1) (Bài 11, Mệnh đề 4(b)); |ρ(1)| ≤ 1/2."""
    return theta / (1 + theta**2)

In [ ]:
%%hocphan evaluation
"""Thước đo độ chính xác, gốc cuốn chiếu và kiểm định Diebold–Mariano (Bài 12, Định nghĩa 5, Định lý 6)."""
from typing import Callable

import numpy as np
from scipy import stats


def _err(actual, forecast) -> np.ndarray:
    return np.asarray(actual, dtype=float) - np.asarray(forecast, dtype=float)


def mae(actual, forecast) -> float:
    return float(np.mean(np.abs(_err(actual, forecast))))


def rmse(actual, forecast) -> float:
    return float(np.sqrt(np.mean(_err(actual, forecast) ** 2)))


def mape(actual, forecast) -> float:
    """MAPE theo %; không xác định khi có giá trị thực bằng 0 (Định lý 6(c))."""
    return float(100 * np.mean(np.abs(_err(actual, forecast) / np.asarray(actual, dtype=float))))


def mase(actual, forecast, y_train, s: int = 1) -> float:
    """MASE = MAE/Q, Q = (1/(n−s)) Σ_{t>s} |yₜ − yₜ₋ₛ| trên dữ liệu huấn luyện (s = 1 nếu không mùa vụ)."""
    y = np.asarray(y_train, dtype=float)
    return mae(actual, forecast) / float(np.mean(np.abs(y[s:] - y[:-s])))


def rolling_origin_forecast(y, fit_forecast: Callable, t0: int, h: int = 1):
    """Đánh giá gốc cuốn chiếu (Bài 12, Định nghĩa 5(c)).

    Với mỗi gốc t = t0..n−h: khớp lại trên y[:t] bằng `fit_forecast(y_train, h)` (trả về mảng h dự báo),
    lấy dự báo bước h cho y[t+h−1]. Trả về (actual, forecast), mỗi mảng có n − h − t0 + 1 phần tử.
    """
    y = np.asarray(y, dtype=float)
    origins = range(t0, len(y) - h + 1)
    forecast = np.array([np.asarray(fit_forecast(y[:t], h))[h - 1] for t in origins])
    return y[[t + h - 1 for t in origins]], forecast


def rolling_origin_horizons(y, fit_forecast: Callable, t0: int, H: int) -> dict:
    """Như rolling_origin_forecast nhưng cho mọi chân trời h = 1..H, mỗi gốc chỉ khớp một lần.

    Trả về {h: (actual, forecast)}; số gốc giảm dần theo h vì cuối mẫu không đủ dữ liệu.
    """
    y = np.asarray(y, dtype=float)
    out = {h: ([], []) for h in range(1, H + 1)}
    for t in range(t0, len(y)):
        hmax = min(H, len(y) - t)
        f = np.asarray(fit_forecast(y[:t], hmax))
        for h in range(1, hmax + 1):
            out[h][0].append(y[t + h - 1])
            out[h][1].append(f[h - 1])
    return {h: (np.array(a), np.array(f)) for h, (a, f) in out.items()}


def diebold_mariano(d, h: int = 1) -> tuple[float, float]:
    """DM = d̄ / √(v̂/m), v̂ ≈ Σ_{|k|<h} γ̂_d(k) (Bài 12, Định lý 6(d)); p-value dùng phân phối t với m − 1 bậc tự do
    (như mã Mục 6 của bài giảng). d_t = L(e₁ₜ) − L(e₂ₜ).

    shortcut: h > 1 dùng trọng số đều (không Bartlett) nên v̂ có thể âm, chỉ nên dùng với h nhỏ; nâng cấp thành HAC Bartlett khi cần.
    """
    d = np.asarray(d, dtype=float)
    m = len(d)
    c = d - d.mean()
    v = c @ c / (m - 1)  # h = 1: đúng var(ddof=1) của bài giảng
    for k in range(1, h):
        v += 2 * (c[k:] @ c[:-k]) / (m - 1)
    dm = d.mean() / np.sqrt(v / m)
    return float(dm), float(2 * stats.t.sf(abs(dm), m - 1))

In [ ]:
%%hocphan forecast_tools
"""Trọng số Ψ, phương sai sai số dự báo, chuyển ngược log, làm trơn mũ, mốc chuẩn (Bài 12). Không gọi statsmodels."""
import numpy as np


def psi_from_polys(ar_full, ma_full, n: int) -> np.ndarray:
    """Hệ số Ψ₀..Ψ_{n−1} của Ψ(z) = M(z)/A(z), với A, M là đa thức đầy đủ (hệ số đầu = 1, tăng dần theo bậc z).

    Đệ quy: Ψⱼ = Mⱼ − Σ_{i≥1} Aᵢ Ψ_{j−i}.
    """
    a, m = np.asarray(ar_full, dtype=float), np.asarray(ma_full, dtype=float)
    psi = np.zeros(n)
    for j in range(n):
        psi[j] = (m[j] if j < len(m) else 0.0) - sum(a[i] * psi[j - i] for i in range(1, min(j, len(a) - 1) + 1))
    return psi


def Psi_weights_arima(phi, theta, d: int, n: int) -> np.ndarray:
    """Ψ₀..Ψ_{n−1} của Ψ(z) = θ(z)/{φ(z)(1−z)^d} (Bài 12, Định lý 3(b), Định lý 4)."""
    ar = np.r_[1.0, -np.asarray(phi, dtype=float)]
    for _ in range(d):
        ar = np.polymul(ar, [1.0, -1.0])
    return psi_from_polys(ar, np.r_[1.0, np.asarray(theta, dtype=float)], n)


def sarima_polys(phi=(), theta=(), Phi=(), Theta=(), d: int = 0, D: int = 0, s: int = 12):
    """Đa thức AR, MA đầy đủ của SARIMA(p,d,q)(P,D,Q)ₛ (Bài 12, Định nghĩa 4(b)); AR đã nhân (1−z)^d (1−zˢ)^D.

    Trả về (ar_full, ma_full) để đưa vào psi_from_polys.
    """
    def seasonal(c):  # 1 + c₁zˢ + c₂z²ˢ + …
        out = np.zeros(len(c) * s + 1)
        out[0] = 1.0
        out[s :: s] = np.asarray(c, dtype=float)
        return out

    ar = np.polymul(np.r_[1.0, -np.asarray(phi, dtype=float)], seasonal(-np.asarray(Phi, dtype=float)))
    ma = np.polymul(np.r_[1.0, np.asarray(theta, dtype=float)], seasonal(Theta))
    for _ in range(d):
        ar = np.polymul(ar, [1.0, -1.0])
    for _ in range(D):
        ar = np.polymul(ar, np.r_[1.0, np.zeros(s - 1), -1.0])
    return ar, ma


def forecast_variance_manual(Psi, sigma2: float, H: int) -> np.ndarray:
    """Var eₙ(h) = σ² Σ_{j<h} Ψⱼ², h = 1..H (Bài 12, Định lý 4(a))."""
    return sigma2 * np.cumsum(np.asarray(Psi, dtype=float)[:H] ** 2)


def back_transform_log(m, s, z: float = 1.96) -> dict:
    """Dự báo trên thang log N(m, s²) → thang gốc (Bài 12, Mệnh đề 2).

    median = eᵐ, mean = e^{m+s²/2}, khoảng chuyển đổi chính xác (e^{m−zs}, e^{m+zs}).
    """
    m, s = np.asarray(m, dtype=float), np.asarray(s, dtype=float)
    return {"median": np.exp(m), "mean": np.exp(m + s**2 / 2), "lower": np.exp(m - z * s), "upper": np.exp(m + z * s)}


def ses_recursion(y, alpha: float, init: float | None = None) -> np.ndarray:
    """Đệ quy làm trơn mũ đơn ℓₜ = αyₜ + (1−α)ℓₜ₋₁, ℓ₁ = init (mặc định y₁) (Bài 12, Định lý 3(c)).

    Trả về ℓ₁..ℓₙ; dự báo mọi bước sau là ℓₙ. Với ARIMA(0,1,1): α = 1 + θ.
    """
    y = np.asarray(y, dtype=float)
    level = np.empty(len(y))
    level[0] = y[0] if init is None else init
    for t in range(1, len(y)):
        level[t] = alpha * y[t] + (1 - alpha) * level[t - 1]
    return level


def local_level_variances(theta: float, sigma2: float) -> tuple[float, float]:
    """(σ̂_ε², σ̂_η²) của mô hình mức cục bộ từ ARIMA(0,1,1): σ_ε² = −θσ², σ_η² = (1+θ)²σ² (Bài 12, Ví dụ 2.2)."""
    return -theta * sigma2, (1 + theta) ** 2 * sigma2


def zero_diffuse_residuals(resid, k: int) -> np.ndarray:
    """Đặt k phần dư đầu về 0 (k = d + D·s). statsmodels khởi tạo khuếch tán nên resid[0] ≈ giá trị quan sát (rất lớn);
    cách này giống R, nơi các phần dư khuếch tán gần 0, để ACF/Ljung–Box của phần dư không bị méo."""
    r = np.array(resid, dtype=float)
    r[:k] = 0.0
    return r


# --- Mốc chuẩn (Bài 12, Định nghĩa 5(a)); y là dữ liệu huấn luyện y₁..yₙ, trả về dự báo h = 1..H ---
def forecast_mean(y, H: int) -> np.ndarray:
    return np.full(H, np.mean(y))


def forecast_naive(y, H: int) -> np.ndarray:
    return np.full(H, np.asarray(y, dtype=float)[-1])


def forecast_seasonal_naive(y, H: int, s: int) -> np.ndarray:
    """ŷₙ₊ₕ = yₙ₊ₕ₋ₛ₍ₖ₊₁₎ với k = ⌊(h−1)/s⌋."""
    y = np.asarray(y, dtype=float)
    n = len(y)
    return np.array([y[n + h - s * ((h - 1) // s + 1) - 1] for h in range(1, H + 1)])


def forecast_drift(y, H: int) -> np.ndarray:
    """ŷₙ₊ₕ = yₙ + h(yₙ − y₁)/(n − 1)."""
    y = np.asarray(y, dtype=float)
    return y[-1] + np.arange(1, H + 1) * (y[-1] - y[0]) / (len(y) - 1)

In [ ]:
%%hocphan simulation
"""Mô phỏng (luôn nhận rng: numpy.random.Generator, sinh từ SEED = 3179)."""
from typing import Callable

import numpy as np
from scipy.signal import lfilter

from .config import SEED


def child_rng(*key: int) -> np.random.Generator:
    """Generator con sinh từ SEED = 3179 và khóa của mục (ví dụ child_rng(11, 3)): kết quả không phụ thuộc thứ tự chạy ô."""
    return np.random.default_rng([SEED, *key])


def simulate_arma(phi, theta, n: int, rng: np.random.Generator, sigma: float = 1.0, burn: int = 200) -> np.ndarray:
    """Chuỗi ARMA(p,q) có nhiễu N(0, σ²), bỏ `burn` giá trị đầu (như burn-in 200 của bài giảng)."""
    eps = sigma * rng.normal(size=n + burn)
    y = lfilter(np.r_[1.0, np.asarray(theta, dtype=float)], np.r_[1.0, -np.asarray(phi, dtype=float)], eps)
    return y[burn:]


def simulate_ar1(phi: float, n: int, rng: np.random.Generator, sigma: float = 1.0, burn: int = 200) -> np.ndarray:
    return simulate_arma([phi], [], n, rng, sigma, burn)


def simulate_random_walk(n: int, rng: np.random.Generator, delta: float = 0.0, sigma: float = 1.0,
                         y0: float = 0.0) -> np.ndarray:
    """Yₜ = Y₀ + δt + Σ_{j≤t} εⱼ, t = 1..n (Bài 12, Định lý 1), εⱼ ~ N(0, σ²)."""
    return y0 + np.cumsum(delta + sigma * rng.normal(size=n))


def coverage_study(simulate: Callable, interval: Callable, truth: float, B: int, rng: np.random.Generator) -> dict:
    """Độ phủ của khoảng tin cậy qua B lần lặp.

    simulate(rng) → một chuỗi; interval(y) → (lo, hi). Trả về độ phủ và MCSE = √{p(1−p)/B}.
    """
    hits = 0
    for _ in range(B):
        lo, hi = interval(simulate(rng))
        hits += lo <= truth <= hi
    p = hits / B
    return {"coverage": p, "mcse": float(np.sqrt(p * (1 - p) / B)), "B": B}

In [ ]:
%%hocphan captions
"""Chú thích (caption), nhãn cột chỉ số và ghi chú của mọi hình và bảng. Tên: fig_b<bài>_<mục>_<nội dung>, tab_b<bài>_<mục>_<nội dung>.

Chú thích mô tả *cái gì được vẽ/tính* (không diễn giải kết luận). Toán dùng $...$ (LaTeX).
"""


def _t(caption, index=None, note=None, fmt=None):
    return {k: v for k, v in dict(caption=caption, index=index, note=note, fmt=fmt).items() if v is not None}


DS = "Bộ dữ liệu"
ARTIFACTS = {
    # ---------------- Khám phá dữ liệu ----------------
    "tab_eda_descriptive": _t("Thống kê mô tả của biến được mô hình hóa trong sáu bộ dữ liệu", DS,
                              "n: số quan sát; SD: độ lệch chuẩn (mẫu số $n-1$); độ lệch: skewness; kurtosis dư: kurtosis trừ 3. HU, NI: thang gốc; LY: $\\log_{10}$; DR: %; AP, DP: log."),
    # ---------------- Bài 11 ----------------
    "tab_b11_01_stationarity": _t("Kiểm tra trực quan tính dừng: trung bình và phương sai trượt, $\\hat\\rho(1)$ và $Q_{LB}(10)$ của chuỗi và của bình phương độ lệch", DS,
                                  "Cửa sổ trượt 20 năm (HU, LY) và 250 ngày giao dịch (DR). $Q_{LB}$ tính trên mức nước gốc của HU (có xu thế)."),
    "tab_b11_02_gamma_properties": _t("Tính chất của hàm tự hiệp phương sai mẫu với mẫu số $n$ và $n-h$", DS,
                                      "Ma trận $\\hat\\Gamma$ cỡ $n\\times n$; trị riêng nhỏ nhất âm nghĩa là không xác định không âm."),
    "tab_b11_03_mean_se": _t("Sai số chuẩn của trung bình mẫu: giả định iid và có xét phụ thuộc", DS,
                             "$\\hat\\tau$ Bartlett cắt cụt ở $K=\\lfloor4(n/100)^{2/9}\\rfloor$; $\\hat\\tau$ Bartlett luôn không âm nên dùng được cho cả ba bộ. Cột ``SE theo công thức đầy đủ'' dùng công thức đầy đủ của $\\Var\\bar Y$ thay $\\hat\\gamma$, chỉ dùng độ trễ $h\\le n/4$; cắt cụt trọng số $(1-h/n)$ làm mất tính xác định dương, nên với ACF dao động (LY) tổng này âm và được ghi dấu –."),
    "tab_b11_03_slope_se": _t("Hệ số dốc xu thế của hồ Huron: OLS (sai số iid) so với ML có sai số AR(2)", "Phương pháp"),
    "tab_b11_03_coverage": _t("Độ phủ của khoảng tin cậy 95% cho trung bình, AR(1) với $\\phi=0.6$, $n=100$, $B=2000$", "Khoảng tin cậy",
                              "MCSE: sai số Monte Carlo $\\sqrt{p(1-p)/B}$. Trung bình thật $\\mu=0$."),
    "tab_b11_04_acf_ljungbox": _t("ACF mẫu: dải tin cậy, số độ trễ vượt dải ($h\\le20$) và kiểm định Ljung–Box với $H=10$", "Chuỗi",
                                  "Hai hàng $Q_{LB}$ (tự cài đặt và thư viện) trùng nhau; dải Bartlett giả định MA(1) với $\\hat\\rho(1)$ của chính chuỗi (không áp dụng cho $r_t^2$)."),
    "tab_b11_04_simulation": _t("Nhiễu iid $N(0,1)$, $n=200$, $B=2000$: tần suất vượt dải và phân phối của $Q_{LB}(10)$", "Đại lượng",
                                 "Xấp xỉ mẫu hữu hạn dùng $\\E\\hat\\rho(h)\\approx-1/n$ và $\\Var\\hat\\rho(h)\\approx(n-h)/\\{n(n+2)\\}$, lấy trung bình theo $h=1,\\dots,20$. MCSE của tỷ lệ vượt dải coi $20B$ chỉ báo là độc lập."),
    "tab_b11_05_psi_acf": _t("AR(2) khớp bằng ML: trọng số $\\psi_j$ và ACF lý thuyết, cài đặt tay so với statsmodels", DS),
    "tab_b11_06_ar1_acf": _t("HU: ACF mẫu của phần dư khử xu thế so với $\\hat\\phi^h$ của mô hình AR(1)", "Độ trễ h"),
    "tab_b11_06_ar1_forecast_mse": _t("HU, AR(1): MSE dự báo $h$ bước, công thức $\\sigma^2(1-\\phi^{2h})/(1-\\phi^2)$ và statsmodels", "Chân trời h"),
    "tab_b11_06_ar1_simulation": _t("AR(1) với $(\\hat\\phi,\\hat\\sigma)$ của HU, $n=98$, $B=2000$: kiểm tra $\\gamma(0)$ và $\\rho(1)$", "Đại lượng",
                                     "Kỳ vọng mẫu hữu hạn: $\\E\\hat\\gamma(0)=\\gamma(0)-\\Var\\bar Y$ (chính xác) và $\\E\\hat\\rho(1)\\approx\\phi-(1+4\\phi)/n$ (xấp xỉ)."),
    "tab_b11_06_ly_ar1": _t("LY: ACF mẫu so với $\\hat\\phi^h$ của mô hình AR(1) khớp bằng cực đại hợp lý", "Độ trễ h"),
    "tab_b11_07_yule_walker": _t("Ước lượng Yule–Walker giải bằng tay: so với thư viện, môđun nghiệm và tính nhân quả", None),
    "tab_b11_08_ar2_triangle": _t("AR(2): vị trí trong tam giác nhân quả, loại nghiệm và chu kỳ", "Mô hình"),
    "tab_b11_09_ma1": _t("MA(1): $\\rho(1)=\\theta/(1+\\theta^2)$, khả nghịch và cặp mô hình $(\\theta,\\sigma^2)$, $(1/\\theta,\\theta^2\\sigma^2)$", "Mô hình"),
    "tab_b11_09_ma2_simulation": _t("MA(2) với $\\hat\\theta$ của HU, $n=98$, $B=2000$: ACF mẫu cắt cụt sau độ trễ 2", "Độ trễ h",
                                    "Dải Bartlett chỉ tính cho $h>q=2$."),
    "tab_b11_10_arma11": _t("ARMA(1,1) khớp bằng ML: nghiệm AR/MA, công thức đóng và cài đặt tổng quát", DS),
    "tab_b11_10_overparameterized": _t("ARMA(1,1) khớp cho nhiễu trắng $n=200$: ước lượng và sai số chuẩn", "Tham số"),
    "tab_b11_10_cancellation": _t("ARMA(1,1) khớp cho nhiễu trắng: dấu hiệu triệt tiêu nghiệm chung", "Đại lượng"),
    "tab_b11_11_pacf": _t("PACF mẫu: Durbin–Levinson tự cài đặt so với statsmodels, độ trễ vượt dải và bậc AR gợi ý", DS),
    "tab_b11_12_estimators": _t("Ước lượng AR(2) bằng Yule–Walker, OLS và cực đại hợp lý", None,
                                "SE của Yule–Walker là SE tiệm cận $\\sqrt{\\operatorname{diag}(\\hat\\sigma^2\\hat\\Gamma_2^{-1}/n)}$."),
    "tab_b11_12_simulation": _t("AR(2) với $\\hat\\phi$ của LY, $n=114$, $B=2000$: phân phối mẫu của $\\hat\\phi$", None),
    "tab_b11_13_ly_ic": _t("LY: AIC và BIC của AR($p$), $p=1,\\dots,14$ (ML)", "Bậc p"),
    "tab_b11_13_hu_models": _t("HU: so sánh các mô hình sai số ARMA cho hồi quy xu thế", "Sai số u_t",
                               "$k$ gồm các hệ số ARMA, hằng số, hệ số dốc và $\\sigma^2$; $Q_{LB}(10)$ với $10-p-q$ bậc tự do."),
    "tab_b11_13_dr_grid": _t("DR: AIC và BIC của ARMA($p,q$), $p,q\\le2$", "(p, q)"),
    "tab_b11_13_ly_rolling": _t("LY: sai số dự báo 1 bước trên gốc cuốn chiếu, 1915--1934 (khớp lại mỗi năm)", "Mô hình",
                                 "Dòng ``AR($p$) chọn theo AIC trong mỗi gốc'' chọn $p\\in\\{1,\\dots,14\\}$ tại từng gốc chỉ bằng dữ liệu trước gốc đó."),
    "tab_b11_13_ly_order_train": _t("LY: bậc AR do AIC và BIC chọn khi dùng toàn bộ chuỗi và khi chỉ dùng dữ liệu trước giai đoạn đánh giá", "Dữ liệu dùng để chọn bậc"),
    "tab_b11_14_residual_diagnostics": _t("Chẩn đoán phần dư của các mô hình được chọn", None,
                                          "$Q_{LB}$ của phần dư dùng $H-p-q$ bậc tự do; của phần dư bình phương dùng $H$ bậc tự do."),
    # ---------------- Bài 12 ----------------
    "tab_b12_01_levels_vs_differences": _t("Chuỗi gốc, khử xu thế tuyến tính và sai phân bậc một: độ lệch chuẩn và $\\hat\\rho(h)$", DS),
    "tab_b12_02_rw_variance": _t("Bước ngẫu nhiên mô phỏng với $(\\hat\\delta,\\hat\\sigma)$ của DP: $\\operatorname{Var}Y_t$ và $EY_t$", "t",
                                 "$B=2000$ đường, $Y_0=0$."),
    "tab_b12_02_rw_correlation": _t("Bước ngẫu nhiên mô phỏng: $\\operatorname{Corr}(Y_t,Y_{t+h})$ so với $\\sqrt{t/(t+h)}$", "(t, h)"),
    "tab_b12_02_rw_sample_acf": _t("Trung bình ACF mẫu của bước ngẫu nhiên ($n=200$, $B=2000$)", "Độ trễ h"),
    "tab_b12_02_spurious_regression": _t("Hồi quy giả mạo: tần suất bác bỏ $\\beta_1=0$ ở mức 5% cho hai bước ngẫu nhiên độc lập ($n=100$)", "Kịch bản"),
    "tab_b12_03_overdifferencing": _t("Sai phân thừa: $\\hat\\rho(1)$, phương sai dài hạn và $\\hat\\theta$ MA(1)", None,
                                      "Phương sai dài hạn: ước lượng Bartlett cắt cụt ở $K=\\lfloor4(n/100)^{2/9}\\rfloor$."),
    "tab_b12_03_cost_simulation": _t("Chi phí của sai phân thừa và thiếu: mô phỏng so với lý thuyết", "Đại lượng"),
    "tab_b12_04_unit_root_tests": _t("Kiểm định nghiệm đơn vị ADF (H$_0$: có nghiệm đơn vị) và KPSS (H$_0$: dừng)", None,
                                     "p-value của KPSS được nội suy từ bảng nên bị chặn trong $[0.01,0.10]$; ``$\\le0.01$'' và ``$\\ge0.10$'' nghĩa là ngoài khoảng bảng. Cột độ trễ: số hạng $\\nabla Y_{t-i}$ của ADF (chọn theo AIC) hoặc độ trễ Newey--West của KPSS (`nlags='auto'`)."),
    "tab_b12_04_null_distribution": _t("Phân phối của thống kê $\\tau$ (có hệ số chặn) dưới H$_0$: bước ngẫu nhiên $n=200$, $B=2000$", "Đại lượng"),
    "tab_b12_03_bartlett_by_k": _t("Ước lượng Bartlett của phương sai dài hạn theo số độ trễ $K$, chuỗi sai phân một lần và sai phân thừa", None,
                                   "$X=\\nabla y$. Cột cuối là giá trị đóng của tổng Bartlett $K$ độ trễ của $\\nabla X$ khi thay $\\gamma_X$ bằng $\\hat\\gamma_X$; nó giảm như $1/(K+1)$."),
    "tab_b12_04_adf_power": _t("Tỷ lệ bác bỏ nghiệm đơn vị ở mức 5% (có hệ số chặn): DF không độ trễ và ADF chọn độ trễ theo AIC, $B=1000$", None,
                               "Hai cột dùng cùng các chuỗi mô phỏng. Cột ADF dùng mặc định của adfuller (độ trễ chọn theo AIC), như mã mẫu của Bài 12."),
    "tab_b12_05_arima_params": _t("Tham số ARIMA đã khớp: ước lượng, sai số chuẩn và thống kê $z$", None),
    "tab_b12_05_fit_stats": _t("Độ trôi và tiêu chuẩn thông tin của các mô hình ARIMA đã khớp", "Mô hình",
                               "AIC/BIC chỉ so sánh được giữa các mô hình có cùng $d$ và cùng dữ liệu. Với $d=1$, hệ số \\texttt{x1} của \\texttt{ARIMA(trend='t')} là $\\mu_W=\\E\\nabla Y_t$, không phải $c=\\phi(1)\\mu_W$."),
    "tab_b12_05_rw_manual_vs_arima": _t("DP: $\\hat\\delta$, $\\hat\\sigma^2$ tính tay từ $\\nabla y$ so với ARIMA(0,1,0) có độ trôi", "Tham số"),
    "tab_b12_06_psi_weights": _t("Trọng số $\\Psi_k$ (ảnh hưởng của một cú sốc đơn vị lên $Y_{t+k}$) và giới hạn $\\theta(1)/\\phi(1)$", "Mô hình"),
    "tab_b12_07_ses_forecasts": _t("NI: dự báo 1971 bằng bốn cách (ARIMA(0,1,1), đệ quy làm trơn tay, `SimpleExpSmoothing` với $\\alpha$ cố định và với $\\hat\\alpha$ tối ưu)", "Cách tính"),
    "tab_b12_07_local_level": _t("NI: phương sai của mô hình mức cục bộ, từ ARIMA(0,1,1) và từ `UnobservedComponents`", "Nguồn ước lượng"),
    "tab_b12_08_forecast_variance": _t("Phương sai sai số dự báo: công thức tay (Ψ và công thức đóng) so với statsmodels", DS),
    "tab_b12_08_coverage": _t("Độ phủ mô phỏng của khoảng dự báo 95% ARIMA(0,1,1) (khớp lại mô hình đúng mỗi lần)", "Thiết kế mô phỏng"),
    "tab_b12_09_ap_backtransform": _t("AP: dự báo airline trên thang gốc, trung vị và trung bình, so với giá trị thực", "Tháng"),
    "tab_b12_09_dp_forecast": _t("DP: dự báo giá DAX từ bước ngẫu nhiên có trôi (giá cuối 5473.72)", "Chân trời h (ngày giao dịch)"),
    "tab_b12_10_sd": _t("AP (log, huấn luyện): độ lệch chuẩn sau các phép sai phân", "Chuỗi"),
    "tab_b12_10_airline_acf": _t("AP: ACF của $\\nabla\\nabla_{12}$ log tại các độ trễ đặc trưng so với giá trị lý thuyết của mô hình airline", "Độ trễ h"),
    "tab_b12_10_seasonality_models": _t("AP: ba cách mô hình hóa mùa vụ, có và không có độ trôi, AIC và RMSE trên tập kiểm định 1959--1960", "Mô hình",
                                        "n dòng: số quan sát còn lại sau sai phân, như Ví dụ 4.2 của Bài 12. AIC chỉ so sánh trực tiếp được giữa các mô hình cùng cách sai phân. Dòng ``+ trôi'' thêm hằng số vào phương trình của $\\nabla y$, tức một độ trôi; Ví dụ 4.2 dùng bản không có độ trôi.",
                                        fmt={"AIC": "{:.1f}", "RMSE giữ lại": "{:.1f}"}),
    "tab_b12_10_seasonal_lags": _t("ACF của chuỗi sai phân bậc một ở độ trễ mùa vụ 12 và 24", "Chuỗi sai phân"),
    "tab_b12_11_ap_benchmarks": _t("AP: độ chính xác trên tập kiểm định 24 tháng (thang gốc)", "Phương pháp", "MASE chia cho MAE của naive mùa vụ ($s=12$) trong mẫu huấn luyện. ``airline'' là trung vị $e^{m}$, ``airline (trung bình)'' là $e^{m+s^2/2}$."),
    "tab_b12_11_ni_benchmarks": _t("NI: độ chính xác dự báo 20 năm cuối (huấn luyện 1871--1950, gốc cố định)", "Phương pháp"),
    "tab_b12_11_dp_one_step": _t("DP: độ chính xác dự báo 1 bước trên 360 ngày kiểm định (log, đơn vị %)", "Phương pháp"),
    "tab_b12_11_dp_multistep": _t("DP: độ chính xác dự báo đa bước từ gốc cố định $t=1500$ (thang giá)", "Phương pháp"),
    "tab_b12_12_ni_rolling": _t("NI: RMSE gốc cuốn chiếu theo chân trời, gồm cả bước chọn mô hình lồng trong mỗi gốc", "Phương pháp"),
    "tab_b12_12_ap_rolling_by_horizon": _t("AP: RMSE gốc cuốn chiếu theo chân trời $h=1,\\dots,12$", "Chân trời h (tháng)"),
    "tab_b12_12_ap_one_step": _t("AP: RMSE 1 bước, gốc $t=120,\\dots,143$", "Phương pháp"),
    "tab_b12_12_dp_rolling": _t("DP: dự báo 1 bước trên 360 ngày với tham số cố định và khớp lại định kỳ", "Phương pháp"),
    "tab_b12_13_diebold_mariano": _t("Kiểm định Diebold–Mariano trên sai số 1 bước của gốc cuốn chiếu", None),
    "tab_b12_13_ranking": _t("AP: thứ hạng sáu dự báo theo RMSE và theo MAE trên tập kiểm định", "Phương pháp"),
    "tab_b12_13_loss_functional": _t("Hàm mất mát và phiếm hàm: cực tiểu trên lưới của tổng bình phương/tuyệt đối (AP, huấn luyện)", "Hàm mất mát"),
    "tab_b12_14_checklist": _t("Checklist báo cáo dự báo ARIMA cho ba bộ dữ liệu", "Mục"),
    "tab_b12_ext_cointegration": _t("Mở rộng: hồi quy $\\log\\mathrm{DAX}$ theo $\\log\\mathrm{SMI}$ và kiểm định Engle--Granger", "Đại lượng"),
    # ---------------- Hình ----------------
    **{f"fig_eda_{c}_overview": _t(t) for c, t in {
        "HU": "HU: mực nước hồ Huron với xu thế tuyến tính OLS, ACF và PACF mẫu",
        "LY": "LY: $\\log_{10}$ số linh miêu bị bẫy, ACF và PACF mẫu",
        "DR": "DR: suất sinh lợi log ngày của DAX, ACF và PACF mẫu",
        "NI": "NI: lưu lượng sông Nile (trung bình trước và sau 1899), ACF và PACF mẫu",
        "AP": "AP: log số hành khách (huấn luyện | kiểm định), ACF và PACF mẫu",
        "DP": "DP: log giá DAX (huấn luyện | kiểm định), ACF và PACF mẫu"}.items()},
    "fig_b11_01_series_rolling": _t("Chuỗi, trung bình trượt (trái) và phương sai trượt (phải) của HU, LY, DR"),
    "fig_b11_02_acf_denominators": _t("$\\hat\\rho(h)$ với mẫu số $n$ và $n-h$ ở mọi độ trễ $h=1,\\dots,n-1$"),
    "fig_b11_04_acf_bands": _t("ACF mẫu và dải 95% (đứt nét: iid $\\pm1.96/\\sqrt n$; chấm: Bartlett MA(1)); độ trễ vượt dải tô cam đỏ"),
    "fig_b11_05_acf_theory": _t("ACF mẫu (que) và ACF lý thuyết của AR(2) khớp bằng ML (đường)"),
    "fig_b11_08_ar2_triangle": _t("Tam giác nhân quả AR(2) với điểm HU, LY (trái) và ACF mẫu của LY so với AR(2)-YW (phải)"),
    "fig_b11_09_ma2_acf": _t("MA(2) mô phỏng: trung bình ACF mẫu so với ACF lý thuyết cắt cụt sau độ trễ 2"),
    "fig_b11_11_pacf": _t("PACF mẫu (Durbin--Levinson) của HU đã khử xu thế, LY và DR"),
    "fig_b11_13_ic_ly": _t("LY: AIC và BIC (trừ giá trị nhỏ nhất) theo bậc AR; dấu sao là bậc được chọn"),
    "fig_b11_14_residual_acf": _t("ACF của phần dư (trái) và của phần dư bình phương (phải) cho các mô hình được chọn"),
    "fig_b12_01_levels_vs_differences": _t("Chuỗi, ACF của chuỗi gốc và ACF của sai phân bậc một cho NI, AP (log, huấn luyện), DP (log)"),
    "fig_b12_02_rw_variance": _t("Bước ngẫu nhiên mô phỏng: $\\operatorname{Var}Y_t$ so với $t\\hat\\sigma^2$ (trái), 50 đường mẫu (phải)"),
    "fig_b12_04_dickey_fuller_null": _t("Phân phối mô phỏng của $\\tau$ dưới H$_0$ so với $N(0,1)$ và giá trị tới hạn 5%"),
    "fig_b12_05_drift_forecast": _t("DP: dự báo 100 ngày giao dịch của log DAX có và không có độ trôi"),
    "fig_b12_06_psi_weights": _t("Ảnh hưởng của một cú sốc đơn vị lên $Y_{t+k}$ ($\\Psi_k$)"),
    "fig_b12_08_se_growth": _t("Sai số chuẩn của dự báo $h$ bước ($\\sigma=1$): $d=0$ so với $d=1$"),
    "fig_b12_08_ni_forecast": _t("NI: dự báo ARIMA(0,1,1) và khoảng dự báo 95% cho 20 năm tiếp theo"),
    "fig_b12_09_ap_forecast": _t("AP: dự báo airline 1959--1960 trên thang gốc (trung vị, khoảng 95%) và giá trị thực"),
    "fig_b12_10_airline_acf": _t("ACF của $\\nabla\\nabla_{12}$ log AP (que) và ACF lý thuyết của mô hình airline (chấm)"),
    "fig_b12_12_ap_rmse_by_horizon": _t("AP: RMSE gốc cuốn chiếu theo chân trời, airline so với naive mùa vụ"),
    "fig_b12_14_residual_acf": _t("ACF của phần dư (trái) và của phần dư bình phương (phải) cho NI, AP, DP"),
}


def caption_of(name: str) -> str:
    if name not in ARTIFACTS:
        raise KeyError(f"'{name}' chưa có caption trong as3179ts/captions.py")
    return ARTIFACTS[name]["caption"]

In [ ]:
%%hocphan reporting
"""Xuất bảng kết quả: CSV (đủ chữ số), Markdown và LaTeX (booktabs) vào outputs/tables/, tên tab_b<bài>_<mục>_<nội dung>.

Chú thích (caption), nhãn cột, định dạng số nằm ở một nơi (captions.py) để mọi bảng trình bày nhất quán.
"""
import re

import numpy as np
import pandas as pd

from .config import TABLE_DIR

# --- Unicode -> LaTeX (pdfLaTeX + T5 không in được chữ Hy Lạp, mũ ^ và chỉ số dưới) ---
GREEK = {"α": r"\alpha", "β": r"\beta", "γ": r"\gamma", "δ": r"\delta", "ε": r"\varepsilon", "η": r"\eta", "θ": r"\theta", "λ": r"\lambda",
         "μ": r"\mu", "ρ": r"\rho", "σ": r"\sigma", "τ": r"\tau", "φ": r"\phi", "ψ": r"\psi", "ω": r"\omega", "Γ": r"\Gamma", "Θ": r"\Theta",
         "Φ": r"\Phi", "Ψ": r"\Psi", "χ": r"\chi", "κ": r"\kappa", "π": r"\pi", "Σ": r"\Sigma", "ℓ": r"\ell"}
SUB = dict(zip("₀₁₂₃₄₅₆₇₈₉ₙ₊₋ₜₖⱼₐₑₒₓₕₘₚₛ", "0123456789n+-tkjaeoxhmps"))
SUP = dict(zip("⁰¹²³⁴⁵⁶⁷⁸⁹⁻", "0123456789-"))
SYM = {"≤": r"\le", "≥": r"\ge", "≈": r"\approx", "∇": r"\nabla", "Δ": r"\Delta", "√": r"\sqrt{}", "∑": r"\sum", "±": r"\pm", "×": r"\times",
       "→": r"\to", "≡": r"\equiv", "⋆": r"\star", "∞": r"\infty", "−": "-", "≠": r"\neq", "∈": r"\in", "·": r"\cdot", "≪": r"\ll"}
HAT = "\u0302"
SPECIAL = {"<": r"$<$", ">": r"$>$", "%": r"\%", "&": r"\&", "#": r"\#", "_": r"\_", "|": r"\textbar{}", "~": r"\textasciitilde{}", "{": r"\{", "}": r"\}"}


def _plain(s: str) -> str:
    """Thoát ký tự đặc biệt cho văn bản thường (dùng trong \\texttt{})."""
    return "".join(SPECIAL.get(c, r"\textbackslash{}" if c == "\\" else c) for c in s)


def _math_inner(s: str) -> str:
    """Nội dung toán của một đoạn Unicode ngắn (dùng bên trong \\sqrt{} hoặc chỉ số dưới): tex() rồi bỏ các dấu $."""
    return tex(s).replace("$", "")


def _closing(p: str, i: int, open_ch: str, close_ch: str) -> int:
    """Vị trí ngoặc đóng khớp với ngoặc mở tại p[i] (−1 nếu không có)."""
    depth = 0
    for j in range(i, len(p)):
        depth += (p[j] == open_ch) - (p[j] == close_ch)
        if depth == 0:
            return j
    return -1


def tex(s) -> str:
    """Chuỗi Unicode (có ký hiệu toán) -> LaTeX an toàn cho pdfLaTeX. Đoạn `$...$` có sẵn được giữ nguyên.

    Ngoài ra: `mã` -> \\texttt{mã}; "trích dẫn" -> ``trích dẫn''; √(…) và √x -> \\sqrt{…}; x_t, σ²_ε, t_{m−1} -> chỉ số dưới
    (chỉ khi trước dấu _ là một ký hiệu đơn, nên tên biến như level_ft vẫn in nguyên).
    """
    s = str(s).replace("Q_LB", "$Q_{LB}$").replace("n_eff", r"$n_{\mathrm{eff}}$")
    s = re.sub(r"(?<!`)`([^`]+)`(?!`)", lambda m: "\x00" + m.group(1) + "\x01", s)   # đánh dấu đoạn mã (không đụng ``…'')
    s = re.sub(r'"([^"$]*)"', r"``\1''", s)
    parts = re.split(r"(\$[^$]*\$|\x00[^\x01]*\x01)", s)
    out = []
    for p in parts:
        if p.startswith("$") and p.endswith("$") and len(p) > 1:
            out.append(p)
            continue
        if p.startswith("\x00"):
            out.append(r"\texttt{" + _plain(p[1:-1]) + "}")
            continue
        buf, i = [], 0
        while i < len(p):
            ch = p[i]
            nxt = p[i + 1] if i + 1 < len(p) else ""
            prev = p[i - 1] if i > 0 else ""
            prev2 = p[i - 2] if i > 1 else " "
            if ch == "√" and nxt == "(" and _closing(p, i + 1, "(", ")") > 0:
                j = _closing(p, i + 1, "(", ")")
                buf.append(r"$\sqrt{" + _math_inner(p[i + 2:j]) + "}$"); i = j + 1
                continue
            if ch == "√" and nxt and (nxt.isalnum() or nxt in GREEK):
                j = i + 1
                while j < len(p) and (p[j].isalnum() or p[j] in GREEK or p[j] == HAT):
                    j += 1
                buf.append(r"$\sqrt{" + _math_inner(p[i + 1:j]) + "}$"); i = j
                continue
            single = prev in GREEK or prev in SUP or prev == HAT or (prev.isalpha() and not prev2.isalpha())
            if ch == "_" and single and nxt == "{" and _closing(p, i + 1, "{", "}") > 0:
                j = _closing(p, i + 1, "{", "}")
                buf.append("$_{" + _math_inner(p[i + 2:j]) + "}$"); i = j + 1
                continue
            if ch == "_" and single and nxt and (nxt.isalnum() or nxt in GREEK):
                j = i + 2
                while nxt.isdigit() and j < len(p) and p[j].isdigit():   # Ψ_100: lấy cả dãy chữ số
                    j += 1
                buf.append("$_{" + _math_inner(p[i + 1:j]) + "}$"); i = j
                continue
            if ch in GREEK:
                g = GREEK[ch]
                if nxt == HAT:
                    g, i = rf"\hat{{{g}}}", i + 1
                buf.append(f"${g}$")
            elif ch in SUB or ch in SUP:
                table, mark = (SUB, "_") if ch in SUB else (SUP, "^")
                run = ""
                while i < len(p) and p[i] in table:
                    run += table[p[i]]; i += 1
                i -= 1
                buf.append(f"${mark}{{{run}}}$")
            elif ch in SYM and ch not in "−":
                buf.append(f"${SYM[ch]}$")
            elif ch == "−":
                buf.append("$-$" if nxt.isdigit() else "–")
            elif ch == "^" and nxt.isalnum():
                buf.append(f"$^{{{nxt}}}$"); i += 1
            elif ch == HAT:
                pass
            elif ch in SPECIAL:
                buf.append(SPECIAL[ch])
            elif ch == "\\":
                buf.append(r"\textbackslash{}")
            else:
                buf.append(ch)
            i += 1
        out.append("".join(buf))
    return re.sub(r"\$\$", "", "".join(out)).replace("$ $", " ")


_SUP_DIGITS = str.maketrans("0123456789-", "⁰¹²³⁴⁵⁶⁷⁸⁹⁻")


def sci(v: float, digits: int = 3) -> str:
    """Số dạng 2.06×10⁴ thay cho 2.06e+04 (dùng khi ghép số vào chuỗi trong bảng); số vừa phải giữ dạng thường."""
    v = float(v)
    if v == 0 or 1e-3 <= abs(v) < 1e4:
        s = f"{v:.{digits}g}"
        if "e" in s:                      # ví dụ 2859.9 với 2 chữ số có nghĩa: "2.9e+03" -> "2900"
            s = f"{round(v, digits - 1 - int(np.floor(np.log10(abs(v))))):.0f}"
        return s
    m, e = f"{v:.{digits - 1}e}".split("e")
    m = m.rstrip("0").rstrip(".")
    return f"{m}×10{str(int(e)).translate(_SUP_DIGITS)}"


# --- Định dạng giá trị ---
def _is_p(col: str) -> bool:
    c = str(col).lower()
    return bool(re.search(r"(^|[\s(])p([\s)\-:=]|$)|p-value|p \(", c))


def fmt_value(v, col: str = "", spec: str | None = None, latex: bool = False) -> str:
    if v is None or (isinstance(v, float) and np.isnan(v)):
        return "–"
    if isinstance(v, (bool, np.bool_)):
        return "Có" if v else "Không"
    if isinstance(v, (int, np.integer)):
        return f"{v:,}".replace(",", " ") if abs(v) >= 10000 else str(v)
    if isinstance(v, (float, np.floating)):
        v = float(v)
        if spec:
            return spec.format(v)
        if _is_p(col):
            if v < 1e-4:
                return "<0.0001" if not latex else r"$<10^{-4}$"
            return f"{v:.4f}"
        if v == 0:
            return "0"
        a = abs(v)
        if a < 1e-3 or a >= 1e7:
            m, e = f"{v:.3e}".split("e")
            m = m.rstrip("0").rstrip(".")
            return rf"${m}\times10^{{{int(e)}}}$" if latex else f"{m}e{int(e)}"
        if a >= 1e4:
            return f"{v:.0f}"
        if a >= 1e3:
            return f"{v:.1f}"
        return f"{v:.4g}" if a >= 0.1 or a == 0 else f"{v:.4f}".rstrip("0")
    return str(v)


def _has_index_column(df: pd.DataFrame) -> bool:
    """Chỉ số có mang thông tin (cần in thành cột) hay chỉ là RangeIndex mặc định 0, 1, 2, ... không tên.

    Chỉ số kiểu range(1, 7) (độ trễ h = 1..6) cũng là RangeIndex nhưng có nghĩa, nên vẫn được in thành cột.
    """
    idx = df.index
    if isinstance(idx, pd.MultiIndex) or not isinstance(idx, pd.RangeIndex):
        return True
    return idx.name is not None or idx.start != 0 or idx.step != 1


def _prepare(df: pd.DataFrame, index_label: str | None) -> pd.DataFrame:
    d = df.copy()
    if isinstance(d.index, pd.MultiIndex):
        d.index = d.index.set_names([n or f"Chỉ tiêu {i + 1}" for i, n in enumerate(d.index.names)])
        d = d.reset_index()
    elif _has_index_column(d):
        d.index.name = index_label or d.index.name or "Chỉ tiêu"
        d = d.reset_index()
    d.columns = [cap(c) for c in d.columns]
    return d


# Từ đứng đầu tiêu đề cột nhưng là ký hiệu hoặc hàm toán, giữ nguyên chữ thường
_KEEP_LOWER = {"n", "k", "h", "p", "q", "s", "t", "x", "y", "r", "e", "d", "sd", "exp", "log", "log10", "max", "min", "var", "ln"}


def cap(s) -> str:
    """Viết hoa chữ cái đầu của tiêu đề (cả từ tiếng Việt ngắn như "bộ", "từ"), trừ khi từ đầu là ký hiệu toán (n, k, h, p, max, exp…)."""
    s = str(s)
    m = re.match(r"[A-Za-zÀ-ỹ0-9]+", s)
    if not m or not s[:1].isalpha() or "Ͱ" <= s[:1] <= "Ͽ" or not s[:1].islower() or m.group(0) in _KEEP_LOWER or s[:1] in "`$":
        return s
    return s[:1].upper() + s[1:]


def to_markdown(df: pd.DataFrame, caption: str | None = None, fmt: dict | None = None, index_label: str | None = None,
                note: str | None = None) -> str:
    d = _prepare(df, index_label)
    fmt = fmt or {}
    esc = lambda s: str(s).replace("|", "\\|")  # noqa: E731
    rows = ["| " + " | ".join(esc(c) for c in d.columns) + " |", "|" + "---|" * len(d.columns)]
    for r in d.itertuples(index=False):
        rows.append("| " + " | ".join(esc(fmt_value(v, c, fmt.get(c))) for v, c in zip(r, d.columns)) + " |")
    head = f"**{caption}**\n\n" if caption else ""
    tail = f"\n*Ghi chú:* {note}\n" if note else ""
    return head + "\n".join(rows) + "\n" + tail


def _to_latex_tall(d: pd.DataFrame, cells: list, n_idx: int, name: str, caption: str, note: str | None) -> str:
    """Bảng rộng mà ít dòng (nhiều đại lượng, vài bộ dữ liệu) được xoay: mỗi hàng một đại lượng, mỗi cột một bộ/mô hình."""
    labels = [cap(", ".join(str(x) for x in row[:n_idx])) for row in cells]
    names = list(d.columns[n_idx:])
    header = " & ".join(tex(c) for c in ["Đại lượng"] + labels) + r" \\"
    body = "\n".join(" & ".join(tex(x) for x in [names[k]] + [row[n_idx + k] for row in cells]) + r" \\" for k in range(len(names)))
    spec = "|" + "|".join("l" + "r" * len(cells)) + "|"
    notes = rf"\par\vspace{{2pt}}{{\footnotesize\textit{{Ghi chú:}} {tex(note)}}}" if note else ""
    return (f"\\begin{{table}}[H]\n\\centering\\footnotesize\n\\caption{{{tex(caption)}}}\\label{{tab:{name}}}\n"
            f"\\begin{{adjustbox}}{{max width=\\linewidth}}\n\\begin{{tabular}}{{{spec}}}\n\\toprule\n{header}\n\\midrule\n{body}\n\\bottomrule\n"
            f"\\end{{tabular}}\n\\end{{adjustbox}}{notes}\n\\end{{table}}\n")


def to_latex(df: pd.DataFrame, name: str, caption: str, fmt: dict | None = None, index_label: str | None = None,
             note: str | None = None) -> str:
    """Bảng booktabs, tự co về vừa bề rộng trang; cột chữ dài dùng p{}; có caption, nhãn `tab:<name>` và ghi chú."""
    d = _prepare(df, index_label)
    fmt = fmt or {}
    cells = [[fmt_value(v, c, fmt.get(c), latex=True) for v, c in zip(r, d.columns)] for r in d.itertuples(index=False)]
    ncol = len(d.columns)
    n_idx = df.index.nlevels if _has_index_column(df) else 0
    if ncol >= 9 and len(cells) <= 4 and n_idx >= 1:
        return _to_latex_tall(d, cells, n_idx, name, caption, note)
    plain = lambda s: re.sub(r"\$.*?\$", "x", str(s))  # noqa: E731  (độ dài hiển thị, bỏ phần toán)
    maxlen = [max(len(plain(c)), *(len(plain(row[j])) for row in cells)) for j, c in enumerate(d.columns)]
    numeric = [pd.api.types.is_numeric_dtype(d.iloc[:, j]) and not pd.api.types.is_bool_dtype(d.iloc[:, j]) for j in range(ncol)]
    long_text = [j for j in range(ncol) if not numeric[j] and maxlen[j] > 28]
    share = {j: min(maxlen[j], 60) for j in long_text}
    frac = 0.97 if len(long_text) == ncol else 0.72
    cols = []
    for j in range(ncol):
        if j in long_text:
            w = frac * share[j] / sum(share.values())
            if len(long_text) < ncol:      # cột chữ cạnh các cột số: không rộng hơn mức chữ cần (≈ 0.0105 bề rộng mỗi ký tự)
                w = min(w, 0.0105 * share[j])
            cols.append(r">{\raggedright\arraybackslash}p{\dimexpr " + f"{w:.3f}" + r"\linewidth-2\tabcolsep\relax}")
        else:
            cols.append("r" if numeric[j] else ("c" if pd.api.types.is_bool_dtype(d.iloc[:, j]) else "l"))
    spec = "|" + "|".join(cols) + "|"   # có đường kẻ dọc giữa các cột
    header = " & ".join(tex(c) for c in d.columns) + r" \\"
    end = r" \\[3pt]" if long_text else r" \\"   # bảng có cột chữ dài: thêm chút khoảng cách giữa các hàng
    body = "\n".join(" & ".join(tex(x) for x in row) + end for row in cells)
    size = r"\scriptsize" if ncol >= 9 else r"\footnotesize"
    notes = rf"\par\vspace{{2pt}}{{\footnotesize\textit{{Ghi chú:}} {tex(note)}}}" if note else ""
    return (f"\\begin{{table}}[H]\n\\centering{size}\n\\caption{{{tex(caption)}}}\\label{{tab:{name}}}\n"
            f"\\begin{{adjustbox}}{{max width=\\linewidth}}\n\\begin{{tabular}}{{{spec}}}\n\\toprule\n{header}\n\\midrule\n{body}\n\\bottomrule\n"
            f"\\end{{tabular}}\n\\end{{adjustbox}}{notes}\n\\end{{table}}\n")


def save_table(df: pd.DataFrame, name: str, caption: str | None = None, fmt: dict | None = None, index_label: str | None = None,
               note: str | None = None) -> pd.DataFrame:
    """Lưu df thành outputs/tables/<name>.csv / .md / .tex; caption, nhãn chỉ số, ghi chú lấy từ captions.ARTIFACTS nếu không truyền.

    Trả về df để hiển thị trong notebook.
    """
    from .captions import ARTIFACTS

    meta = ARTIFACTS.get(name, {})
    caption = caption or meta.get("caption")
    if caption is None:
        raise KeyError(f"Bảng '{name}' chưa có caption trong as3179ts/captions.py")
    index_label = index_label or meta.get("index")
    note = note or meta.get("note")
    fmt = {**meta.get("fmt", {}), **(fmt or {})}
    TABLE_DIR.mkdir(parents=True, exist_ok=True)
    out = df.reset_index() if _has_index_column(df) else df
    if index_label and out.columns[0] != index_label and not isinstance(df.index, pd.MultiIndex) and df.index.name is None:
        out = out.rename(columns={out.columns[0]: index_label})
    out.to_csv(TABLE_DIR / f"{name}.csv", index=False, encoding="utf-8-sig")
    (TABLE_DIR / f"{name}.md").write_text(to_markdown(df, caption, fmt, index_label, note), encoding="utf-8")
    (TABLE_DIR / f"{name}.tex").write_text(to_latex(df, name, caption, fmt, index_label, note), encoding="utf-8")
    return df

In [ ]:
%%hocphan plotting
"""Style hình thống nhất và lưu hình vào outputs/figures/ (PNG 300 dpi, tên fig_b<bài>_<mục>_<nội dung>)."""
import matplotlib

matplotlib.use("Agg")  # chạy không cần màn hình (nbconvert, run_all)
import matplotlib.pyplot as plt  # noqa: E402
import numpy as np  # noqa: E402
from cycler import cycler  # noqa: E402
from matplotlib.ticker import MaxNLocator  # noqa: E402

from .config import FIG_DIR  # noqa: E402

# Bảng màu Okabe–Ito: phân biệt được khi in đen trắng và với người mù màu
BLUE, ORANGE, GREEN, RED, PURPLE, SKY, GREY = "#0072B2", "#E69F00", "#009E73", "#D55E00", "#CC79A7", "#56B4E9", "#6b6b6b"

plt.rcParams.update({
    "figure.figsize": (7.5, 3.8), "figure.facecolor": "white", "savefig.facecolor": "white",
    "axes.prop_cycle": cycler(color=[BLUE, ORANGE, GREEN, RED, PURPLE, SKY]),
    "axes.grid": True, "grid.alpha": 0.25, "grid.linewidth": 0.6, "axes.spines.top": False, "axes.spines.right": False,
    "axes.titlesize": 11, "axes.titleweight": "bold", "axes.labelsize": 10, "xtick.labelsize": 9, "ytick.labelsize": 9,
    "legend.fontsize": 8.5, "legend.frameon": True, "legend.framealpha": 0.9, "legend.edgecolor": "#cccccc",
    "lines.linewidth": 1.3, "font.size": 10, "axes.unicode_minus": True,
})


def save_figure(fig, name: str) -> str:
    """Lưu fig thành outputs/figures/<name>.png (300 dpi, nền trắng); trả về đường dẫn."""
    from .captions import caption_of

    caption_of(name)  # mọi hình phải có chú thích trong captions.py
    FIG_DIR.mkdir(parents=True, exist_ok=True)
    fig.tight_layout()
    path = FIG_DIR / f"{name}.png"
    fig.savefig(path, dpi=300, bbox_inches="tight")
    plt.close(fig)
    return str(path)


def show_figure(path: str):
    """Hiển thị hình đã lưu trong notebook (backend Agg không tự vẽ inline)."""
    import io

    from IPython.display import Image, display
    from PIL import Image as PILImage

    # nhúng bản thu nhỏ (rộng ≤ 1600 px) để notebook nhẹ; file PNG 300 dpi gốc vẫn nằm ở outputs/figures
    img = PILImage.open(path)
    img.thumbnail((1600, 1600))
    buf = io.BytesIO()
    img.save(buf, format="PNG", optimize=True)
    display(Image(data=buf.getvalue(), width=760))


def panel_label(ax, text: str) -> None:
    """Nhãn panel đậm (a), (b), … ở góc trên trái, ngoài khung."""
    ax.text(-0.02, 1.12, text, transform=ax.transAxes, fontsize=12, fontweight="bold", va="bottom", ha="right")


def acf_bars(ax, values, n: int, title: str = "", band: float | None = None, start: int = 1, ylabel: str = "ACF",
             band_label: str | None = None, highlight: bool = True) -> None:
    """Biểu đồ ACF/PACF dạng que-chấm từ độ trễ `start`, kèm dải ±band (mặc định iid ±1.96/√n).

    Độ trễ vượt dải được tô màu cam đỏ; trục độ trễ chỉ có tick nguyên.
    """
    values = np.asarray(values, dtype=float)
    band = 1.96 / np.sqrt(n) if band is None else band
    lags = np.arange(start, len(values))
    v = values[start:]
    out = np.abs(v) > band if highlight else np.zeros(len(v), bool)
    ax.vlines(lags, 0, v, color=np.where(out, RED, BLUE), lw=2.0, zorder=3)
    ax.plot(lags, v, "o", ms=3, color=BLUE, zorder=4, mfc=BLUE)
    if out.any():
        ax.plot(lags[out], v[out], "o", ms=3, color=RED, zorder=5)
    ax.axhline(0, color="k", lw=0.8)
    ax.axhline(band, color=GREY, ls="--", lw=1)
    ax.axhline(-band, color=GREY, ls="--", lw=1)
    top = max(np.abs(v).max(), band) * 1.12
    ax.set_ylim(-top, top)
    last = len(values) - 1
    step = 1 if last <= 12 else (2 if last <= 24 else 5)
    ax.set_xticks(list(range(step, last + 1, step)) if step > 1 else list(range(start, last + 1)))
    ax.set_xlim(start - 0.8, last + 0.8)
    # thông tin dải đặt ở nhãn trục x (không dùng legend để không đè lên cột)
    ax.set(title=title, xlabel=f"độ trễ h    (- - - : {band_label or f'dải 95% ±{band:.3f}'})", ylabel=ylabel)


def overlay_theory(ax, lags, values, label: str):
    """Chồng đường lý thuyết lên biểu đồ ACF mẫu."""
    ax.plot(lags, values, "-o", color=ORANGE, ms=3.5, lw=1.4, label=label, zorder=6)
    ax.legend(loc="best")

## Dữ liệu đầu vào và tiền xử lý

Kiểm tra SHA-256 của năm CSV trước khi tạo sáu chuỗi HU, LY, DR, NI, AP và DP.

In [ ]:
from as3179ts.config import DATA_RAW
from hashlib import sha256

raw_sha256 = {'LakeHuron.csv': '7fe634970d5b28128ade898c3932dbc4015fecbeb6dabc443efaf88a2fd037d8', 'lynx.csv': '0766d4043eb8a20ac83afefff1c78063beff3891571c59e8e38d72503ca86cf1', 'EuStockMarkets.csv': 'aa5513f3ca55daf51294accae9a3f946e957ba53f405dc4fafb4e125e5c03818', 'Nile.csv': 'd0452bea38c61e796a4eeb950bf91d20fb5c7f13d5822eadf5990fe54f9c8d07', 'AirPassengers.csv': 'bdb98adbd418a6de6842a742e0602f363c3b841c26677e7e61a1e055e9509bd8'}
for filename, expected in raw_sha256.items():
    path = DATA_RAW / filename
    if sha256(path.read_bytes()).hexdigest() != expected:
        raise ValueError(f"CSV khác bản gốc: {path}")

In [ ]:
"""Sinh 6 file data/processed/*.csv từ data/raw/ theo đúng biến đổi của bài giảng, kèm kiểm tra và bảng tóm tắt."""
import numpy as np
import pandas as pd

from as3179ts.config import (
    AP_TEST_N, AP_TRAIN_N, DAX_FREQ, DAX_START_TIME, DATA_PROCESSED, DATA_RAW, DP_TEST_N, DP_TRAIN_N,
)


def check(df: pd.DataFrame, name: str, n: int, time_col: str) -> None:
    """Kiểm tra số dòng, NA, trùng lặp, đơn điệu thời gian."""
    assert len(df) == n, f"{name}: {len(df)} dòng, kỳ vọng {n}"
    assert not df.isna().any().any(), f"{name}: có NA"
    assert not df.duplicated().any(), f"{name}: có dòng trùng"
    assert df[time_col].is_monotonic_increasing and df[time_col].is_unique, f"{name}: thời gian không đơn điệu"


def main() -> None:
    DATA_PROCESSED.mkdir(parents=True, exist_ok=True)
    raw = lambda n: pd.read_csv(DATA_RAW / f"{n}.csv")  # noqa: E731

    # HU: không biến đổi; x = năm - 1920
    d = raw("LakeHuron")
    hu = pd.DataFrame({"year": d["time"].astype(int), "level_ft": d["value"]})
    hu["x_centered"] = hu["year"] - 1920
    check(hu, "HU", 98, "year"); assert hu.year.iloc[0] == 1875 and hu.year.iloc[-1] == 1972

    # LY: log10(count)
    d = raw("lynx")
    ly = pd.DataFrame({"year": d["time"].astype(int), "count": d["value"].astype(int)})
    assert (ly["count"] > 0).all(), "LY: count <= 0 trước khi log"
    ly["log10_count"] = np.log10(ly["count"])
    check(ly, "LY", 114, "year"); assert ly.year.iloc[0] == 1821 and ly.year.iloc[-1] == 1934

    # DAX: chỉ chỉ số ngày giao dịch t; time đúng như time() của R (start = c(1991,130), freq 260). KHÔNG có ngày lịch.
    dax = raw("EuStockMarkets")["DAX"].to_numpy()
    assert len(dax) == 1860 and (dax > 0).all()
    t = np.arange(1, len(dax) + 1)
    time = DAX_START_TIME + (t - 1) / DAX_FREQ
    log_p = np.log(dax)

    dr = pd.DataFrame({"t": t[1:], "time": time[1:], "dax_close": dax[1:], "ret_pct": 100 * np.diff(log_p)})
    check(dr, "DR", 1859, "t")

    dp = pd.DataFrame({"t": t, "time": time, "dax_close": dax, "log_close": log_p})
    dp["split"] = np.where(dp["t"] <= DP_TRAIN_N, "train", "test")
    check(dp, "DP", DP_TRAIN_N + DP_TEST_N, "t")

    # NI: post_1899 để thảo luận đổi mức
    d = raw("Nile")
    ni = pd.DataFrame({"year": d["time"].astype(int), "flow": d["value"]})
    ni["post_1899"] = (ni["year"] >= 1899).astype(int)
    check(ni, "NI", 100, "year"); assert ni.year.iloc[0] == 1871 and ni.year.iloc[-1] == 1970

    # AP: log; train 1949-01..1958-12, test 1959-01..1960-12
    d = raw("AirPassengers")
    k = np.rint((d["time"].to_numpy() - 1949) * 12).astype(int)  # time là số thực -> chỉ số tháng
    assert (k == np.arange(len(d))).all(), "AP: chỉ số tháng không liên tục"
    ap = pd.DataFrame({"month": pd.period_range("1949-01", periods=len(d), freq="M").astype(str),
                       "passengers": d["value"].astype(int)})
    assert (ap["passengers"] > 0).all()
    ap["log_passengers"] = np.log(ap["passengers"])
    ap["split"] = np.where(np.arange(len(ap)) < AP_TRAIN_N, "train", "test")
    check(ap, "AP", AP_TRAIN_N + AP_TEST_N, "month")
    assert ap.month.iloc[0] == "1949-01" and ap.month.iloc[-1] == "1960-12"

    files = {"HU_lakehuron": hu, "LY_lynx_log10": ly, "DR_dax_returns": dr,
             "NI_nile": ni, "AP_airpassengers": ap, "DP_dax_logprice": dp}
    for name, df in files.items():
        df.to_csv(DATA_PROCESSED / f"{name}.csv", index=False, encoding="utf-8")

    # Bảng tóm tắt
    spec = [("HU", hu, "level_ft", "year"), ("LY", ly, "log10_count", "year"), ("DR", dr, "ret_pct", "t"),
            ("NI", ni, "flow", "year"), ("AP", ap, "passengers", "month"), ("DP", dp, "log_close", "t")]
    rows = [(c, len(df), f"{df[tc].iloc[0]}..{df[tc].iloc[-1]}", v, df[v].mean(), df[v].std(ddof=1))
            for c, df, v, tc in spec]
    print(pd.DataFrame(rows, columns=["bộ", "n", "thời gian", "biến", "trung bình", "SD"]).to_string(index=False))
    print("\nĐối chiếu với bài giảng:")
    print(f"  HU trung bình {hu.level_ft.mean():.1f} (579.0), SD {hu.level_ft.std():.2f} (1.32)")
    print(f"  DR trung bình {dr.ret_pct.mean():.3f} (0.065), SD {dr.ret_pct.std():.2f} (1.03)")
    print(f"  NI trung bình {ni.flow.mean():.0f} (919), SD {ni.flow.std():.0f} (169), "
          f"trước 1899: {ni.flow[ni.year < 1899].mean():.0f} (1098)")
    print(f"  DAX cuối {dax[-1]:.2f} (5473.72)")

main()

# Bài tập lớn AS3179: khám phá dữ liệu (Mục 4.3)

Chỉ những gì cần cho mô hình: đồ thị chuỗi, ACF/PACF (tự cài đặt, `as3179ts.acf_tools`) và thống kê mô tả của 6 bộ dữ liệu đã làm sạch. Nguồn, đơn vị và các bước làm sạch: `data/DATA_SOURCES.md`. Nhận xét từng bộ được đồng bộ từ báo cáo, mục 3.2(b).

In [ ]:
import warnings; warnings.filterwarnings("ignore")
import sys
import numpy as np, pandas as pd, scipy, statsmodels, matplotlib
import matplotlib.pyplot as plt
from IPython.display import display
from scipy import stats

from as3179ts.config import SEED
from as3179ts.data import load_processed
from as3179ts import acf_tools as A
from as3179ts.plotting import save_figure, show_figure, acf_bars, panel_label, BLUE, ORANGE, GREEN, RED, GREY
from as3179ts.reporting import save_table

print("Python", sys.version.split()[0], "| numpy", np.__version__, "| pandas", pd.__version__, "| scipy", scipy.__version__,
      "| statsmodels", statsmodels.__version__, "| matplotlib", matplotlib.__version__)
print("SEED =", SEED, "(notebook này không dùng số ngẫu nhiên)")
CODES = ("HU", "LY", "DR", "NI", "AP", "DP")
DATA = {c: load_processed(c) for c in CODES}
# (biến cần mô hình hóa, nhãn trục, cột thời gian)
SPEC = {"HU": ("level_ft", "mực nước (ft)", "year"), "LY": ("log10_count", "log10(số linh miêu)", "year"), "DR": ("ret_pct", "suất sinh lợi (%)", "t"),
        "NI": ("flow", "lưu lượng (10⁸ m³)", "year"), "AP": ("log_passengers", "log(hành khách)", "month"), "DP": ("log_close", "log DAX", "t")}
TIME = {"HU": DATA["HU"].year.to_numpy(), "LY": DATA["LY"].year.to_numpy(), "DR": DATA["DR"].t.to_numpy(), "NI": DATA["NI"].year.to_numpy(),
        "AP": 1949 + np.arange(144) / 12, "DP": DATA["DP"].t.to_numpy()}   # trục thời gian thật (AP: năm thập phân; DAX: ngày giao dịch)
XLAB = {"HU": "năm", "LY": "năm", "DR": "ngày giao dịch t", "NI": "năm", "AP": "năm", "DP": "ngày giao dịch t"}
TITLE = {"HU": "HU: mực nước hồ Huron", "LY": "LY: số linh miêu bị bẫy (log₁₀)", "DR": "DR: suất sinh lợi log ngày của DAX",
         "NI": "NI: lưu lượng sông Nile tại Aswan", "AP": "AP: số hành khách hàng không (log)", "DP": "DP: log giá đóng cửa DAX"}
pd.options.display.float_format = lambda v: f"{v:.4g}"

## Thống kê mô tả

In [ ]:
rows = []
for c in CODES:
    col, _, tcol = SPEC[c]; v = DATA[c][col].to_numpy(); t = DATA[c][tcol]
    rows.append({"bộ": c, "biến": col, "n": len(v), "từ": t.iloc[0], "đến": t.iloc[-1], "trung bình": v.mean(), "SD": v.std(ddof=1),
                 "min": v.min(), "max": v.max(), "độ lệch": stats.skew(v), "kurtosis dư": stats.kurtosis(v)})
tab_desc = save_table(pd.DataFrame(rows).set_index("bộ"), "tab_eda_descriptive")

In [ ]:
tab_desc

## HU

Mực nước hồ Huron 1875–1972 (Bài 11, ví dụ (i)); đường gạch là xu thế tuyến tính OLS theo $x=$ năm $-1920$.

In [ ]:
CODE = "HU"
fig, axes = plt.subplots(1, 3, figsize=(13.5, 3.9), gridspec_kw={"width_ratios": [1.6, 1, 1]}); y = DATA[CODE][SPEC[CODE][0]].to_numpy(); n = len(y); tt = TIME[CODE]
axes[0].plot(tt, y, lw=0.9, color=BLUE); axes[0].set(title=TITLE[CODE], xlabel=XLAB[CODE], ylabel=SPEC[CODE][1])
x = DATA["HU"].x_centered.to_numpy(float); axes[0].plot(tt, np.polyval(np.polyfit(x, y, 1), x), color=RED, ls="--", lw=1.6, label="xu thế tuyến tính (OLS)"); axes[0].legend(loc="upper right")
acf_bars(axes[1], A.sample_acf(y, 30), n, f"{CODE}: ACF mẫu", ylabel="ACF"); acf_bars(axes[2], A.pacf_durbin_levinson(A.sample_acf(y, 30), 30), n, f"{CODE}: PACF mẫu", ylabel="PACF")
for k, ax in enumerate(axes): panel_label(ax, "(" + "abc"[k] + ")")
fig_path = save_figure(fig, f"fig_eda_{CODE}_overview")

In [ ]:
show_figure(fig_path)

**Nhận xét (HU):** mực nước quanh $579$ ft, độ lệch chuẩn $1.32$ ft, có xu thế giảm; ACF giảm dần.

*Đồng bộ từ báo cáo, mục 3.2(b); việc chuỗi nào dừng được kiểm chứng ở Chương 4 và 5.*

## LY

Số linh miêu bị bẫy hằng năm, $y=\log_{10}$(số lượng) (Bài 11, ví dụ (ii)); chu kỳ ≈ 10 năm.

In [ ]:
CODE = "LY"
fig, axes = plt.subplots(1, 3, figsize=(13.5, 3.9), gridspec_kw={"width_ratios": [1.6, 1, 1]}); y = DATA[CODE][SPEC[CODE][0]].to_numpy(); n = len(y); tt = TIME[CODE]
axes[0].plot(tt, y, lw=0.9, color=BLUE); axes[0].set(title=TITLE[CODE], xlabel=XLAB[CODE], ylabel=SPEC[CODE][1])

acf_bars(axes[1], A.sample_acf(y, 30), n, f"{CODE}: ACF mẫu", ylabel="ACF"); acf_bars(axes[2], A.pacf_durbin_levinson(A.sample_acf(y, 30), 30), n, f"{CODE}: PACF mẫu", ylabel="PACF")
for k, ax in enumerate(axes): panel_label(ax, "(" + "abc"[k] + ")")
fig_path = save_figure(fig, f"fig_eda_{CODE}_overview")

In [ ]:
show_figure(fig_path)

**Nhận xét (LY):** dao động theo chu kỳ rõ; ACF đổi dấu xen kẽ theo chu kỳ đó.

*Đồng bộ từ báo cáo, mục 3.2(b); việc chuỗi nào dừng được kiểm chứng ở Chương 4 và 5.*

## DR

Suất sinh lợi log của DAX $r_t=100\,\Delta\log P_t$ (Bài 11, ví dụ (iii)); chỉ số $t$ là ngày giao dịch, **không có ngày lịch**.

In [ ]:
CODE = "DR"
fig, axes = plt.subplots(1, 3, figsize=(13.5, 3.9), gridspec_kw={"width_ratios": [1.6, 1, 1]}); y = DATA[CODE][SPEC[CODE][0]].to_numpy(); n = len(y); tt = TIME[CODE]
axes[0].plot(tt, y, lw=0.9, color=BLUE); axes[0].set(title=TITLE[CODE], xlabel=XLAB[CODE], ylabel=SPEC[CODE][1])
axes[0].axhline(0, color="k", lw=0.6)
acf_bars(axes[1], A.sample_acf(y, 30), n, f"{CODE}: ACF mẫu", ylabel="ACF"); acf_bars(axes[2], A.pacf_durbin_levinson(A.sample_acf(y, 30), 30), n, f"{CODE}: PACF mẫu", ylabel="PACF")
for k, ax in enumerate(axes): panel_label(ax, "(" + "abc"[k] + ")")
fig_path = save_figure(fig, f"fig_eda_{CODE}_overview")

In [ ]:
show_figure(fig_path)

**Nhận xét (DR):** trung bình gần $0$, chỉ $0.065%$, độ lệch chuẩn $1.03%$, đuôi nặng với kurtosis dư $6.28$ và hơi lệch âm với độ lệch $-0.55$; biến động tụ thành cụm, còn ACF hầu như nằm trong dải $95%$.

*Đồng bộ từ báo cáo, mục 3.2(b); việc chuỗi nào dừng được kiểm chứng ở Chương 4 và 5.*

## NI

Lưu lượng sông Nile tại Aswan 1871–1970 (Bài 12, ví dụ (i)); đường đỏ là trung bình trước/sau 1899.

In [ ]:
CODE = "NI"
fig, axes = plt.subplots(1, 3, figsize=(13.5, 3.9), gridspec_kw={"width_ratios": [1.6, 1, 1]}); y = DATA[CODE][SPEC[CODE][0]].to_numpy(); n = len(y); tt = TIME[CODE]
axes[0].plot(tt, y, lw=0.9, color=BLUE); axes[0].set(title=TITLE[CODE], xlabel=XLAB[CODE], ylabel=SPEC[CODE][1])
m1, m2 = y[tt < 1899].mean(), y[tt >= 1899].mean(); axes[0].axvline(1899, color=GREY, ls="--", lw=1.2); axes[0].hlines([m1, m2], [tt[0], 1899], [1899, tt[-1]], color=RED, lw=1.8, label=f"trung bình {m1:.0f} (trước 1899) và {m2:.0f} (từ 1899)"); axes[0].legend(loc="upper right", fontsize=8)
acf_bars(axes[1], A.sample_acf(y, 30), n, f"{CODE}: ACF mẫu", ylabel="ACF"); acf_bars(axes[2], A.pacf_durbin_levinson(A.sample_acf(y, 30), 30), n, f"{CODE}: PACF mẫu", ylabel="PACF")
for k, ax in enumerate(axes): panel_label(ax, "(" + "abc"[k] + ")")
fig_path = save_figure(fig, f"fig_eda_{CODE}_overview")

In [ ]:
show_figure(fig_path)

**Nhận xét (NI):** lưu lượng trung bình giảm từ $1098$ trước năm 1899 xuống $850$ từ năm 1899, tức một lần đổi mức.

*Đồng bộ từ báo cáo, mục 3.2(b); việc chuỗi nào dừng được kiểm chứng ở Chương 4 và 5.*

## AP

Hành khách hàng không, $\log$ (Bài 12, ví dụ (ii)); 120 tháng huấn luyện | 24 tháng kiểm định.

In [ ]:
CODE = "AP"
fig, axes = plt.subplots(1, 3, figsize=(13.5, 3.9), gridspec_kw={"width_ratios": [1.6, 1, 1]}); y = DATA[CODE][SPEC[CODE][0]].to_numpy(); n = len(y); tt = TIME[CODE]
axes[0].plot(tt, y, lw=0.9, color=BLUE); axes[0].set(title=TITLE[CODE], xlabel=XLAB[CODE], ylabel=SPEC[CODE][1])
axes[0].axvline(1959, color=GREY, ls="--", lw=1.2); axes[0].text(1959.1, axes[0].get_ylim()[0] + 0.05, "kiểm định →", fontsize=8, color=GREY); axes[0].text(1958.9, axes[0].get_ylim()[0] + 0.05, "← huấn luyện", fontsize=8, color=GREY, ha="right")
acf_bars(axes[1], A.sample_acf(y, 30), n, f"{CODE}: ACF mẫu", ylabel="ACF"); acf_bars(axes[2], A.pacf_durbin_levinson(A.sample_acf(y, 30), 30), n, f"{CODE}: PACF mẫu", ylabel="PACF")
for k, ax in enumerate(axes): panel_label(ax, "(" + "abc"[k] + ")")
fig_path = save_figure(fig, f"fig_eda_{CODE}_overview")

In [ ]:
show_figure(fig_path)

**Nhận xét (AP):** xu thế tăng và mùa vụ rõ; ACF giảm rất chậm.

*Đồng bộ từ báo cáo, mục 3.2(b); việc chuỗi nào dừng được kiểm chứng ở Chương 4 và 5.*

## DP

Log giá đóng cửa DAX (Bài 12, ví dụ (iii)); 1500 ngày huấn luyện | 360 ngày kiểm định; chỉ số ngày giao dịch.

In [ ]:
CODE = "DP"
fig, axes = plt.subplots(1, 3, figsize=(13.5, 3.9), gridspec_kw={"width_ratios": [1.6, 1, 1]}); y = DATA[CODE][SPEC[CODE][0]].to_numpy(); n = len(y); tt = TIME[CODE]
axes[0].plot(tt, y, lw=0.9, color=BLUE); axes[0].set(title=TITLE[CODE], xlabel=XLAB[CODE], ylabel=SPEC[CODE][1])
axes[0].axvline(1500, color=GREY, ls="--", lw=1.2); axes[0].text(1520, axes[0].get_ylim()[0] + 0.05, "kiểm định →", fontsize=8, color=GREY); axes[0].text(1480, axes[0].get_ylim()[0] + 0.05, "← huấn luyện", fontsize=8, color=GREY, ha="right")
acf_bars(axes[1], A.sample_acf(y, 30), n, f"{CODE}: ACF mẫu", ylabel="ACF"); acf_bars(axes[2], A.pacf_durbin_levinson(A.sample_acf(y, 30), 30), n, f"{CODE}: PACF mẫu", ylabel="PACF")
for k, ax in enumerate(axes): panel_label(ax, "(" + "abc"[k] + ")")
fig_path = save_figure(fig, f"fig_eda_{CODE}_overview")

In [ ]:
show_figure(fig_path)

**Nhận xét (DP):** tăng dài hạn; ACF gần $1$ ở mọi độ trễ được vẽ và PACF chỉ đáng kể ở độ trễ $1$.

*Đồng bộ từ báo cáo, mục 3.2(b); việc chuỗi nào dừng được kiểm chứng ở Chương 4 và 5.*

# Bài 11 – Chuỗi thời gian I: phụ thuộc và mô hình ARMA

Notebook kiểm chứng **Bài 11** trên 3 bộ dữ liệu HU (Huron), LY (lynx), DR (suất sinh lợi DAX). Mỗi mục có đúng 6 tiểu mục theo đặc tả: *Nội dung lý thuyết*, *Cách triển khai*,
*Mã lệnh Python*, *Kết quả*, *Phân tích và diễn giải*, *Nhận xét*. Mọi con số ở đây do code sinh ra; hai tiểu mục cuối được đồng bộ từ báo cáo (`report/chapters/`).

Ký hiệu: **[D]** tính trên dữ liệu thật; **[M]** mô phỏng có điều khiển (seed 3179, `S.child_rng(...)`, kèm MCSE).
Hình lưu ở `outputs/figures/` (PNG 300 dpi), bảng ở `outputs/tables/` (csv, md, tex; tên ổn định).

*Cảnh báo của statsmodels* (khởi tạo tham số MA, hội tụ, nội suy p-value của KPSS) được ẩn có chủ ý; bài giảng cũng ghi nhận chúng không ảnh hưởng các con số.

In [ ]:
import warnings; warnings.filterwarnings("ignore")
import sys
import numpy as np, pandas as pd, scipy, statsmodels, matplotlib
import statsmodels.api as sm
import matplotlib.pyplot as plt
from IPython.display import display
from scipy import stats
from scipy.linalg import toeplitz
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.ar_model import AutoReg
from statsmodels.tsa.stattools import acf, pacf
from statsmodels.tsa.arima_process import arma2ma, arma_acf
from statsmodels.stats.diagnostic import acorr_ljungbox
from statsmodels.regression.linear_model import yule_walker

from as3179ts.config import SEED
from as3179ts.data import load_processed
from as3179ts import acf_tools as A, arma_tools as R, forecast_tools as F, evaluation as E, simulation as S
from as3179ts.plotting import save_figure, show_figure, acf_bars, panel_label, overlay_theory, BLUE, ORANGE, GREEN, RED, PURPLE, GREY
from as3179ts.reporting import save_table
pd.options.display.float_format = lambda v: f"{v:.4g}"
warnings.filterwarnings("ignore")  # statsmodels bật lại cảnh báo khi import nên tắt lại ở đây

print("Python", sys.version.split()[0], "| numpy", np.__version__, "| pandas", pd.__version__, "| scipy", scipy.__version__,
      "| statsmodels", statsmodels.__version__, "| matplotlib", matplotlib.__version__)
print("SEED =", SEED)
rng = np.random.default_rng(SEED)  # generator gốc; mỗi mục dùng S.child_rng(11, k) nên kết quả không phụ thuộc thứ tự chạy ô

DATA = {c: load_processed(c) for c in ("HU", "LY", "DR")}
NAMES = {"HU": "Hồ Huron", "LY": "Linh miêu (log10)", "DR": "Suất sinh lợi DAX"}
Y = {"HU": DATA["HU"].level_ft.to_numpy(), "LY": DATA["LY"].log10_count.to_numpy(), "DR": DATA["DR"].ret_pct.to_numpy()}
X_HU = DATA["HU"].x_centered.to_numpy(dtype=float)            # xu thế: x = năm − 1920
U_HU = sm.OLS(Y["HU"], sm.add_constant(X_HU)).fit().resid      # HU đã khử xu thế tuyến tính bằng OLS
Z = {"HU": U_HU, "LY": Y["LY"], "DR": Y["DR"]}                 # chuỗi dùng để đọc ACF/PACF (HU đã khử xu thế)
TIME = {"HU": DATA["HU"].year.to_numpy(), "LY": DATA["LY"].year.to_numpy(), "DR": DATA["DR"].t.to_numpy()}   # trục thời gian thật
XLAB = {"HU": "năm", "LY": "năm", "DR": "ngày giao dịch t"}
YLAB = {"HU": "mực nước (ft)", "LY": "log₁₀(số linh miêu)", "DR": "suất sinh lợi (%)"}


def fit_arma(code, order, **kw):
    """ARMA(p,q) bằng ML Gauss. HU có xu thế tuyến tính trong exog (như bài giảng, Mục 6)."""
    return ARIMA(Y[code], exog=X_HU if code == "HU" else None, order=order).fit(**kw)


def sig2(res):
    """σ̂² của ARIMAResults (nằm trong params, không phải thuộc tính)."""
    return float(dict(zip(res.param_names, res.params))["sigma2"])


def lb(resid, H, model_df=0):
    """(Q_LB, p-value) bằng hàm thư viện, để đối chiếu với ljung_box_manual."""
    r = acorr_ljungbox(resid, lags=[H], model_df=model_df)
    return float(r["lb_stat"].iloc[0]), float(r["lb_pvalue"].iloc[0])

## 11.1 Chuỗi thời gian, dừng và nhiễu trắng

*Bộ dữ liệu:* HU, LY, DR

### Nội dung lý thuyết

**Định nghĩa 1–2.** Chuỗi $\{Y_t\}$ dừng yếu nếu $\mu_t\equiv\mu$ và $\gamma(t+h,t)=\gamma(h)$ chỉ phụ thuộc độ trễ $h$.
Nhiễu trắng $WN(0,\sigma^2)$ chỉ yêu cầu không tương quan; nhiễu iid yêu cầu độc lập và cùng phân phối, nên *mạnh hơn*.
Không tương quan chưa suy ra độc lập: $r_t$ có thể không tự tương quan trong khi $r_t^2$ thì có (cụm biến động).

### Cách triển khai

Với mỗi bộ: vẽ chuỗi, trung bình trượt và phương sai trượt (cửa sổ 20 năm cho HU, LY; 250 ngày giao dịch cho DR) để
quan sát dấu hiệu không dừng. Sau đó tính $Q_{LB}(10)$ của chuỗi và của bình phương độ lệch so với trung bình. **[D]**
Với HU dùng mức nước gốc (có xu thế); phần khử xu thế xem các mục sau.

### Mã lệnh Python

In [ ]:
WIN = {"HU": 20, "LY": 20, "DR": 250}
fig, axes = plt.subplots(3, 2, figsize=(11, 8.2), gridspec_kw={"width_ratios": [1.5, 1]})
rows = []
for i, c in enumerate(("HU", "LY", "DR")):
    s, w, tt = pd.Series(Y[c]), WIN[c], TIME[c]
    rm, rv = s.rolling(w).mean(), s.rolling(w).var()
    axes[i, 0].plot(tt, s.values, lw=0.7, color=BLUE, label="chuỗi quan sát"); axes[i, 0].plot(tt, rm.values, color=RED, lw=1.8, label=f"trung bình trượt ({w})")
    axes[i, 0].set(title=f"{NAMES[c]}: chuỗi và trung bình trượt", xlabel=XLAB[c], ylabel=YLAB[c]); axes[i, 0].legend(loc="best", ncol=2, fontsize=8)
    axes[i, 1].plot(tt, rv.values, color=GREEN, lw=1.6, label=f"phương sai trượt ({w})")
    axes[i, 1].set(title=f"{c}: phương sai trượt", xlabel=XLAB[c], ylabel="phương sai"); axes[i, 1].legend(loc="best", fontsize=8)
    panel_label(axes[i, 0], "(" + "abc"[i] + ")")
    sq = Y[c] ** 2 if c == "DR" else (Y[c] - Y[c].mean()) ** 2
    rows.append({"bộ": c, "n": len(s), "cửa sổ": w, "SD của TB trượt / SD của y": rm.std() / s.std(),
                 "phương sai trượt max/min": rv.max() / rv.min(), "ρ̂(1)": A.sample_acf(Y[c], 1)[1],
                 "p-value Q_LB(10) của y": A.ljung_box_manual(Y[c], 10)[1], "p-value Q_LB(10) của y²": A.ljung_box_manual(sq, 10)[1]})
fig_path = save_figure(fig, "fig_b11_01_series_rolling")
tab = save_table(pd.DataFrame(rows).set_index("bộ"), "tab_b11_01_stationarity")

### Kết quả

In [ ]:
show_figure(fig_path)
tab

### Phân tích và diễn giải

*Đồng bộ từ báo cáo, mục 4.1(a).*

**HU.** Trung bình trượt dao động mạnh, bằng 52% độ lệch chuẩn của chuỗi, và phương sai trượt thay đổi nhiều với tỷ lệ lớn nhất trên nhỏ nhất là $7.6$. Thêm vào đó $\hat\rho(1)=0.83$ và Ljung–Box bác bỏ nhiễu trắng với $p<0.0001$. Các chỉ số này phù hợp với một chuỗi không dừng quanh mức hằng số, nhưng một chuỗi AR(1) dừng với $\phi$ khoảng $0.8$ cũng có thể cho chúng. Bằng chứng chắc hơn nằm ở 4.1(c): hệ số dốc xu thế vẫn khác $0$ ở mức $5%$ ngay cả khi sai số được mô hình hóa bằng AR(2), với $t=-2.65$. Vì vậy HU được coi là dừng quanh một xu thế giảm chứ không quanh một mức hằng số, và các nội dung sau dùng HU đã khử xu thế.

**LY.** Trung bình trượt ổn định hơn nhiều, chỉ bằng 23% độ lệch chuẩn, nên chuỗi gần dừng về mức; nhưng $\hat\rho(1)=0.79$ và $p<0.0001$ cho thấy chuỗi dừng mà không phải nhiễu trắng.

**DR.** $\hat\rho(1)\approx0$ và Ljung–Box không bác bỏ nhiễu trắng với $p=0.78$. Tuy nhiên bình phương suất sinh lợi vẫn tự tương quan rất rõ với $p<0.0001$, và phương sai trượt thay đổi gấp $5.9$ lần: chuỗi phù hợp với nhiễu trắng ở $10$ độ trễ đầu nhưng rõ ràng không iid, vì biến động tụ thành cụm.

### Nhận xét

*Đồng bộ từ báo cáo, mục 4.1(a).*

Kết quả **phù hợp** với lý thuyết: ba chuỗi minh họa đúng ba tình huống, gồm không dừng quanh mức hằng số ở HU, dừng nhưng không phải nhiễu trắng ở LY, và phù hợp với nhiễu trắng nhưng không iid ở DR. Các chỉ số trượt chỉ mang tính tham khảo chứ chưa phải kiểm định chính thức về tính dừng.

## 11.2 Tính chất của hàm tự hiệp phương sai

*Bộ dữ liệu:* HU, LY, DR

### Nội dung lý thuyết

**Mệnh đề 1.** (a) $\gamma(0)\ge0$, $|\gamma(h)|\le\gamma(0)$ nên $|\rho(h)|\le1$; (b) $\gamma(-h)=\gamma(h)$;
(c) $\Gamma_n=[\gamma(i-j)]$ xác định không âm. **Định nghĩa 3:** $\hat\gamma(h)=\frac1n\sum_{t=1}^{n-h}(y_{t+h}-\bar y)(y_t-\bar y)$
(mẫu số $n$) bảo đảm $\hat\Gamma_n$ xác định không âm; mẫu số $n-h$ thì không còn bảo đảm (Bài tập 4(1)).

### Cách triển khai

Với mỗi bộ dựng $\hat\Gamma_n$ đầy đủ ($n\times n$) từ $\hat\gamma$ với mẫu số $n$ và với mẫu số $n-h$; tính trị riêng nhỏ nhất và
$\max_h|\hat\rho(h)|$ cho cả hai. **[D]** $\gamma(-h)=\gamma(h)$ đúng theo cách dựng ma trận Toeplitz đối xứng.

### Mã lệnh Python

In [ ]:
rows, curves = [], {}
for c in ("HU", "LY", "DR"):
    y = Y[c]; n = len(y)
    g_n, g_adj = A.sample_autocov(y, n - 1, "n"), A.sample_autocov(y, n - 1, "n-h")
    curves[c] = (g_n / g_n[0], g_adj / g_adj[0])
    rows.append({"bộ": c, "n": n, "γ̂(0) ≥ 0": bool(g_n[0] >= 0),
                 "max|ρ̂(h)|, mẫu số n": np.abs(curves[c][0]).max(), "max|ρ̂(h)|, mẫu số n−h": np.abs(curves[c][1]).max(),
                 "trị riêng nhỏ nhất của Γ̂, mẫu số n": np.linalg.eigvalsh(toeplitz(g_n)).min(),
                 "trị riêng nhỏ nhất của Γ̂, mẫu số n−h": np.linalg.eigvalsh(toeplitz(g_adj)).min()})
fig, axes = plt.subplots(1, 3, figsize=(12.5, 3.6))
for k, (ax, c) in enumerate(zip(axes, curves)):
    h = np.arange(len(curves[c][0]))
    ax.plot(h, curves[c][1], lw=1.0, color=ORANGE, label="mẫu số n−h"); ax.plot(h, curves[c][0], lw=1.4, color=BLUE, label="mẫu số n")
    ax.axhline(1, color=GREY, ls=":", lw=0.9); ax.axhline(-1, color=GREY, ls=":", lw=0.9, label="|ρ| = 1")
    ax.set(title=f"{c}: ρ̂(h) với h = 0, …, n−1", xlabel="độ trễ h", ylabel="ρ̂(h)"); ax.legend(loc="lower left")
    panel_label(ax, "(" + "abc"[k] + ")")
fig_path = save_figure(fig, "fig_b11_02_acf_denominators")
tab = save_table(pd.DataFrame(rows).set_index("bộ"), "tab_b11_02_gamma_properties")

### Kết quả

In [ ]:
show_figure(fig_path)
tab

### Phân tích và diễn giải

*Đồng bộ từ báo cáo, mục 4.1(b).*

Cả ba chuỗi cho cùng kết luận. Với mẫu số $n$, $|\hat\rho(h)|\le1$ và trị riêng nhỏ nhất dương, bằng $0.0127$ ở HU, $9.4\times10^{-4}$ ở LY và $0.0588$ ở DR. Với mẫu số $n-h$, trị riêng nhỏ nhất âm, lần lượt là $-7.32$, $-0.48$ và $-4.39$, và ở HU và DR còn có $|\hat\rho(h)|>1$, lên tới $1.148$ và $2.001$: ước lượng này không còn là một hàm tự hiệp phương sai hợp lệ.

### Nhận xét

*Đồng bộ từ báo cáo, mục 4.1(b).*

Kết quả **phù hợp** với lý thuyết: chỉ mẫu số $n$ cho ma trận xác định không âm, và đó là lý do Yule–Walker ở mục 4.2 dùng mẫu số $n$.

## 11.3 Trung bình mẫu khi có phụ thuộc

*Bộ dữ liệu:* HU, LY, DR

### Nội dung lý thuyết

**Định lý 1.** $\operatorname{Var}\bar Y=\frac1n\sum_{|h|<n}(1-\frac{|h|}n)\gamma(h)\approx\gamma(0)\tau/n$, $\tau=1+2\sum_{h\ge1}\rho(h)$;
cỡ mẫu hiệu dụng $n_{\rm eff}=n/\tau$. Với AR(1): $\tau=(1+\phi)/(1-\phi)$. Sai số chuẩn iid $s/\sqrt n$ vì vậy quá lạc quan khi $\tau>1$.
Cùng nguyên lý: sai số chuẩn của hệ số dốc xu thế khi sai số có tự tương quan (checklist Bài 11, ý cuối).

### Cách triển khai

**[D]** Với từng bộ: SE iid $s/\sqrt n$ so với SE có xét phụ thuộc $\sqrt{\hat\gamma(0)\hat\tau/n}$, $\hat\tau$ ước lượng bằng tổng Bartlett cắt cụt ở
$K=\lfloor4(n/100)^{2/9}\rfloor$ (kiểu Newey–West) và bằng công thức AR(1) $\hat\tau=(1+\hat\rho_1)/(1-\hat\rho_1)$; kèm công thức chính xác Định lý 1(a)
thay $\hat\gamma$. Với HU, so SE hệ số dốc OLS (−0.0242) với SE khi sai số AR(2) (ML).
**[M]** AR(1), $\phi=0.6$, $n=100$, $B=2000$, seed 3179: độ phủ của KTC 95% ngây thơ và đã hiệu chỉnh $\sqrt\tau$.

### Mã lệnh Python

In [ ]:
rows = []
for c in ("HU", "LY", "DR"):
    y = Y[c]; n = len(y); rho = A.sample_acf(y, n - 1); g = A.sample_autocov(y, n - 1)
    K = int(np.floor(4 * (n / 100) ** (2 / 9)))
    tau_bart = 1 + 2 * np.sum((1 - np.arange(1, K + 1) / (K + 1)) * rho[1:K + 1])
    tau_ar1 = (1 + rho[1]) / (1 - rho[1])
    m = n // 4; var_exact = (g[0] + 2 * np.sum((1 - np.arange(1, m + 1) / n) * g[1:m + 1])) / n   # chỉ dùng h ≤ n/4 (quy tắc của bài giảng)
    rows.append({"bộ": c, "n": n, "K": K, "SE iid": y.std(ddof=1) / np.sqrt(n), "τ̂ (Bartlett)": tau_bart,
                 "SE hiệu chỉnh": np.sqrt(g[0] * tau_bart / n), "τ̂ (AR(1))": tau_ar1,
                 "SE theo công thức đầy đủ, h ≤ n/4": np.sqrt(var_exact) if var_exact > 0 else np.nan, "n_eff = n/τ̂ (Bartlett)": n / tau_bart})
tab_mean = save_table(pd.DataFrame(rows).set_index("bộ"), "tab_b11_03_mean_se")

ols = sm.OLS(Y["HU"], sm.add_constant(X_HU)).fit()
ar2 = ARIMA(Y["HU"], exog=X_HU, order=(2, 0, 0)).fit(cov_type="oim")
tab_slope = save_table(pd.DataFrame({
    "hệ số dốc (ft/năm)": [ols.params[1], dict(zip(ar2.param_names, ar2.params))["x1"]],
    "SE (ft/năm)": [ols.bse[1], dict(zip(ar2.param_names, ar2.bse))["x1"]],
}, index=["OLS (sai số iid)", "ML, sai số AR(2)"]).assign(**{"thống kê t": lambda d: d["hệ số dốc (ft/năm)"] / d["SE (ft/năm)"]}), "tab_b11_03_slope_se")

In [ ]:
B, n_sim, phi = 2000, 100, 0.6
tau = (1 + phi) / (1 - phi)
simulate = lambda r: S.simulate_ar1(phi, n_sim, r)
ci_naive = lambda y: (y.mean() - 1.96 * y.std(ddof=1) / np.sqrt(n_sim), y.mean() + 1.96 * y.std(ddof=1) / np.sqrt(n_sim))
ci_corr = lambda y: (y.mean() - 1.96 * y.std(ddof=1) * np.sqrt(tau / n_sim), y.mean() + 1.96 * y.std(ddof=1) * np.sqrt(tau / n_sim))
# cùng generator con cho hai khoảng => cùng 2000 chuỗi mô phỏng
cov_naive = S.coverage_study(simulate, ci_naive, 0.0, B, S.child_rng(11, 3))
cov_corr = S.coverage_study(simulate, ci_corr, 0.0, B, S.child_rng(11, 3))
tab_cov = save_table(pd.DataFrame({"độ phủ": [cov_naive["coverage"], cov_corr["coverage"]],
                                   "MCSE": [cov_naive["mcse"], cov_corr["mcse"]], "danh nghĩa": [0.95, 0.95]},
                                  index=["ngây thơ  ȳ ± 1.96 s/√n", f"hiệu chỉnh  ȳ ± 1.96 s√(τ/n), τ = {tau:g}"]), "tab_b11_03_coverage")

### Kết quả

In [ ]:
display(tab_mean)
display(tab_slope)
tab_cov

### Phân tích và diễn giải

*Đồng bộ từ báo cáo, mục 4.1(c).*

Mô phỏng cho thấy hiệu ứng lý thuyết rõ ràng: khoảng ngây thơ chỉ phủ trung bình thật $64.7%$ số lần thay vì $95%$, còn khoảng hiệu chỉnh phủ $94.4%$, lệch khỏi $95%$ chưa tới $1.2$ lần sai số Monte Carlo là $0.005$. Trên HU, sai số chuẩn của trung bình tăng từ $0.133$ lên $0.233$ khi hiệu chỉnh bằng $\hat\tau=3.09$, tương ứng cỡ mẫu hiệu dụng chỉ khoảng $32$ thay vì $98$. Độ lớn của $\hat\tau$ phụ thuộc cách ước lượng, chẳng hạn công thức AR(1) cho $10.9$, nhưng mọi cách đều cho $\tau>1$; ngoài ra HU có xu thế nên con số $32$ chỉ là mức tham khảo. Với LY, $\hat\tau=2.36$ nâng sai số chuẩn từ $0.052$ lên $0.080$, tương ứng cỡ mẫu hiệu dụng khoảng $48$ thay vì $114$. Riêng cột “SE theo công thức đầy đủ” của LY bị trống: cách cộng $\hat\gamma(h)$ với trọng số $(1-h/n)$ rồi cắt ở $h\le n/4$ không bảo đảm xác định dương, nên với ACF dao động mạnh tổng này âm, còn ước lượng Bartlett thì không gặp vấn đề đó. Với DR, $\hat\tau=0.92$, sai số chuẩn gần như không đổi ($0.0239$ so với $0.0229$), đúng như lý thuyết dự đoán cho chuỗi không tự tương quan. Với hệ số dốc xu thế, hai ước lượng gần nhau, là $-0.0242$ và $-0.0216$ ft/năm, nhưng sai số chuẩn tăng gấp đôi từ $0.0040$ lên $0.0081$, nên $t$ giảm từ $-6.0$ xuống $-2.65$: hệ số dốc vẫn có ý nghĩa ở mức $5%$, song bằng chứng yếu hơn nhiều.

### Nhận xét

*Đồng bộ từ báo cáo, mục 4.1(c).*

Kết quả **phù hợp** với lý thuyết: mô phỏng khớp trong dung sai, dữ liệu thực xác nhận sai số chuẩn giả định iid quá lạc quan ở HU và LY, và DR cho đúng $\tau\approx1$ như dự đoán. Độ lớn của hiệu chỉnh phụ thuộc cách ước lượng $\tau$, nên kết luận định tính chắc hơn kết luận định lượng.

## 11.4 ACF mẫu, dải iid, Bartlett và kiểm định Ljung–Box

*Bộ dữ liệu:* HU, LY, DR

### Nội dung lý thuyết

**Định lý 2.** (a) Dưới nhiễu iid, $\sqrt n(\hat\rho(1),\dots,\hat\rho(H))\to N(0,I_H)$, dải 95% $\pm1.96/\sqrt n$ cho từng độ trễ;
(b) $Q_{LB}(H)=n(n+2)\sum_{h=1}^H\hat\rho(h)^2/(n-h)\to\chi^2_H$ (với phần dư ARMA(p,q): $\chi^2_{H-p-q}$);
(c) Bartlett: với MA(q), $h>q$: $\sqrt n\hat\rho(h)\to N(0,1+2\sum_{k\le q}\rho(k)^2)$.

### Cách triển khai

**[D]** Tự tính $\hat\rho(h)$, $h\le20$, so với `acf`; đếm số độ trễ vượt dải iid; $Q_{LB}(10)$ tự tính so với `acorr_ljungbox`
(DR: cả $r_t$ và $r_t^2$); dải Bartlett giả định MA(1) từ $\hat\rho(1)$. **[M]** nhiễu iid $N(0,1)$, $n=200$, $B=2000$: tần suất $|\hat\rho(h)|$ vượt dải
(≈ 5%), phân phối của $Q_{LB}(10)$ so với $\chi^2_{10}$.

### Mã lệnh Python

In [ ]:
series = {"HU: mức nước gốc": Y["HU"], "LY: log₁₀ số linh miêu": Y["LY"], "DR: suất sinh lợi r": Y["DR"], "DR: bình phương r²": Y["DR"] ** 2}
rows = []
fig, axes = plt.subplots(2, 2, figsize=(11.5, 6.6))
for k, (ax, (name, y)) in enumerate(zip(axes.ravel(), series.items())):
    n = len(y); r = A.sample_acf(y, 20); band = A.bartlett_band(n)
    bart = A.bartlett_band(n, r, q=1) if "r²" not in name else np.nan   # Bartlett MA(1) không có nghĩa cho r²
    acf_bars(ax, r, n, name, band_label=f"iid ±1.96/√n = ±{band:.3f}")
    if not np.isnan(bart) and abs(bart - band) > 0.003:   # DR: dải Bartlett trùng dải iid nên không vẽ thêm
        ax.axhline(bart, color=ORANGE, ls=":", lw=1.6); ax.axhline(-bart, color=ORANGE, ls=":", lw=1.6)
        ax.set_xlabel(ax.get_xlabel() + f";  ····: Bartlett MA(1) ±{bart:.3f}")
    panel_label(ax, "(" + "abcd"[k] + ")")
    q, p = A.ljung_box_manual(y, 10); q_ref, p_ref = lb(y, 10)
    rows.append({"chuỗi": name, "n": n, "dải iid ±": band, "dải Bartlett MA(1) ±": bart,
                 "số độ trễ vượt dải iid (h ≤ 20)": int((np.abs(r[1:]) > band).sum()),
                 "max|ρ̂ tay − ρ̂ thư viện|": np.abs(r - acf(y, nlags=20, fft=False)).max(),
                 "Q_LB(10), tự cài đặt": q, "Q_LB(10), thư viện": q_ref, "p-value": p})
fig_path = save_figure(fig, "fig_b11_04_acf_bands")
tab = save_table(pd.DataFrame(rows).set_index("chuỗi"), "tab_b11_04_acf_ljungbox")

In [ ]:
rng = S.child_rng(11, 4)
B, n_sim, H = 2000, 200, 10
band = A.bartlett_band(n_sim); outside = 0; Q = np.empty(B)
for b in range(B):
    e = rng.normal(size=n_sim)
    outside += np.sum(np.abs(A.sample_acf(e, 20)[1:]) > band)
    Q[b] = A.ljung_box_manual(e, H)[0]
reject = np.mean(Q > stats.chi2.ppf(0.95, H))
# xấp xỉ mẫu hữu hạn cho tỷ lệ vượt dải: E ρ̂(h) ≈ −1/n, Var ρ̂(h) ≈ (n−h)/(n(n+2)), lấy trung bình theo h = 1..20
hs = np.arange(1, 21); mu_h, sd_h = -1 / n_sim, np.sqrt((n_sim - hs) / (n_sim * (n_sim + 2)))
finite = np.mean(stats.norm.cdf((-band - mu_h) / sd_h) + stats.norm.sf((band - mu_h) / sd_h))
tab_sim = save_table(pd.DataFrame({
    "mô phỏng": [outside / (B * 20), reject, Q.mean(), Q.var(ddof=1), stats.kstest(Q, "chi2", args=(H,)).pvalue],
    "lý thuyết (tiệm cận)": [0.05, 0.05, H, 2 * H, np.nan],
    "xấp xỉ mẫu hữu hạn": [finite, np.nan, np.nan, np.nan, np.nan],
    "MCSE": [np.sqrt(0.05 * 0.95 / (B * 20)), np.sqrt(0.05 * 0.95 / B), Q.std(ddof=1) / np.sqrt(B), np.nan, np.nan],
}, index=["Tỷ lệ |ρ̂(h)| vượt ±1.96/√n (h = 1..20)", "Tỷ lệ Q_LB(10) > phân vị 95% của χ²₁₀", "Trung bình của Q_LB", "Phương sai của Q_LB",
          "p-value kiểm định KS so với χ²₁₀"]), "tab_b11_04_simulation")

### Kết quả

In [ ]:
show_figure(fig_path)
display(tab)
tab_sim

### Phân tích và diễn giải

*Đồng bộ từ báo cáo, mục 4.3(a).*

HU có $9$ trong $20$ độ trễ vượt dải $\pm0.198$ và $Q_{LB}(10)=189.9$, LY có $16$ độ trễ vượt dải $\pm0.184$ và $Q_{LB}(10)=286.0$; cả hai có $p<0.0001$ nên bác bỏ mạnh giả thuyết không tự tương quan. DR chỉ có $2$ độ trễ vượt dải $\pm0.0455$, gần mức $1$ độ trễ do ngẫu nhiên, và $Q_{LB}(10)=6.37$ với $p=0.78$, tức không bác bỏ nhiễu trắng; nhưng với bình phương suất sinh lợi thì $9$ độ trễ vượt dải và $Q_{LB}(10)=110.7$ với $p<0.0001$. Giá trị tự cài đặt trùng thư viện ở mọi chuỗi. Mô phỏng nhiễu iid cho tỷ lệ vượt dải $4.35%$, thấp hơn mức tiệm cận $5%$ khoảng $6$ MCSE nhưng trùng mức mẫu hữu hạn $4.36%$, tính từ $\mathrm{E}\hat\rho(h)\approx-1/n$ và $\operatorname{Var}\hat\rho(h)\approx(n-h)/\{n(n+2)\}$ với $n=200$: ở mẫu hữu hạn, dải $\pm1.96/\sqrt n$ hơi rộng nên ít bị vượt hơn $5%$. Trung bình $Q_{LB}$ là $10.08$ so với $10$, lệch $0.7$ MCSE, phương sai $22.4$ so với $20$, và kiểm định Kolmogorov–Smirnov so với $\chi^2_{10}$ cho $p=0.47$. Tỷ lệ bác bỏ ở mức $5%$ là $6.15%$, cao hơn khoảng $2.3$ lần sai số Monte Carlo $0.005$, tức hơi tự do nhưng vẫn trong dung sai $3$ MCSE.

### Nhận xét

*Đồng bộ từ báo cáo, mục 4.3(a).*

Kết quả **phù hợp** với lý thuyết: mô phỏng khớp phân phối lý thuyết trong dung sai, và trên dữ liệu thực các công cụ phân biệt đúng chuỗi có cấu trúc là HU và LY với chuỗi gần nhiễu trắng là DR. Dải iid chỉ đúng với nhiễu iid, vì DR không tự tương quan nhưng bình phương của nó thì có.

## 11.5 Quá trình tuyến tính, toán tử lùi, trọng số ψ và ACF lý thuyết

*Bộ dữ liệu:* HU, LY, DR

### Nội dung lý thuyết

**Định nghĩa 4, Mệnh đề 2.** $\phi(B)(Y_t-\mu)=\theta(B)\varepsilon_t$ có biểu diễn nhân quả $Y_t=\mu+\sum_{j\ge0}\psi_j\varepsilon_{t-j}$ với
$\sum\psi_jz^j=\theta(z)/\phi(z)$ và $\gamma(h)=\sigma^2\sum_j\psi_j\psi_{j+h}$.

### Cách triển khai

**[D]** Khớp AR(2) bằng ML cho HU (với xu thế trong exog) và LY. Tính $\psi_j$ bằng đệ quy tay (`psi_weights`) so với `arma2ma`; tính ACF lý thuyết
$\gamma(h)=\sigma^2\sum\psi_j\psi_{j+h}$ (`theoretical_acf`) so với `arma_acf`, rồi chồng lên ACF mẫu.

### Mã lệnh Python

In [ ]:
rows, fig, axes = [], *plt.subplots(1, 2, figsize=(11.5, 3.8))
for k, (ax, c) in enumerate(zip(axes, ("HU", "LY"))):
    res = fit_arma(c, (2, 0, 0)); phi = res.arparams
    psi, psi_ref = R.psi_weights(phi, [], 15), arma2ma(np.r_[1, -phi], [1], 15)
    rho, rho_ref = R.theoretical_acf(phi, [], 20), arma_acf(np.r_[1, -phi], [1], 21)
    rows.append({"bộ": c, "φ̂₁": phi[0], "φ̂₂": phi[1], "σ̂²": sig2(res), "ψ₁": psi[1], "ψ₂": psi[2], "ψ₃": psi[3],
                 "max|ψ tay − arma2ma|": np.abs(psi - psi_ref).max(), "max|ρ tay − arma_acf|": np.abs(rho - rho_ref).max()})
    acf_bars(ax, A.sample_acf(Z[c], 20), len(Z[c]), f"{c}: {'phần dư khử xu thế' if c == 'HU' else 'log₁₀ số linh miêu'}", highlight=False)
    overlay_theory(ax, range(1, 21), rho[1:], "ACF lý thuyết AR(2) (ML)"); panel_label(ax, "(" + "ab"[k] + ")")
fig_path = save_figure(fig, "fig_b11_05_acf_theory")
tab = save_table(pd.DataFrame(rows).set_index("bộ"), "tab_b11_05_psi_acf")
print("DR: suất sinh lợi gần nhiễu trắng nên mô hình khớp có ψ = (1, 0, 0, …) và ACF lý thuyết = 0 với h ≥ 1; "
      "không có cấu trúc tuyến tính nhân quả để chồng lên ACF mẫu (xem 11.9, 11.14).")

### Kết quả

In [ ]:
show_figure(fig_path)
tab

### Phân tích và diễn giải

*Đồng bộ từ báo cáo, mục 4.1(d).*

Các giá trị tự cài đặt trùng `statsmodels`, với chênh lệch $0$ ở $\psi$ và $10^{-16}$ ở ACF. HU có $\hat\phi=(1.005,-0.291)$ nên $\psi=(1,1.005,0.718,0.429,\dots)$ giảm dần, và ACF lý thuyết bám sát ACF mẫu trong 10 độ trễ đầu. LY có $\hat\phi=(1.378,-0.740)$ nên $\psi=(1,1.378,1.158,0.576,\dots)$; ACF lý thuyết tái hiện dạng sóng của ACF mẫu nhưng biên độ nhỏ hơn rõ rệt, vì ở độ trễ 9 và 10 ACF mẫu đạt khoảng $0.55$ đến $0.6$ còn ACF lý thuyết chỉ khoảng $0.2$. Nói cách khác, AR(2) bắt được chu kỳ nhưng chưa bắt hết độ bền của nó, điều sẽ được xét lại ở mục 4.3. Với DR, ACF mẫu đúng như $\gamma(h)=0$ mà $\psi=(1,0,0,\dots)$ dự đoán: chỉ $2$ trong $20$ độ trễ vượt dải $\pm0.0455$, gần mức $1$ độ trễ do ngẫu nhiên (bảng `tab_b11_04_acf_ljungbox`).

### Nhận xét

*Đồng bộ từ báo cáo, mục 4.1(d).*

Kết quả **phù hợp** với lý thuyết: công thức quá trình tuyến tính được tính đúng và cho ACF có dạng đúng với dữ liệu ở cả ba chuỗi. Việc AR(2) chưa khớp hết biên độ của LY là hạn chế của mô hình chứ không phải của lý thuyết.

## 11.6 Mô hình AR(1)

*Bộ dữ liệu:* HU, LY, DR

### Nội dung lý thuyết

**Định lý 3.** $|\phi|<1$: nghiệm dừng nhân quả duy nhất $Y_t=\mu+\sum\phi^j\varepsilon_{t-j}$; $\gamma(h)=\sigma^2\phi^{|h|}/(1-\phi^2)$,
$\rho(h)=\phi^{|h|}$; dự báo $\hat Y_{n+h}=\mu+\phi^h(Y_n-\mu)$ với MSE $\sigma^2(1-\phi^{2h})/(1-\phi^2)\to\gamma(0)$.

### Cách triển khai

**[D]** Khớp AR(1) cho HU (xu thế trong exog): so ACF mẫu của phần dư khử xu thế với $\hat\phi^h$, $\gamma(0)=\hat\sigma^2/(1-\hat\phi^2)$ với phương sai mẫu,
và MSE dự báo $h$ bước (công thức tay so với `get_forecast`). **[M]** mô phỏng AR(1) với $(\hat\phi,\hat\sigma)$, $n=98$, $B=2000$ kiểm tra $\gamma(0)$, $\rho(1)$.

### Mã lệnh Python

In [ ]:
res = fit_arma("HU", (1, 0, 0)); phi, s2 = res.arparams[0], sig2(res)
g0_theory = s2 / (1 - phi ** 2)
rho_hat = A.sample_acf(U_HU, 6)
tab_acf = save_table(pd.DataFrame({"ρ̂(h) mẫu (phần dư khử xu thế)": rho_hat[1:], "φ̂^h": phi ** np.arange(1, 7)},
                                  index=pd.Index(range(1, 7), name="h")), "tab_b11_06_ar1_acf")
H = [1, 2, 5, 10]
fc = res.get_forecast(10, exog=np.arange(53, 63.0).reshape(-1, 1))
tab_fc = save_table(pd.DataFrame({
    r"MSE tay $\sigma^2(1-\phi^{2h})/(1-\phi^2)$": [s2 * (1 - phi ** (2 * h)) / (1 - phi ** 2) for h in H],
    "MSE từ statsmodels": [fc.var_pred_mean[h - 1] for h in H], "γ(0) giới hạn": g0_theory}, index=pd.Index(H, name="h")),
    "tab_b11_06_ar1_forecast_mse")
sim_g0, sim_r1 = [], []
rng = S.child_rng(11, 6)
for _ in range(2000):
    y = S.simulate_ar1(phi, 98, rng, sigma=np.sqrt(s2))
    sim_g0.append(A.sample_autocov(y, 0)[0]); sim_r1.append(A.sample_acf(y, 1)[1])
# kỳ vọng mẫu hữu hạn (n = 98): E γ̂(0) = γ(0) − Var(Ȳ) đúng chính xác; E ρ̂(1) ≈ φ − (1 + 4φ)/n (xấp xỉ Kendall)
n_ = 98; hh = np.arange(-(n_ - 1), n_)
var_ybar = np.sum((1 - np.abs(hh) / n_) * g0_theory * phi ** np.abs(hh)) / n_
tab_sim = save_table(pd.DataFrame({
    "lý thuyết (tiệm cận)": [g0_theory, phi], "kỳ vọng mẫu hữu hạn": [g0_theory - var_ybar, phi - (1 + 4 * phi) / n_],
    "trung bình mô phỏng": [np.mean(sim_g0), np.mean(sim_r1)],
    "MCSE": [np.std(sim_g0, ddof=1) / np.sqrt(2000), np.std(sim_r1, ddof=1) / np.sqrt(2000)]},
    index=["γ̂(0) (mẫu số n)", "ρ̂(1)"]), "tab_b11_06_ar1_simulation")
# LY: khớp AR(1) để thấy ACF lý thuyết φ̂^h không đổi dấu, trái với ACF mẫu dạng sóng
res_ly1 = fit_arma("LY", (1, 0, 0)); phi_ly = res_ly1.arparams[0]
tab_ly1 = save_table(pd.DataFrame({"ρ̂(h) mẫu": A.sample_acf(Y["LY"], 10)[1:], "φ̂^h của AR(1)": phi_ly ** np.arange(1, 11)},
                                  index=pd.Index(range(1, 11), name="h")), "tab_b11_06_ly_ar1")
print(f"φ̂ = {phi:.4f}, σ̂² = {s2:.4f}, γ(0) lý thuyết = {g0_theory:.4f}, phương sai mẫu phần dư khử xu thế = {A.sample_autocov(U_HU, 0)[0]:.4f}")
print(f"LY: AR(1) có φ̂ = {phi_ly:.4f}; ACF mẫu đổi dấu còn φ̂^h luôn dương. DR: φ̂ ≈ 0 vì gần nhiễu trắng (xem 11.4, 11.14).")

### Kết quả

In [ ]:
display(tab_acf)
display(tab_fc)
tab_sim

### Phân tích và diễn giải

*Đồng bộ từ báo cáo, mục 4.2(a).*

Công thức sai số dự báo trùng `statsmodels` ở mọi chân trời, bắt đầu từ $0.4965$ ở $h=1$ và tăng dần về giới hạn $\gamma(0)=1.286$. Tuy nhiên ACF mẫu của phần dư giảm nhanh hơn $\hat\phi^h$: ở độ trễ 2 là $0.46$ so với $0.61$, ở độ trễ 4 là $0.14$ so với $0.38$, nên AR(1) không mô tả tốt HU, như AIC tốt hơn của AR(2) ở mục 4.3 cũng cho thấy. Với LY, AR(1) cho $\hat\phi=0.792$ và khớp ACF mẫu ở độ trễ 1 ($0.785$), nhưng ACF mẫu đổi dấu từ độ trễ 3 và xuống $-0.62$ ở độ trễ 5, trong khi $\hat\phi^h$ luôn dương và giảm đều. Đúng như lý thuyết, AR(1) với $\phi>0$ không thể tạo ra dao động, nên LY không phải AR(1). Với DR, $\hat\phi=\hat\rho(1)=-0.0004$ nên $\hat\phi^h\approx0$ ở mọi $h\ge1$, khớp ACF mẫu gần như nằm trọn trong dải ở 4.3(a). Mô phỏng cho thấy các thống kê mẫu chệch xuống so với giá trị tiệm cận, với $\hat\gamma(0)$ trung bình $1.176$ so với $1.286$ và $\hat\rho(1)$ trung bình $0.738$ so với $0.784$. Độ chệch này tính được: với $n=98$, $\mathrm{E}\hat\gamma(0)=\gamma(0)-\operatorname{Var}\bar Y=1.182$ đúng chính xác, và $\mathrm{E}\hat\rho(1)\approx\phi-(1+4\phi)/n=0.741$. Mô phỏng chỉ lệch $0.8$ và $2.3$ MCSE khỏi hai giá trị này, trong dung sai $3$ MCSE.

### Nhận xét

*Đồng bộ từ báo cáo, mục 4.2(a).*

Kết quả **phù hợp một phần** với lý thuyết. Các công thức về $\gamma(0)$, $\rho(h)$ và sai số dự báo đúng và khớp thư viện, và mô phỏng khớp kỳ vọng mẫu hữu hạn. Phần chưa khớp nằm ở dữ liệu: ACF mẫu của HU giảm nhanh hơn $\hat\phi^h$ và ACF của LY dao động, tức AR(1) không đủ cho hai chuỗi này; DR khớp đúng trường hợp $\phi\approx0$.

## 11.7 AR(p): tính nhân quả và phương trình Yule–Walker

*Bộ dữ liệu:* HU, LY, DR

### Nội dung lý thuyết

**Định lý 4.** AR(p) có nghiệm dừng nhân quả duy nhất ⇔ mọi nghiệm của $\phi(z)=1-\phi_1z-\dots-\phi_pz^p$ có $|z|>1$.
Yule–Walker: $\Gamma_p\phi=\gamma_p$, $\sigma^2=\gamma(0)-\phi^\top\gamma_p$. **Định lý 6(a):** với mẫu số $n$, AR(p) khớp bằng YW luôn nhân quả.

### Cách triển khai

**[D]** Giải $\hat\Gamma_p\hat\phi=\hat\gamma_p$ bằng tay cho LY ($p=2$ và $p=11$) và HU đã khử xu thế ($p=2$); so với `yule_walker(method="mle")`; tính môđun các nghiệm
và kiểm tra nhân quả. DR gần nhiễu trắng nên không có cấu trúc AR đáng khớp.

### Mã lệnh Python

In [ ]:
rows = []
for c, p in (("LY", 2), ("LY", 11), ("HU", 2), ("DR", 2)):
    y = Z[c]; phi, s2 = R.yule_walker_manual(y, p)
    ref, sig = yule_walker(y, order=p, method="mle")
    chk = R.causal_invertible_check(phi, [])
    resid_norm = np.linalg.norm(A.gamma_matrix(y, p) @ phi - A.sample_autocov(y, p)[1:])
    rows.append({"bộ": c, "p": p, "φ̂₁": phi[0], "φ̂₂": phi[1], "σ̂² (tay)": s2, "max|φ̂ tay − thư viện|": np.abs(phi - ref).max(),
                 "|Γ̂φ̂ − γ̂|": resid_norm, "min |z|": chk["min_ar_modulus"], "nhân quả": chk["causal"]})
tab = save_table(pd.DataFrame(rows).set_index(["bộ", "p"]), "tab_b11_07_yule_walker")
print("DR: r_t gần nhiễu trắng (11.4) nên φ̂ ≈ 0; mô hình vẫn nhân quả như lý thuyết bảo đảm.")

### Kết quả

In [ ]:
tab

### Phân tích và diễn giải

*Đồng bộ từ báo cáo, mục 4.2(b).*

Nghiệm tự giải trùng thư viện, với chênh lệch cỡ $10^{-14}$ hoặc nhỏ hơn, và thỏa hệ phương trình đến sai số máy. Cả bốn mô hình đều nhân quả: môđun nghiệm nhỏ nhất là $1.178$ ở AR(2) của LY, $1.016$ ở AR(11) của LY, rất sát vòng tròn đơn vị nhưng vẫn nằm ngoài, $1.905$ ở AR(2) của HU và $6.12$ ở AR(2) của DR. Với DR, $\hat\phi=(-0.0004,-0.027)$, gần như bằng $0$, nên nghiệm nằm rất xa vòng tròn đơn vị: chuỗi hầu như không có cấu trúc AR.

### Nhận xét

*Đồng bộ từ báo cáo, mục 4.2(b).*

Kết quả **phù hợp** với lý thuyết: hệ Yule–Walker được giải đúng và cả bốn mô hình đều nhân quả như lý thuyết bảo đảm. Môđun $1.016$ của AR(11) cho thấy mô hình bậc cao có thể suýt chạm biên.

## 11.8 AR(2): tam giác nhân quả và chu kỳ

*Bộ dữ liệu:* HU, LY, DR

### Nội dung lý thuyết

**Mệnh đề 3.** AR(2) nhân quả ⇔ $\phi_1+\phi_2<1$, $\phi_2-\phi_1<1$, $|\phi_2|<1$ (tam giác). $\rho(1)=\phi_1/(1-\phi_2)$,
$\rho(h)=\phi_1\rho(h-1)+\phi_2\rho(h-2)$. Nghiệm phức ⇔ $\phi_1^2+4\phi_2<0$; khi đó $\rho(h)=ar^h\cos(\omega h+b)$, $r=\sqrt{-\phi_2}$,
$\cos\omega=\phi_1/(2r)$, chu kỳ tắt dần $2\pi/\omega$.

### Cách triển khai

**[D]** LY: Yule–Walker AR(2) (ví dụ mẫu trang 32), kiểm tra ba bất đẳng thức, chu kỳ từ $\omega$ và từ nghiệm đa thức ($2\pi/|\arg z|$), so với khoảng cách
giữa các đỉnh của ACF mẫu. HU: AR(2) ML (Ví dụ 3.2). Vẽ hai điểm trong tam giác nhân quả.

Với DR, ta khớp thêm AR(2) và xét cả môđun nghiệm lẫn mức ý nghĩa của các hệ số.

### Mã lệnh Python

In [ ]:
pts = {}
pts["LY (Yule–Walker)"], _ = R.yule_walker_manual(Y["LY"], 2)
pts["HU (ML, có xu thế)"] = fit_arma("HU", (2, 0, 0)).arparams
rows = []
for name, (p1, p2) in pts.items():
    z = R.ar_roots(np.array([p1, p2])); complex_roots = p1 ** 2 + 4 * p2 < 0
    rows.append({"mô hình": name, "φ₁": p1, "φ₂": p2, "φ₁+φ₂<1": p1 + p2 < 1, "φ₂−φ₁<1": p2 - p1 < 1, "|φ₂|<1": abs(p2) < 1,
                 "trong tam giác": R.ar2_in_triangle(p1, p2), "nghiệm phức": complex_roots, "môđun nghiệm": abs(z[0]),
                 "chu kỳ 2π/ω": R.ar2_period(p1, p2), "chu kỳ 2π/|arg z|": 2 * np.pi / abs(np.angle(z[0])) if complex_roots else np.nan})
tab = save_table(pd.DataFrame(rows).set_index("mô hình"), "tab_b11_08_ar2_triangle")
r_ly = A.sample_acf(Y["LY"], 20)
p1, p2 = pts["LY (Yule–Walker)"]
rho_ar2 = R.theoretical_acf([p1, p2], [], 20)
fig, axes = plt.subplots(1, 2, figsize=(11.5, 4.0), gridspec_kw={"width_ratios": [1, 1.25]})
t = np.linspace(-2, 2, 400)
axes[0].fill_between(t, -t ** 2 / 4, 1 - np.abs(t), color=BLUE, alpha=0.13, label="nghiệm thực")
axes[0].fill_between(t, -1, -t ** 2 / 4, color=ORANGE, alpha=0.25, label="nghiệm phức (chu kỳ)")
axes[0].plot(t, -t ** 2 / 4, color=GREY, ls="--", lw=1, label="φ₁² + 4φ₂ = 0")
axes[0].plot([-2, 2, 0, -2], [-1, -1, 1, -1], color="k", lw=1)
for (name, (a, b)), col, mk in zip(pts.items(), (RED, GREEN), ("o", "s")):
    axes[0].plot(a, b, mk, color=col, ms=7, label=name, zorder=5)
axes[0].set(xlabel="φ₁", ylabel="φ₂", title="Tam giác nhân quả AR(2)", xlim=(-2.15, 2.15), ylim=(-1.1, 1.1)); axes[0].legend(loc="upper center", bbox_to_anchor=(0.5, -0.17), ncol=2, fontsize=8)
acf_bars(axes[1], r_ly, len(Y["LY"]), "LY: ACF mẫu và ACF lý thuyết của AR(2)-YW", highlight=False)
overlay_theory(axes[1], range(1, 21), rho_ar2[1:], "ACF lý thuyết AR(2)")
panel_label(axes[0], "(a)"); panel_label(axes[1], "(b)")
fig_path = save_figure(fig, "fig_b11_08_ar2_triangle")
print("Độ trễ của đỉnh ρ̂(h) trong 5 ≤ h ≤ 15 (LY):", 5 + int(np.argmax(r_ly[5:16])), "| ρ(10) lý thuyết:", round(rho_ar2[10], 3), "| ρ̂(10) mẫu:", round(r_ly[10], 3))

In [ ]:
# DR • AR(2) ước lượng bằng cực đại hợp lý
fit_dr2 = fit_arma("DR", (2, 0, 0))
phi_dr = fit_dr2.arparams
print("DR: phi, p-values, môđun nghiệm, chu kỳ",
      phi_dr, fit_dr2.pvalues[1:3], abs(fit_dr2.arroots), R.ar2_period(*phi_dr))

### Kết quả

In [ ]:
show_figure(fig_path)
tab

### Phân tích và diễn giải

*Đồng bộ từ báo cáo, mục 4.2(c).*

Điểm $(1.35,-0.72)$ của LY nằm trong tam giác nhân quả và trong vùng nghiệm phức vì $\phi_1^2+4\phi_2=-1.06<0$, với môđun nghiệm $1.178$. Hai công thức cho cùng chu kỳ $9.66$ năm, khớp với ACF mẫu, vốn đạt cực đại thứ hai gần độ trễ 10 với giá trị khoảng $0.6$ và cực tiểu quanh độ trễ 5. HU cũng có nghiệm phức, vì $\hat\phi_1^2+4\hat\phi_2=-0.16<0$, với chu kỳ $16.8$ năm, nhưng $r=\sqrt{-\hat\phi_2}=0.54$ nên biên độ dao động chỉ còn khoảng $0.54^{16.8}\approx3\times10^{-5}$ sau một chu kỳ. Đây đúng là trường hợp lý thuyết ở 2.1.2(c) cho phép: có nghiệm phức nhưng dao động tắt quá nhanh để thấy trên ACF, và giải thích vì sao ACF của HU giảm nhanh về $0$ mà không đổi dấu rõ.

Hai nghiệm AR của DR có môđun 6.11, tương ứng biên độ chỉ còn 0.164 sau mỗi bước. Chu kỳ hình thức là 4.00 phiên, nhưng các hệ số có $p=0.981$ và $0.088$. Dữ liệu chỉ ủng hộ điều kiện nhân quả, chưa cho thấy chu kỳ bốn phiên.

### Nhận xét

*Đồng bộ từ báo cáo, mục 4.2(c).*

Kết quả **phù hợp** với lý thuyết: chỉ từ hai tham số $\hat\phi_1$ và $\hat\phi_2$ ta suy ra đúng chu kỳ $9.66$ năm quan sát trên ACF, khẳng định cặp nghiệm phức của AR(2) tạo ra dao động chu kỳ. HU cho thấy mặt còn lại của cùng kết quả: khi $r$ nhỏ, nghiệm phức vẫn có nhưng dao động tắt gần như ngay lập tức.

## 11.9 Mô hình MA(q): ACF cắt cụt, MA(1), khả nghịch

*Bộ dữ liệu:* HU, LY, DR

### Nội dung lý thuyết

**Mệnh đề 4.** MA(q): $\gamma(h)=\sigma^2\sum_{j}\theta_j\theta_{j+|h|}$ bị cắt cụt sau $q$. MA(1): $\rho(1)=\theta/(1+\theta^2)$ nên $|\rho(1)|\le\frac12$;
$(\theta,\sigma^2)$ và $(1/\theta,\theta^2\sigma^2)$ cùng hàm tự hiệp phương sai; chọn mô hình khả nghịch $|\theta|<1$ (nhiễu khôi phục được từ quá khứ).

### Cách triển khai

**[D]** Khớp MA(1) cho DR, MA(1) và MA(2) cho HU (xu thế trong exog): so $\rho(1)=\hat\theta/(1+\hat\theta^2)$ với $\hat\rho(1)$ của phần dư/chuỗi, kiểm tra $|\rho(1)|\le0.5$,
và tính $\gamma(0),\gamma(1)$ của cặp $(\hat\theta,\hat\sigma^2)$, $(1/\hat\theta,\hat\theta^2\hat\sigma^2)$. **[M]** mô phỏng MA(2) với $\hat\theta$ của HU, $n=98$, $B=2000$:
ACF mẫu có cắt cụt sau độ trễ 2 không.

Ta khớp MA(1) cho cả ba chuỗi và so ACF ở các độ trễ sau bậc mô hình với dải Bartlett.

### Mã lệnh Python

In [ ]:
dr_ma1 = fit_arma("DR", (0, 0, 1)); hu_ma1 = fit_arma("HU", (0, 0, 1)); hu_ma2 = fit_arma("HU", (0, 0, 2))
rows = []
for name, res, y in (("DR: MA(1)", dr_ma1, Y["DR"]), ("HU: MA(1)", hu_ma1, U_HU)):
    th, s2 = res.maparams[0], sig2(res)
    g_inv = R.theoretical_acvf([], [th], 1, s2); g_flip = R.theoretical_acvf([], [1 / th], 1, th ** 2 * s2)
    rows.append({"mô hình": name, "θ̂": th, "SE(θ̂)": res.bse[res.param_names.index("ma.L1")], "σ̂²": s2,
                 "ρ(1)=θ/(1+θ²)": R.ma1_ratio(th), "ρ̂(1) mẫu": A.sample_acf(y, 1)[1], "|ρ(1)| ≤ 0.5": abs(R.ma1_ratio(th)) <= 0.5,
                 "khả nghịch": abs(th) < 1, "γ(0),γ(1) của (θ,σ²)": f"{g_inv[0]:.4g}, {g_inv[1]:.4g}",
                 "γ(0),γ(1) của (1/θ,θ²σ²)": f"{g_flip[0]:.4g}, {g_flip[1]:.4g}"})
tab = save_table(pd.DataFrame(rows).set_index("mô hình"), "tab_b11_09_ma1")
print("HU MA(2): θ̂ =", hu_ma2.maparams.round(4), "| σ̂² =", round(sig2(hu_ma2), 4), "| AIC =", round(hu_ma2.aic, 2))
print("LY: ACF suy giảm dạng sóng (chu kỳ) nên MA hữu hạn không phù hợp; cấu trúc AR được dùng ở 11.7–11.8.")

In [ ]:
th = hu_ma2.maparams; s = np.sqrt(sig2(hu_ma2)); n_sim, B = 98, 2000
rho_th = R.theoretical_acf([], th, 8)
rng = S.child_rng(11, 9)
R_hat = np.array([A.sample_acf(S.simulate_arma([], th, n_sim, rng, sigma=s), 8) for _ in range(B)])
band_iid, band_bart = A.bartlett_band(n_sim), A.bartlett_band(n_sim, rho_th, q=2)
tab_sim = save_table(pd.DataFrame({
    "ρ(h) lý thuyết": rho_th[1:], "trung bình ρ̂(h)": R_hat[:, 1:].mean(0),
    "tỷ lệ vượt dải iid": (np.abs(R_hat[:, 1:]) > band_iid).mean(0),
    "tỷ lệ vượt dải Bartlett (q=2)": np.where(np.arange(1, 9) > 2, (np.abs(R_hat[:, 1:]) > band_bart).mean(0), np.nan)},
    index=pd.Index(range(1, 9), name="h")), "tab_b11_09_ma2_simulation")
fig, ax = plt.subplots(figsize=(7.5, 3.6)); hh = np.arange(1, 9)
ax.bar(hh, R_hat[:, 1:].mean(0), width=0.55, color=BLUE, alpha=0.75, label="trung bình ρ̂(h) mô phỏng (B = 2000)")
ax.plot(hh, rho_th[1:], "o-", color=ORANGE, label="ρ(h) lý thuyết"); ax.axhline(0, color="k", lw=0.8)
ax.axvline(2.5, color=GREY, ls=":", lw=1); ax.text(2.55, ax.get_ylim()[1] * 0.9, "q = 2", color=GREY, fontsize=9)
ax.set(xlabel="độ trễ h", ylabel="ACF", title="MA(2) mô phỏng: ACF cắt cụt sau độ trễ 2"); ax.legend(loc="upper right")
fig_path = save_figure(fig, "fig_b11_09_ma2_acf")

In [ ]:
# LY • Kiểm tra hệ quả ACF cắt cụt của MA(1) trên chuỗi quan sát.
fit_ly_ma = fit_arma("LY", (0, 0, 1))
theta_ly = float(fit_ly_ma.maparams[0])
rho_ly = A.sample_acf(Y["LY"], 10)
print("LY: theta, rho(1) mô hình, rho(1,2,10) mẫu, dải tham khảo",
      theta_ly, R.ma1_ratio(theta_ly), rho_ly[[1, 2, 10]], A.bartlett_band(len(Y["LY"]), [1, R.ma1_ratio(theta_ly)], q=1))

### Kết quả

In [ ]:
display(tab)
show_figure(fig_path)
tab_sim

### Phân tích và diễn giải

*Đồng bộ từ báo cáo, mục 4.2(d).*

Với DR, $\hat\theta\approx-0.0005$ với sai số chuẩn $0.018$, tức không khác $0$, và $\rho(1)$ trùng $\hat\rho(1)$ mẫu, nên chuỗi không có cấu trúc MA, hợp với việc nó gần nhiễu trắng. Với HU, $\hat\theta=0.782$ khả nghịch và $\rho(1)=0.485$ sát biên $0.5$; hai tham số hóa cho cùng tự hiệp phương sai, với $\gamma(0)=0.9688$ và $\gamma(1)=0.4702$. Tuy nhiên $\hat\rho(1)$ mẫu của HU là $0.76>0.5$, nên MA(1) không phù hợp với HU, và AIC $237.2$ kém hẳn AR(2). Mô phỏng MA(2) xác nhận tính cắt cụt: trung bình $\hat\rho(1)=0.635$ và $\hat\rho(2)=0.183$ gần $0.655$ và $0.212$, còn ở $h>2$ chỉ khoảng $-0.03$. Dải iid bị vượt $14$ đến $15%$ số lần ở $h>2$, nhưng dải Bartlett chỉ bị vượt $3$ đến $5%$, đúng như công thức Bartlett dự đoán.

Với LY, MA(1) cho $\hat\theta=0.908$ và $\rho(1)=0.498$, trong khi ACF mẫu tại các độ trễ 1, 2 và 10 là 0.785, 0.340 và 0.606. Hai độ trễ sau vượt dải Bartlett $\pm0.224$. MA(1) không mô tả được dao động của LY.

### Nhận xét

*Đồng bộ từ báo cáo, mục 4.2(d).*

Kết quả **phù hợp** với lý thuyết: tính cắt cụt, giới hạn $|\rho(1)|\le\tfrac12$, tính tương đương của hai tham số hóa và vai trò của dải Bartlett đều được xác nhận. MA(1) không khớp được HU vì $\hat\rho(1)>0.5$, đó là hạn chế của mô hình chứ không mâu thuẫn với lý thuyết.

## 11.10 ARMA(p, q): nhân quả, khả nghịch, nghiệm chung

*Bộ dữ liệu:* HU, LY, DR

### Nội dung lý thuyết

**Định lý 5.** $\phi(B)(Y_t-\mu)=\theta(B)\varepsilon_t$, $\phi,\theta$ không nghiệm chung: nhân quả ⇔ $\phi(z)\ne0$ với $|z|\le1$, khả nghịch ⇔ $\theta(z)\ne0$ với $|z|\le1$;
$\psi(z)=\theta(z)/\phi(z)$. ARMA(1,1): $\psi_j=(\phi+\theta)\phi^{j-1}$, $\gamma(0)=\sigma^2\frac{1+2\phi\theta+\theta^2}{1-\phi^2}$,
$\rho(1)=\frac{(1+\phi\theta)(\phi+\theta)}{1+2\phi\theta+\theta^2}$, $\rho(h)=\phi\rho(h-1)$. Nghiệm chung bị triệt tiêu ⇒ mô hình thừa tham số, không nhận dạng được (Ví dụ 4.2).

### Cách triển khai

**[D]** Khớp ARMA(1,1) (ML) cho HU (xu thế trong exog), LY và DR; kiểm tra nghiệm AR/MA, khoảng cách nghiệm AR–MA, công thức đóng của $\psi_j$, $\gamma(0)$, $\rho(1)$ so với
cài đặt tổng quát. Minh họa thừa tham số: $Y_t=0.5Y_{t-1}+\varepsilon_t-0.5\varepsilon_{t-1}$ chính là nhiễu trắng; khớp ARMA(1,1) cho nhiễu trắng $n=200$.

### Mã lệnh Python

In [ ]:
rows = []
for c in ("HU", "LY", "DR"):
    res = fit_arma(c, (1, 0, 1)); ph, th, s2 = res.arparams[0], res.maparams[0], sig2(res)
    chk = R.causal_invertible_check([ph], [th])
    psi, psi_closed = R.psi_weights([ph], [th], 6), np.r_[1.0, (ph + th) * ph ** np.arange(0, 5)]
    g0_closed = s2 * (1 + 2 * ph * th + th ** 2) / (1 - ph ** 2)
    r1_closed = (1 + ph * th) * (ph + th) / (1 + 2 * ph * th + th ** 2)
    rows.append({"bộ": c, "φ̂": ph, "θ̂": th, "σ̂²": s2, "nhân quả": chk["causal"], "khả nghịch": chk["invertible"],
                 "min|z| AR": chk["min_ar_modulus"], "min|z| MA": chk["min_ma_modulus"], "khoảng cách nghiệm AR–MA": chk["min_ar_ma_root_gap"],
                 "max|ψ tổng quát − công thức đóng|": np.abs(psi - psi_closed).max(),
                 "γ(0) công thức đóng": g0_closed, "γ(0) = σ²Σψ²": R.theoretical_acvf([ph], [th], 0, s2)[0],
                 "ρ(1) công thức đóng": r1_closed, "ρ(1) = γ(1)/γ(0)": R.theoretical_acf([ph], [th], 1)[1], "AIC": res.aic})
tab = save_table(pd.DataFrame(rows).set_index("bộ"), "tab_b11_10_arma11")

In [ ]:
rng = S.child_rng(11, 10)
same = S.simulate_arma([0.5], [-0.5], 400, S.child_rng(11, 10), burn=0)
white = S.child_rng(11, 10).normal(size=400)
print("Yₜ = 0.5Yₜ₋₁ + εₜ − 0.5εₜ₋₁ trùng khít với nhiễu trắng εₜ:", np.allclose(same, white))
wn = S.simulate_arma([], [], 200, rng)
res_wn = ARIMA(wn, order=(1, 0, 1)).fit()
chk = R.causal_invertible_check(res_wn.arparams, res_wn.maparams)
tab_over = save_table(pd.DataFrame({"ước lượng": list(res_wn.params), "SE": list(res_wn.bse)}, index=["hằng số μ", "φ", "θ", "σ²"]), "tab_b11_10_overparameterized")
tab_cancel = save_table(pd.DataFrame({"giá trị": [res_wn.arparams[0] + res_wn.maparams[0], chk["min_ar_ma_root_gap"]]},
                                     index=["φ̂ + θ̂ (≈ 0 nếu triệt tiêu)", "khoảng cách giữa nghiệm AR và nghiệm MA"]), "tab_b11_10_cancellation")

### Kết quả

In [ ]:
tab.T
display(tab_over)
tab_cancel

### Phân tích và diễn giải

*Đồng bộ từ báo cáo, mục 4.2(e).*

HU cho $\hat\phi=0.653$ và $\hat\theta=0.357$, LY cho $\hat\phi=0.670$ và $\hat\theta=0.715$. Cả hai mô hình nhân quả và khả nghịch, với hai nghiệm cách nhau $4.34$ ở HU và $2.89$ ở LY. Các công thức đóng cho $\psi_j$, $\gamma(0)$ và $\rho(1)$ trùng khít với tính toán tổng quát, sai khác $10^{-17}$, và AIC của HU là $212.4$, ngang AR(2). Dấu hiệu triệt tiêu nghiệm thể hiện rõ ở DR, với $\hat\phi=0.737$, $\hat\theta=-0.753$, tổng gần $0$ và hai nghiệm cách nhau chỉ $0.029$, và ở nhiễu trắng mô phỏng, nơi $\hat\phi+\hat\theta=0.047$ nhưng sai số chuẩn của từng hệ số lên tới $1.36$ và $1.38$: hai tham số hầu như không xác định được dù tổng của chúng được ước lượng tốt.

### Nhận xét

*Đồng bộ từ báo cáo, mục 4.2(e).*

Kết quả **phù hợp** với lý thuyết: điều kiện nhân quả, khả nghịch và các công thức đóng đều đúng, và hiện tượng thừa tham số xuất hiện đúng như lý thuyết mô tả. Khi mô hình quá phức tạp so với dữ liệu, sai số chuẩn rất lớn và nghiệm AR gần nghiệm MA là tín hiệu để giảm bậc.

## 11.11 Tự tương quan riêng phần (PACF)

*Bộ dữ liệu:* HU, LY, DR

### Nội dung lý thuyết

**Định nghĩa 6, Mệnh đề 5.** $\alpha(h)=\phi_{hh}$ từ $\Gamma_h\phi_h=\gamma_h$; $\alpha(1)=\rho(1)$, $\alpha(2)=\frac{\rho(2)-\rho(1)^2}{1-\rho(1)^2}$; AR(p) nhân quả: $\alpha(h)=0$ với $h>p$;
MA khả nghịch: PACF suy giảm. Dải $\pm1.96/\sqrt n$ (MĐ 5(d)). Đệ quy Durbin–Levinson tính mọi $\phi_{hh}$ từ $\rho$.

### Cách triển khai

**[D]** Tính PACF mẫu bằng Durbin–Levinson tự cài, so với `pacf(method="ldb")`; kiểm tra công thức $\hat\alpha(1),\hat\alpha(2)$ theo $\hat\rho$; liệt kê độ trễ vượt dải $\pm1.96/\sqrt n$ và đề xuất bậc $p$.

### Mã lệnh Python

In [ ]:
rows = []; fig, axes = plt.subplots(1, 3, figsize=(12.5, 3.8))
for k, (ax, c) in enumerate(zip(axes, ("HU", "LY", "DR"))):
    y, n = Z[c], len(Z[c]); rho = A.sample_acf(y, 20); a = A.pacf_durbin_levinson(rho, 20); band = A.bartlett_band(n)
    sig = [h for h in range(1, 21) if abs(a[h]) > band]
    rows.append({"bộ": c + (" (khử xu thế)" if c == "HU" else ""), "α̂(1) − ρ̂(1)": a[1] - rho[1],
                 "α̂(2) − (ρ̂₂−ρ̂₁²)/(1−ρ̂₁²)": a[2] - (rho[2] - rho[1] ** 2) / (1 - rho[1] ** 2),
                 "max|PACF tay − pacf()|": np.abs(a - pacf(y, nlags=20, method="ldb")).max(), "dải ±": band,
                 "độ trễ vượt dải (h≤20)": str(sig), "bậc AR gợi ý (số độ trễ vượt dải liên tiếp từ h = 1)": next(h for h in range(1, 22) if h not in sig) - 1})
    acf_bars(ax, a, n, f"{c}: PACF" + (" (khử xu thế)" if c == "HU" else ""), ylabel="PACF α̂(h)"); panel_label(ax, "(" + "abc"[k] + ")")
fig_path = save_figure(fig, "fig_b11_11_pacf")
tab = save_table(pd.DataFrame(rows).set_index("bộ"), "tab_b11_11_pacf")

### Kết quả

In [ ]:
show_figure(fig_path)
tab

### Phân tích và diễn giải

*Đồng bộ từ báo cáo, mục 4.3(b).*

Đệ quy tự cài đặt trùng `statsmodels` đến $10^{-15}$ và hai công thức đầu đúng chính xác. HU và LY có PACF mạnh ở độ trễ 1 và 2, với $\hat\alpha(2)\approx-0.7$ ở LY, tức dạng AR(2); HU chỉ còn độ trễ 10 vượt dải nhẹ, còn LY có thêm các độ trễ 4, 7, 10 và 11 với biên độ nhỏ hơn, trong đó độ trễ 11 khoảng $-0.3$, cho thấy cấu trúc phức tạp hơn AR(2). DR chỉ có hai độ trễ là 11 và 17 vượt dải một chút, không có dạng cắt cụt nào, hợp với nhiễu trắng.

### Nhận xét

*Đồng bộ từ báo cáo, mục 4.3(b).*

Kết quả **phù hợp** với lý thuyết: đệ quy đúng ở độ chính xác máy và hình dạng PACF khớp tính chất cắt cụt, với HU và LY gợi ý AR(2) còn DR không có cấu trúc. Các độ trễ vượt dải rải rác ở xa đúng với mức ngẫu nhiên $5%$ nên không nên coi là bằng chứng của bậc cao.

## 11.12 Ước lượng: Yule–Walker, OLS, cực đại hợp lý

*Bộ dữ liệu:* HU, LY, DR

### Nội dung lý thuyết

**Định lý 6.** (a) Yule–Walker $\hat\phi=\hat\Gamma_p^{-1}\hat\gamma_p$, $\hat\sigma^2=\hat\gamma(0)-\hat\phi^\top\hat\gamma_p$ (mẫu số $n$ ⇒ luôn nhân quả);
(b) YW, bình phương tối thiểu có điều kiện và ML Gauss cùng có $\sqrt n(\hat\phi-\phi)\to N_p(0,\sigma^2\Gamma_p^{-1})$; AR(1): $\operatorname{Var}\hat\phi\approx(1-\phi^2)/n$;
(c) ARMA: MLE Gauss nhất quán và chuẩn tiệm cận, hiệp phương sai từ thông tin quan sát.

### Cách triển khai

**[D]** AR(2) cho LY và cho HU đã khử xu thế (YW, OLS bằng `AutoReg`) và HU với xu thế trong exog (ML, `cov_type="oim"`): bảng $\hat\phi_1,\hat\phi_2$ và SE (SE của YW là SE tiệm cận $\sqrt{\operatorname{diag}(\hat\sigma^2\hat\Gamma_2^{-1}/n)}$).
**[M]** AR(2) với $\hat\phi$ của LY, $n=114$, $B=2000$: phân phối của $\hat\phi$ (YW và OLS) so với phân phối tiệm cận.

### Mã lệnh Python

In [ ]:
rows = []
for c in ("LY", "HU", "DR"):
    y = Z[c]; n = len(y)
    phi_yw, s2_yw = R.yule_walker_manual(y, 2)
    se_yw = np.sqrt(np.diag(s2_yw * np.linalg.inv(A.gamma_matrix(y, 2)) / n))
    ols = AutoReg(y, lags=2, trend="c").fit()
    ml = fit_arma(c, (2, 0, 0), cov_type="oim"); i1 = ml.param_names.index("ar.L1")
    for name, ph, se in (("Yule–Walker", phi_yw, se_yw), ("OLS (AutoReg)", ols.params[1:3], ols.bse[1:3]),
                         ("ML (ARIMA)", ml.arparams, ml.bse[i1:i1 + 2])):
        rows.append({"bộ": c, "phương pháp": name, "φ̂₁": ph[0], "SE(φ̂₁)": se[0], "φ̂₂": ph[1], "SE(φ̂₂)": se[1]})
tab = save_table(pd.DataFrame(rows).set_index(["bộ", "phương pháp"]), "tab_b11_12_estimators")

In [ ]:
phi_true, s2_true = R.yule_walker_manual(Y["LY"], 2); n_sim, B = 114, 2000
rng = S.child_rng(11, 12); est = {"YW": [], "OLS": []}
for _ in range(B):
    y = S.simulate_arma(phi_true, [], n_sim, rng, sigma=np.sqrt(s2_true))
    est["YW"].append(R.yule_walker_manual(y, 2)[0])
    Xd = np.column_stack([np.ones(n_sim - 2), y[1:-1], y[:-2]])
    est["OLS"].append(np.linalg.lstsq(Xd, y[2:], rcond=None)[0][1:])
gamma_p = toeplitz(R.theoretical_acvf(phi_true, [], 1, s2_true))
se_asym = np.sqrt(np.diag(s2_true * np.linalg.inv(gamma_p) / n_sim))
rows = []
for k, e in est.items():
    e = np.array(e)
    for j in (0, 1):
        rows.append({"phương pháp": k, "tham số": f"φ{j + 1}", "giá trị thật": phi_true[j], "trung bình mô phỏng": e[:, j].mean(),
                     "SD mô phỏng": e[:, j].std(ddof=1), "SD tiệm cận": se_asym[j]})
tab_sim = save_table(pd.DataFrame(rows).set_index(["phương pháp", "tham số"]), "tab_b11_12_simulation")

### Kết quả

In [ ]:
display(tab)
tab_sim

### Phân tích và diễn giải

*Đồng bộ từ báo cáo, mục 4.3(c).*

LY cho ba ước lượng gần nhau, với $\hat\phi_1$ từ $1.350$ đến $1.384$, chênh khoảng nửa sai số chuẩn $\approx0.06$, và $\hat\phi_2$ từ $-0.720$ đến $-0.748$; HU cũng vậy, với $\hat\phi_1$ từ $0.971$ đến $1.005$ và sai số chuẩn $\approx0.097$; ở HU, Yule–Walker và OLS chạy trên phần dư đã khử xu thế, tức ước lượng hai bước, còn cực đại hợp lý ước lượng xu thế và AR(2) đồng thời, nên ba phương pháp không ước lượng đúng cùng một mô hình và một phần chênh lệch đến từ đó. Với DR, ba phương pháp cho $\hat\phi_1$ từ $-0.0007$ đến $-0.0004$, chênh nhau chưa tới $0.02$ sai số chuẩn, và sai số chuẩn $0.0232$ trùng giá trị tiệm cận $1/\sqrt{1859}=0.0232$. Mô phỏng cho độ lệch chuẩn mẫu từ $0.064$ đến $0.073$, gần giá trị tiệm cận $0.065$ và lệch nhiều nhất khoảng $13%$ ở Yule–Walker với $\hat\phi_1$. Yule–Walker còn chệch rõ hơn OLS: trung bình $\hat\phi_1$ là $1.299$ so với giá trị thật $1.35$, tức chệch $-0.05$, còn OLS cho $1.334$, chệch $-0.016$.

### Nhận xét

*Đồng bộ từ báo cáo, mục 4.3(c).*

Kết quả **phù hợp** với lý thuyết ở HU và DR, và **phù hợp một phần** ở LY; theo quy tắc gộp ở mục 7.1, nội dung này được xếp phù hợp. Ba phương pháp cho ước lượng và sai số chuẩn gần nhau đúng như lý thuyết tiệm cận, nhưng ở LY với $n=114$ Yule–Walker có độ chệch đáng kể mà lý thuyết tiệm cận không phản ánh, nên khi mẫu nhỏ, OLS hoặc cực đại hợp lý đáng tin hơn.

## 11.13 Chọn bậc bằng AIC/BIC và thẩm định cuốn chiếu

*Bộ dữ liệu:* HU, LY, DR

### Nội dung lý thuyết

AIC $=-2\hat\ell+2k$, BIC $=-2\hat\ell+k\log n$ (Bài 6), $k$ gồm các $\phi,\theta$, trung bình/hệ số hồi quy và $\sigma^2$; chỉ so các khớp cùng quy ước hợp lý.
AIC nhắm rủi ro KL dự báo, BIC nhất quán khi bậc hữu hạn đúng nằm trong ứng viên. Thẩm định cuốn chiếu (rolling origin): khớp lại trên dữ liệu trước mỗi năm rồi dự báo 1 bước.

### Cách triển khai

**[D]** LY: AR($p$), $p=1..14$ (ML): AIC chọn $p=11$, BIC chọn $p=2$; so dự báo cuốn chiếu 1 bước 1915–1934 (20 gốc, khớp lại mỗi năm) của AR(2) và AR(11) với mốc trung bình.
HU (xu thế trong exog): AR(1), AR(2), AR(3), MA(1), MA(2), ARMA(1,1) với $k$, AIC, BIC, $Q_{LB}(10)$ ($H-p-q$ bậc tự do). DR: lưới ARMA($p,q$), $p,q\le2$.

### Mã lệnh Python

In [ ]:
fits = {p: ARIMA(Y["LY"], order=(p, 0, 0)).fit() for p in range(1, 15)}
tab_ly = save_table(pd.DataFrame({"AIC": {p: f.aic for p, f in fits.items()}, "BIC": {p: f.bic for p, f in fits.items()}}).rename_axis("p"), "tab_b11_13_ly_ic")
p_aic, p_bic = tab_ly["AIC"].idxmin(), tab_ly["BIC"].idxmin()
fig, ax = plt.subplots(figsize=(7.5, 3.8))
for k, col in (("AIC", BLUE), ("BIC", ORANGE)):
    d_ = tab_ly[k] - tab_ly[k].min()
    ax.plot(tab_ly.index, d_, "o-", ms=4, color=col, label=k); ax.plot(d_.idxmin(), 0, "*", ms=15, color=col, mec="k", zorder=5)
ax.set(xlabel="bậc AR p", ylabel="giá trị − giá trị nhỏ nhất", ylim=(-2, 40), title="LY: tiêu chuẩn thông tin theo bậc AR (★ = bậc được chọn)")
ax.text(0.97, 0.62, f"p = 1 nằm ngoài thang (giá trị gốc):\nAIC = {tab_ly.loc[1, 'AIC']:.1f}, BIC = {tab_ly.loc[1, 'BIC']:.1f}", transform=ax.transAxes, ha="right", fontsize=8.5, color=GREY)
ax.xaxis.set_major_locator(plt.MaxNLocator(integer=True)); ax.legend(loc="upper left")
fig_path = save_figure(fig, "fig_b11_13_ic_ly")
print(f"LY: AIC chọn p = {p_aic}, BIC chọn p = {p_bic}")

In [ ]:
models = {"AR(1)": (1, 0, 0), "AR(2)": (2, 0, 0), "AR(3)": (3, 0, 0), "MA(1)": (0, 0, 1), "MA(2)": (0, 0, 2), "ARMA(1,1)": (1, 0, 1)}
rows = []
for name, o in models.items():
    f = fit_arma("HU", o); q, p = lb(f.resid, 10, o[0] + o[2])
    rows.append({"sai số u_t": name, "k": len(f.params), "AIC": f.aic, "BIC": f.bic, "Q_LB(10)": q, "p-value": p})
tab_hu = save_table(pd.DataFrame(rows).set_index("sai số u_t"), "tab_b11_13_hu_models")
rows = []
for p in range(0, 3):
    for q in range(0, 3):
        f = ARIMA(Y["DR"], order=(p, 0, q)).fit(); rows.append({"(p,q)": f"({p},{q})", "AIC": f.aic, "BIC": f.bic})
tab_dr = save_table(pd.DataFrame(rows).set_index("(p,q)"), "tab_b11_13_dr_grid")

In [ ]:
# rolling origin 1 bước, 1915–1934 (chỉ số 94..113), khớp lại bằng toàn bộ dữ liệu trước đó
t0 = int(np.where(DATA["LY"].year.to_numpy() == 1915)[0][0])
rows = []
for name, ff in (("AR(2)", lambda tr, h: ARIMA(tr, order=(2, 0, 0)).fit().forecast(h)),
                 ("AR(11)", lambda tr, h: ARIMA(tr, order=(11, 0, 0)).fit().forecast(h)),
                 ("AR(p) chọn theo AIC trong mỗi gốc", lambda tr, h: min((ARIMA(tr, order=(p, 0, 0)).fit() for p in range(1, 15)), key=lambda f: f.aic).forecast(h)),
                 ("trung bình quá khứ", lambda tr, h: np.full(h, tr.mean()))):
    a, f = E.rolling_origin_forecast(Y["LY"], ff, t0, 1)
    rows.append({"mô hình": name, "số gốc": len(a), "RMSE 1 bước": E.rmse(a, f), "MAE 1 bước": E.mae(a, f)})
tab_ro = save_table(pd.DataFrame(rows).set_index("mô hình"), "tab_b11_13_ly_rolling")
# chọn bậc chỉ trên dữ liệu trước giai đoạn đánh giá (1821–1914), để không dùng dữ liệu kiểm định khi chọn mô hình
fits_tr = {p: ARIMA(Y["LY"][:t0], order=(p, 0, 0)).fit() for p in range(1, 15)}
aic_tr, bic_tr = {p: f.aic for p, f in fits_tr.items()}, {p: f.bic for p, f in fits_tr.items()}
tab_ord = save_table(pd.DataFrame([
    {"dữ liệu dùng để chọn bậc": "1821–1934 (toàn bộ)", "AIC chọn p": int(tab_ly["AIC"].idxmin()), "BIC chọn p": int(tab_ly["BIC"].idxmin())},
    {"dữ liệu dùng để chọn bậc": "1821–1914 (trước giai đoạn đánh giá)", "AIC chọn p": min(aic_tr, key=aic_tr.get), "BIC chọn p": min(bic_tr, key=bic_tr.get)},
]).set_index("dữ liệu dùng để chọn bậc"), "tab_b11_13_ly_order_train")

### Kết quả

In [ ]:
show_figure(fig_path)
display(tab_ly.T.round(2))
display(tab_hu)
display(tab_dr.T.round(2))
tab_ro

### Phân tích và diễn giải

*Đồng bộ từ báo cáo, mục 4.3(d).*

Với LY, AIC chọn $p=11$ với AIC $-24.0$, còn BIC chọn $p=2$ với BIC $5.9$, đúng xu hướng BIC chọn mô hình gọn hơn; cả hai đều khác xa $p=1$ với AIC $84.1$. Về dự báo một bước, AR(11) cho sai số nhỏ nhất, RMSE $0.203$ và MAE $0.152$, tốt hơn AR(2) với $0.234$ và $0.189$, và hơn hẳn trung bình quá khứ với RMSE $0.544$. Tuy nhiên bậc 11 được chọn trên toàn bộ 1821–1934, gồm cả giai đoạn đánh giá, điều 2.3.3(b) dặn tránh, nên so sánh này hơi lạc quan cho AR(11). Khi chỉ dùng 1821–1914, AIC chọn $p=12$ và BIC vẫn chọn $p=2$; khi chọn lại bậc theo AIC tại mỗi gốc, RMSE là $0.209$ và MAE $0.165$, kém AR(11) cố định một chút nhưng vẫn tốt hơn rõ AR(2). Như vậy lợi thế của bậc cao vẫn đứng vững khi bỏ rò rỉ, dù chỉ dựa trên $20$ gốc nên không nên khái quát. Với HU, AR(2) và ARMA(1,1) cùng đạt AIC $212.4$ và BIC $225.3$, tốt nhất trong sáu mô hình; AR(3), AR(1) và MA(2) kém hơn với AIC $214.0$, $218.5$ và $219.8$, còn MA(1) kém nhất với $237.2$. Phần dư của AR(2) và ARMA(1,1) giống nhiễu trắng, với $p=0.86$ và $0.88$ ở $Q_{LB}(10)$, còn MA(1) bị bác bỏ với $p=0.0002$. Với DR, AIC và BIC cùng chọn ARMA$(0,0)$, tức chỉ có trung bình, với AIC $5388.8$ và BIC $5399.9$ thấp hơn mọi lựa chọn $p,q\le2$ khác, đúng với một chuỗi gần nhiễu trắng; khoảng cách tới mô hình tốt thứ hai là $2.0$ theo AIC nhưng $7.5$ theo BIC, vì BIC phạt mỗi tham số nặng hơn.

### Nhận xét

*Đồng bộ từ báo cáo, mục 4.3(d).*

Kết quả **phù hợp** với lý thuyết: hai tiêu chuẩn cho kết quả đúng như lý thuyết mô tả, vì BIC gọn hơn AIC khi hai tiêu chuẩn khác nhau, như $p=2$ so với $11$ ở LY, và bậc được chọn không phải bậc “thật”. Ở HU, hai mô hình tương đương cho thấy AIC và BIC chỉ phân biệt được khi có khác biệt đáng kể, còn ở DR cả hai tiêu chuẩn cùng chọn đúng mô hình không có cấu trúc.

## 11.14 Chẩn đoán phần dư và checklist báo cáo ARMA

*Bộ dữ liệu:* HU, LY, DR

### Nội dung lý thuyết

Phần dư $\hat\varepsilon_t$ của mô hình tốt trông như nhiễu trắng: đồ thị, ACF, $Q_{LB}(H)$ so với $\chi^2_{H-p-q}$ (tham số ARMA ước lượng làm mất $p+q$ bậc tự do),
và ACF của $\hat\varepsilon_t^2$ (cụm biến động). **Checklist:** (1) vẽ đồ thị → (2) làm dừng → (3) ACF/PACF chọn ứng viên → (4) khớp ML, nhân quả/khả nghịch, AIC/BIC → (5) phần dư trắng? Nếu không, hiệu chỉnh.
Báo cáo: biến đổi đã làm, mô hình + SE + $\hat\sigma^2$ + nghiệm, cách chọn bậc, chẩn đoán phần dư, SE có xét phụ thuộc.

### Cách triển khai

**[D]** Áp checklist cho: HU (xu thế tuyến tính + sai số AR(2), $H=10$), LY (AR(2) và AR(11), $H=20$), DR (mô hình chỉ có trung bình ⇒ phần dư $=r_t-\bar r$, $H=10$). Mỗi mô hình:
$k$, AIC, BIC, nghiệm AR, $Q_{LB}$ của phần dư (df $=H-p-q$) và của phần dư bình phương; vẽ ACF phần dư và ACF phần dư bình phương. Các bước 1–3 đã thực hiện ở 11.1, 11.11, 11.13.

### Mã lệnh Python

In [ ]:
spec = [("HU", "AR(2) + xu thế", (2, 0, 0), 10), ("LY", "AR(2)", (2, 0, 0), 20), ("LY", "AR(11)", (11, 0, 0), 20), ("DR", "chỉ trung bình", (0, 0, 0), 10)]
rows = []; fig, axes = plt.subplots(len(spec), 2, figsize=(11.5, 11))
for i, (c, name, o, H) in enumerate(spec):
    res = fit_arma(c, o); e = np.asarray(res.resid); df = o[0] + o[2]
    q, p = A.ljung_box_manual(e, H, df); q_lib, p_lib = lb(e, H, df)
    q2, p2 = A.ljung_box_manual(e ** 2, H, 0)
    chk = R.causal_invertible_check(res.arparams, res.maparams)
    rows.append({"bộ": c, "mô hình": name, "k": len(res.params), "AIC": res.aic, "BIC": res.bic, "σ̂²": sig2(res),
                 "min|z| AR": chk["min_ar_modulus"], "nhân quả": chk["causal"], "H": H, "df": H - df,
                 "Q_LB phần dư": q, "p": p, "|tay − thư viện|": abs(q - q_lib), "Q_LB phần dư²": q2, "p (phần dư²)": p2})
    acf_bars(axes[i, 0], A.sample_acf(e, 20), len(e), f"{c}, {name}: ACF của phần dư")
    acf_bars(axes[i, 1], A.sample_acf(e ** 2, 20), len(e), f"{c}, {name}: ACF của phần dư²"); panel_label(axes[i, 0], "(" + "abcd"[i] + ")")
fig_path = save_figure(fig, "fig_b11_14_residual_acf")
tab = save_table(pd.DataFrame(rows).set_index(["bộ", "mô hình"]), "tab_b11_14_residual_diagnostics")
ols = sm.OLS(Y["HU"], sm.add_constant(X_HU)).fit(); ar2 = fit_arma("HU", (2, 0, 0), cov_type="oim")
print(f"HU hệ số dốc: OLS {ols.params[1]:.4f} (SE {ols.bse[1]:.4f}) so với ML sai số AR(2) {ar2.params[1]:.4f} (SE {ar2.bse[1]:.4f})")

### Kết quả

In [ ]:
show_figure(fig_path)
tab.T

### Phân tích và diễn giải

*Đồng bộ từ báo cáo, mục 4.3(e).*

Với HU, AR(2) có xu thế cho $Q_{LB}(10)=3.95$ với $p=0.86$ và phần dư bình phương có $p=0.17$: phần dư phù hợp với nhiễu trắng, và ngoài một gai ở độ trễ 9 của phần dư bình phương thì chưa thấy dấu hiệu phương sai thay đổi. Với LY, AR(2) bị bác bỏ vì $Q_{LB}(20)=35.2$ với $p=0.009$, còn AR(11) đạt với $Q_{LB}(20)=7.18$ và $p=0.62$, khớp nhận xét ở mục 4.1(d) rằng AR(2) bắt được chu kỳ nhưng chưa đủ. Với DR, mô hình chỉ có trung bình cho phần dư là nhiễu trắng với $p=0.78$, nhưng phần dư bình phương còn tự tương quan mạnh với $Q_{LB}(10)=108.7$ và $p<0.0001$. Con số này khác $110.7$ ở 4.3(a) vì ở đó là bình phương suất sinh lợi chưa trừ trung bình, còn ở đây là bình phương phần dư sau khi trừ trung bình. Như vậy chưa phát hiện tự tương quan còn lại trong trung bình ở $10$ độ trễ đầu, nhưng phương sai thì chưa được mô hình hóa, và cần các mô hình ngoài phạm vi báo cáo như GARCH. hình `fig_b11_14_residual_acf` cho thấy cùng các kết luận đó qua ACF của phần dư và của phần dư bình phương.

### Nhận xét

*Đồng bộ từ báo cáo, mục 4.3(e).*

Kết quả **phù hợp** với lý thuyết: bộ công cụ chẩn đoán làm đúng việc lý thuyết mô tả, vì xác nhận HU, phát hiện AR(2) chưa đủ cho LY và phát hiện cụm biến động còn lại ở DR dù phần dư không tự tương quan. Các giá trị tự cài đặt trùng thư viện, với chênh lệch tối đa $10^{-15}$.

## Bảng tổng hợp: ID × bộ dữ liệu → phù hợp lý thuyết?

Nguyên liệu cho Mục 4.6 của báo cáo (Bài 11). Cột cuối lấy từ Bảng 7.1 của báo cáo (P: phù hợp, M: phù hợp một phần, K: không phù hợp).

| ID | Nội dung | Bộ | Phù hợp lý thuyết? |
|---|---|---|---|
| 11.1 | Chuỗi thời gian, dừng và nhiễu trắng | HU, LY, DR | HU: P, LY: P, DR: P (mục 4.1(a)) |
| 11.2 | Tính chất của hàm tự hiệp phương sai | HU, LY, DR | HU: P, LY: P, DR: P (mục 4.1(b)) |
| 11.3 | Trung bình mẫu khi có phụ thuộc | HU, LY, DR | HU: P, LY: P, DR: P (mục 4.1(c)) |
| 11.4 | ACF mẫu, dải iid, Bartlett và kiểm định Ljung–Box | HU, LY, DR | HU: P, LY: P, DR: P (mục 4.3(a)) |
| 11.5 | Quá trình tuyến tính, toán tử lùi, trọng số ψ và ACF lý thuyết | HU, LY, DR | HU: P, LY: P, DR: P (mục 4.1(d)) |
| 11.6 | Mô hình AR(1) | HU, LY, DR | HU: M, LY: M, DR: P (mục 4.2(a)) |
| 11.7 | AR(p): tính nhân quả và phương trình Yule–Walker | HU, LY, DR | HU: P, LY: P, DR: P (mục 4.2(b)) |
| 11.8 | AR(2): tam giác nhân quả và chu kỳ | HU, LY, DR | HU: P, LY: P, DR: M (mục 4.2(c)) |
| 11.9 | Mô hình MA(q): ACF cắt cụt, MA(1), khả nghịch | HU, LY, DR | HU: P, LY: K, DR: P (mục 4.2(d)) |
| 11.10 | ARMA(p, q): nhân quả, khả nghịch, nghiệm chung | HU, LY, DR | HU: P, LY: P, DR: P (mục 4.2(e)) |
| 11.11 | Tự tương quan riêng phần (PACF) | HU, LY, DR | HU: P, LY: P, DR: P (mục 4.3(b)) |
| 11.12 | Ước lượng: Yule–Walker, OLS, cực đại hợp lý | HU, LY, DR | HU: P, LY: M, DR: P (mục 4.3(c)) |
| 11.13 | Chọn bậc bằng AIC/BIC và thẩm định cuốn chiếu | HU, LY, DR | HU: P, LY: P, DR: P (mục 4.3(d)) |
| 11.14 | Chẩn đoán phần dư và checklist báo cáo ARMA | HU, LY, DR | HU: P, LY: P, DR: P (mục 4.3(e)) |

# Bài 12 – Chuỗi thời gian II: ARIMA và dự báo

Notebook kiểm chứng **Bài 12** trên 3 bộ dữ liệu NI (Nile), AP (AirPassengers), DP (log giá DAX). Mỗi mục có đúng 6 tiểu mục theo đặc tả: *Nội dung lý thuyết*, *Cách triển khai*,
*Mã lệnh Python*, *Kết quả*, *Phân tích và diễn giải*, *Nhận xét*. Mọi con số ở đây do code sinh ra; hai tiểu mục cuối được đồng bộ từ báo cáo (`report/chapters/`).

Ký hiệu: **[D]** tính trên dữ liệu thật; **[M]** mô phỏng có điều khiển (seed 3179, `S.child_rng(...)`, kèm MCSE).
Hình lưu ở `outputs/figures/` (PNG 300 dpi), bảng ở `outputs/tables/` (csv, md, tex; tên ổn định).

*Cảnh báo của statsmodels* (khởi tạo tham số MA, hội tụ, nội suy p-value của KPSS) được ẩn có chủ ý; bài giảng cũng ghi nhận chúng không ảnh hưởng các con số.

In [ ]:
import warnings; warnings.filterwarnings("ignore")
import sys
import numpy as np, pandas as pd, scipy, statsmodels, matplotlib
import statsmodels.api as sm
import matplotlib.pyplot as plt
from IPython.display import display
from scipy import stats
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.tsa.statespace.structural import UnobservedComponents
from statsmodels.tsa.holtwinters import SimpleExpSmoothing
from statsmodels.tsa.stattools import acf, adfuller, kpss, coint
from statsmodels.stats.diagnostic import acorr_ljungbox

from as3179ts.config import SEED, AP_TRAIN_N, DP_TRAIN_N
from as3179ts.data import load_processed
from as3179ts import acf_tools as A, arma_tools as R, forecast_tools as F, evaluation as E, simulation as S
from as3179ts.plotting import save_figure, show_figure, acf_bars, panel_label, overlay_theory, BLUE, ORANGE, GREEN, RED, PURPLE, GREY
from as3179ts.reporting import save_table, sci
pd.options.display.float_format = lambda v: f"{v:.4g}"
warnings.filterwarnings("ignore")  # statsmodels bật lại cảnh báo khi import nên tắt lại ở đây

print("Python", sys.version.split()[0], "| numpy", np.__version__, "| pandas", pd.__version__, "| scipy", scipy.__version__,
      "| statsmodels", statsmodels.__version__, "| matplotlib", matplotlib.__version__)
print("SEED =", SEED)
rng = np.random.default_rng(SEED)  # generator gốc; mỗi mục dùng S.child_rng(12, k) nên kết quả không phụ thuộc thứ tự chạy ô

DATA = {c: load_processed(c) for c in ("NI", "AP", "DP")}
NAMES = {"NI": "Nile", "AP": "AirPassengers (log, huấn luyện)", "DP": "log giá DAX"}
NI = DATA["NI"].flow.to_numpy()
AP = DATA["AP"].passengers.to_numpy(dtype=float)           # thang gốc, 144 tháng
LAP = DATA["AP"].log_passengers.to_numpy()                  # log, 144 tháng
TR = LAP[:AP_TRAIN_N]                                       # huấn luyện 120 tháng (1949–1958)
DAX = DATA["DP"].dax_close.to_numpy()
LD = DATA["DP"].log_close.to_numpy()                        # log giá, 1860 ngày giao dịch
SERIES = {"NI": NI, "AP": TR, "DP": LD}
TIME = {"NI": DATA["NI"].year.to_numpy(), "AP": 1949 + np.arange(120) / 12, "DP": DATA["DP"].t.to_numpy()}   # trục thời gian thật
XLAB = {"NI": "năm", "AP": "năm", "DP": "ngày giao dịch t"}
YLAB = {"NI": "lưu lượng (10⁸ m³)", "AP": "log(số hành khách)", "DP": "log DAX"}
PLAB = {"ma.L1": "θ", "ma.S.L12": "Θ (mùa vụ)", "ar.L1": "φ", "sigma2": "σ²", "x1": "μ_W (độ trôi)", "const": "hằng số"}


def sig2(res):
    """σ̂² của kết quả ARIMA/SARIMAX (nằm trong params)."""
    return float(dict(zip(res.param_names, res.params))["sigma2"])


def lb(resid, H, model_df=0):
    r = acorr_ljungbox(resid, lags=[H], model_df=model_df)
    return float(r["lb_stat"].iloc[0]), float(r["lb_pvalue"].iloc[0])


def airline(z):
    """SARIMA(0,1,1)(0,1,1)₁₂ không hằng số (Bài 12, Mục 6)."""
    return SARIMAX(z, order=(0, 1, 1), trend="n", seasonal_order=(0, 1, 1, 12)).fit(disp=0)

## 12.1 Xu thế tất định và xu thế ngẫu nhiên; I(d)

*Bộ dữ liệu:* NI, AP, DP

### Nội dung lý thuyết

**Định nghĩa 1.** Dừng quanh xu thế: $Y_t=\mu_t+X_t$ ($X$ dừng yếu); tích hợp bậc $d$, $Y\sim I(d)$: $\nabla^dY_t$ dừng yếu với phương sai dài hạn $v\in(0,\infty)$ và
$\nabla^{d-1}Y_t$ không dừng. Cú sốc lên chuỗi dừng quanh xu thế tắt dần; cú sốc lên chuỗi I(1) dịch chuyển cả quỹ đạo tương lai. Bước ngẫu nhiên có trôi $Y_t=\delta+Y_{t-1}+\varepsilon_t$ là trường hợp điển hình của I(1).

### Cách triển khai

**[D]** Với mỗi bộ (AP: log, 120 tháng huấn luyện): vẽ chuỗi, ACF của chuỗi gốc, ACF của chuỗi sai phân $\nabla y$; thêm ACF của phần dư sau khi khử xu thế tuyến tính bằng OLS
(đối chứng với xu thế tất định). Bảng $\hat\rho(1)$, $\hat\rho(10)$ và độ lệch chuẩn. Sai phân mùa vụ của AP xem 12.10.

### Mã lệnh Python

In [ ]:
fig, axes = plt.subplots(3, 3, figsize=(13, 8.8), gridspec_kw={"width_ratios": [1.5, 1, 1]}); rows = []
for i, (c, y) in enumerate(SERIES.items()):
    n = len(y); d1 = np.diff(y); t = np.arange(n)
    detr = y - np.polyval(np.polyfit(t, y, 1), t)
    axes[i, 0].plot(TIME[c], y, lw=0.9, color=BLUE); axes[i, 0].set(title=NAMES[c], xlabel=XLAB[c], ylabel=YLAB[c]); panel_label(axes[i, 0], "(" + "abc"[i] + ")")
    acf_bars(axes[i, 1], A.sample_acf(y, 30), n, f"{c}: ACF của chuỗi gốc y"); acf_bars(axes[i, 2], A.sample_acf(d1, 30), len(d1), f"{c}: ACF của sai phân ∇y")
    rows.append({"bộ": c, "n": n, "SD của y": y.std(ddof=1), "SD của ∇y": d1.std(ddof=1), "ρ̂(1) của y": A.sample_acf(y, 10)[1], "ρ̂(10) của y": A.sample_acf(y, 10)[10],
                 "ρ̂(1) sau khử xu thế OLS": A.sample_acf(detr, 10)[1], "ρ̂(10) sau khử xu thế OLS": A.sample_acf(detr, 10)[10], "ρ̂(1) của ∇y": A.sample_acf(d1, 10)[1]})
fig_path = save_figure(fig, "fig_b12_01_levels_vs_differences")
tab = save_table(pd.DataFrame(rows).set_index("bộ"), "tab_b12_01_levels_vs_differences")

### Kết quả

In [ ]:
show_figure(fig_path)
tab

### Phân tích và diễn giải

*Đồng bộ từ báo cáo, mục 5.1(a).*

**NI.** Chuỗi không có xu thế dài mà chỉ đổi mức quanh năm 1899. Sai phân gần như không giảm độ lệch chuẩn, chỉ từ $169.2$ xuống $168.1$, và đưa $\hat\rho(1)$ về $-0.40$, một giá trị âm đáng kể, dấu hiệu của sai phân thừa sẽ được xét ở mục 5.3. NI không thuộc hẳn loại nào trong hai loại xu thế.

**AP.** $\hat\rho(1)=0.95$ và $\hat\rho(10)=0.71$. Trừ xu thế kéo $\hat\rho(10)$ xuống $0.18$, nghĩa là phần lớn tính dai dẳng đến từ xu thế, và sai phân giảm độ lệch chuẩn từ $0.39$ xuống $0.11$. Nhưng ACF của sai phân vẫn có đỉnh lớn ở độ trễ 12 và 24, tức mùa vụ chưa được xử lý, như sẽ xét ở mục 5.2. Chỉ riêng ACF khó phân biệt hai loại xu thế ở đây.

**DP.** Đây là ví dụ rõ nhất của xu thế ngẫu nhiên: $\hat\rho(1)=0.998$ và $\hat\rho(10)=0.979$, và trừ xu thế gần như không thay đổi gì, chỉ còn $0.996$ và $0.960$. Sai phân đưa chuỗi về gần nhiễu trắng với $\hat\rho(1)=-0.0004$, và độ lệch chuẩn giảm từ $0.363$ xuống $0.0103$.

### Nhận xét

*Đồng bộ từ báo cáo, mục 5.1(a).*

Kết quả **phù hợp một phần** với lý thuyết. DP khớp hoàn toàn lý thuyết. Với AP, trừ xu thế làm giảm mạnh tự tương quan nhưng sai phân còn mùa vụ, nên ACF chưa đủ để kết luận. NI chỉ đổi mức, không thuộc cả hai loại.

## 12.2 Bước ngẫu nhiên

*Bộ dữ liệu:* NI, AP, DP

### Nội dung lý thuyết

**Định lý 1.** $Y_t=Y_0+\delta t+\sum_{j\le t}\varepsilon_j$: $EY_t=Y_0+\delta t$, $\operatorname{Var}Y_t=t\sigma^2$, $\operatorname{Cov}(Y_s,Y_t)=\min(s,t)\sigma^2$,
$\operatorname{Corr}(Y_t,Y_{t+h})=\sqrt{t/(t+h)}\to1$; $\hat\rho(h)\to1$ nên biểu đồ tương quan giảm rất chậm. Hệ quả: hồi quy giả mạo giữa hai bước ngẫu nhiên độc lập (Bài 11, Ví dụ 1.2).

### Cách triển khai

**[M]** Ước lượng $\hat\delta,\hat\sigma$ từ 1859 suất sinh lợi log của DP, mô phỏng $B=2000$ bước ngẫu nhiên dài 250: so $\operatorname{Var}Y_t$ với $t\hat\sigma^2$, $\operatorname{Corr}(Y_t,Y_{t+h})$ với $\sqrt{t/(t+h)}$;
ACF mẫu trung bình của bước ngẫu nhiên $n=200$. **Hồi quy giả mạo:** hai bước ngẫu nhiên độc lập $n=100$, OLS $Y$ theo $X$, tần suất bác bỏ $H_0:\beta_1=0$ ở 5%.
NI và AP không phải bước ngẫu nhiên (NI có đổi mức, AP có mùa vụ) nên không dùng làm "sự thật" mô phỏng.

Ta kiểm tra gia số của cả ba chuỗi bằng Ljung–Box (10 độ trễ ở NI và DP, 12 ở AP). AP và DP dùng mẫu huấn luyện.

### Mã lệnh Python

In [ ]:
r = np.diff(LD); delta, sigma = r.mean(), r.std(ddof=1)
rng = S.child_rng(12, 2); B, T = 2000, 250
paths = np.array([S.simulate_random_walk(T, rng, delta, sigma) for _ in range(B)])   # Y_t, t = 1..T (Y_0 = 0)
ts = [1, 10, 50, 100, 250]
tab_var = pd.DataFrame({"t": ts, "t·σ̂² (lý thuyết)": [t * sigma ** 2 for t in ts], "Var mô phỏng": [paths[:, t - 1].var(ddof=1) for t in ts],
                        "E Yₜ = δ̂t": [delta * t for t in ts], "trung bình mô phỏng": [paths[:, t - 1].mean() for t in ts]}).set_index("t")
tab_var["tỷ số Var"] = tab_var["Var mô phỏng"] / tab_var["t·σ̂² (lý thuyết)"]
pairs = [(50, 10), (100, 50), (100, 150)]
tab_corr = pd.DataFrame({"(t, h)": [str(p) for p in pairs], "√(t/(t+h))": [np.sqrt(t / (t + h)) for t, h in pairs],
                         "tương quan mô phỏng": [np.corrcoef(paths[:, t - 1], paths[:, t + h - 1])[0, 1] for t, h in pairs]}).set_index("(t, h)")
tab_var = save_table(tab_var, "tab_b12_02_rw_variance"); tab_corr = save_table(tab_corr, "tab_b12_02_rw_correlation")
fig, axes = plt.subplots(1, 2, figsize=(11.5, 4.0)); tgrid = np.arange(1, T + 1)
axes[0].plot(tgrid, paths.var(axis=0, ddof=1), color=BLUE, label="Var mô phỏng (B = 2000)"); axes[0].plot(tgrid, tgrid * sigma ** 2, color=ORANGE, ls="--", label="t·σ̂² (lý thuyết)")
axes[0].set(xlabel="t (ngày giao dịch)", ylabel="Var Yₜ", title="Phương sai tăng tuyến tính theo t"); axes[0].legend(loc="upper left")
for pth in paths[:50]: axes[1].plot(tgrid, pth, lw=0.5, color=BLUE, alpha=0.5)
axes[1].plot(tgrid, delta * tgrid, color=RED, lw=1.8, label="E Yₜ = δ̂t"); axes[1].legend(loc="upper left")
axes[1].set(xlabel="t (ngày giao dịch)", ylabel="Yₜ (log giá, Y₀ = 0)", title="50 đường mô phỏng của bước ngẫu nhiên có trôi")
panel_label(axes[0], "(a)"); panel_label(axes[1], "(b)"); fig_path = save_figure(fig, "fig_b12_02_rw_variance")
rho_bar = np.mean([A.sample_acf(S.simulate_random_walk(200, rng, 0.0, 1.0), 20) for _ in range(B)], axis=0)
tab_rho = save_table(pd.DataFrame({"trung bình ρ̂(h), n = 200": rho_bar[[1, 2, 5, 10, 20]]}, index=pd.Index([1, 2, 5, 10, 20], name="h")), "tab_b12_02_rw_sample_acf")

In [ ]:
rng = S.child_rng(12, 20); B, n = 2000, 100
crit = stats.t.ppf(0.975, n - 2); reject = 0
for _ in range(B):
    x, y = S.simulate_random_walk(n, rng), S.simulate_random_walk(n, rng)
    xc = x - x.mean(); b1 = (xc @ (y - y.mean())) / (xc @ xc)
    resid = (y - y.mean()) - b1 * xc; se = np.sqrt(resid @ resid / (n - 2) / (xc @ xc))
    reject += abs(b1 / se) > crit
tab_spur = save_table(pd.DataFrame({"tần suất bác bỏ β₁ = 0 (mức 5%)": [reject / B], "MCSE": [np.sqrt(reject / B * (1 - reject / B) / B)], "danh nghĩa": [0.05]},
                                   index=["hai bước ngẫu nhiên độc lập, n = 100"]), "tab_b12_02_spurious_regression")

In [ ]:
# Với bước ngẫu nhiên có trôi, gia số không tự tương quan.
for rw_name, rw_series, rw_lag in (("NI", NI, 10), ("AP", TR, 12), ("DP", LD[:DP_TRAIN_N], 10)):
    rw_inc = np.diff(rw_series)
    print(rw_name, "trôi mẫu, ACF(1), p Ljung–Box:", rw_inc.mean(),
          A.sample_acf(rw_inc, 1)[1], A.ljung_box_manual(rw_inc, rw_lag)[1])

### Kết quả

In [ ]:
show_figure(fig_path)
print(f'δ̂ = {delta:.6f}, σ̂ = {sigma:.6f} (từ {len(r)} suất sinh lợi log)')
display(tab_var)
display(tab_corr)
display(tab_rho)
tab_spur

### Phân tích và diễn giải

*Đồng bộ từ báo cáo, mục 5.1(b).*

Phương sai mô phỏng bám sát $t\hat\sigma^2$ ở mọi mốc (tỷ số $1.00$ đến $1.06$) và trung bình mô phỏng gần $\hat\delta t$. Tương quan mô phỏng $0.911$, $0.824$, $0.639$ gần $\sqrt{t/(t+h)}=0.913$, $0.816$, $0.632$ ở ba cặp $(t,h)$ được thử. ACF mẫu của bước ngẫu nhiên $n=200$ giảm chậm gần tuyến tính, với giá trị $0.963$, $0.927$, $0.825$, $0.675$ và $0.432$ ở các độ trễ 1, 2, 5, 10 và 20, nhỏ hơn $1$ vì $n$ hữu hạn. Hậu quả rõ nhất là hồi quy giả mạo: hai bước ngẫu nhiên độc lập mà kiểm định vẫn bác bỏ $\beta_1=0$ tới $76.9%$ số lần, với sai số Monte Carlo $0.009$, gấp hơn $15$ lần mức $5%$. Với phần mở rộng, hồi quy log DAX theo log SMI cho $\hat\beta=0.82$ và $R^2=0.965$, nhưng Engle–Granger cho $-2.50$ với $p=0.28$ không bác bỏ không đồng tích hợp: $R^2$ cao giữa hai chuỗi $I(1)$ không phải bằng chứng đồng tích hợp.

Nếu là bước ngẫu nhiên với gia số không tương quan, gia số phải vượt qua kiểm định Ljung–Box. NI cho $p=0.00078$ và AP cho $p<10^{-20}$, bác bỏ giả thiết này. DP cho $p=0.404$ trên mẫu huấn luyện, phù hợp về tự tương quan tuyến tính; phương sai thay đổi vẫn là hạn chế.

### Nhận xét

*Đồng bộ từ báo cáo, mục 5.1(b).*

Mô phỏng trên tham số DP **phù hợp** với lý thuyết: phương sai, tương quan, dạng ACF và hồi quy giả mạo đều khớp lý thuyết trong dung sai với tỷ số phương sai lệch tối đa $6.4%$, khoảng $2$ lần sai số mô phỏng, và phần mở rộng cũng đúng với cảnh báo của lý thuyết.

Với dữ liệu thực, mô hình bước ngẫu nhiên bị bác bỏ trên NI và AP theo kiểm định gia số.

## 12.3 Sai phân sửa được gì và làm hỏng gì: sai phân quá mức

*Bộ dữ liệu:* NI, AP, DP

### Nội dung lý thuyết

**Mệnh đề 1.** $\nabla$ của chuỗi dừng có phương sai dài hạn $\sum_h\gamma_{\nabla X}(h)=0$ (sai phân quá mức). Nếu $X_t\sim WN$ thì $\nabla X_t$ là MA(1) với $\theta=-1$, $\rho(1)=-\tfrac12$,
đa thức MA có nghiệm trên vòng tròn đơn vị (không khả nghịch, $\hat\theta\to-1$). Chi phí: dự báo bước ngẫu nhiên cho nhiễu trắng có sai số $2\sigma^2$ (gấp đôi); thiếu sai phân: dự báo bước ngẫu nhiên bằng
$\bar Y_n$ có $\operatorname{Var}=\sigma^2\{(n-1)(2n-1)/(6n)+h\}$ ($33.8\sigma^2$ khi $n=100,h=1$).

### Cách triển khai

**[D]** NI và DP: $\hat\rho(1)$, độ lệch chuẩn, phương sai dài hạn (Bartlett cắt cụt, $K=\lfloor4(n/100)^{2/9}\rfloor$) và $\hat\theta$ của MA(1) cho $y$, $\nabla y$, $\nabla^2y$.
**[M]** nhiễu trắng $n=400$, $B=1000$: $\hat\rho(1)$ của $\nabla\varepsilon$; sai số dự báo của bước ngẫu nhiên (nhiễu trắng) và của $\bar Y_n$ (bước ngẫu nhiên), $B=5000$.

Với AP ta so chuỗi log sau sai phân thường và mùa vụ với chính chuỗi đó sau khi sai phân thường thêm một lần.

### Mã lệnh Python

In [ ]:
def long_run_var(w, K=None):
    """Ước lượng Bartlett của phương sai dài hạn với K độ trễ (mặc định K = ⌊4(n/100)^{2/9}⌋)."""
    n = len(w); K = int(np.floor(4 * (n / 100) ** (2 / 9))) if K is None else K; g = A.sample_autocov(w, K)
    return g[0] + 2 * np.sum((1 - np.arange(1, K + 1) / (K + 1)) * g[1:])
rows = []
for c, y in (("NI", NI), ("DP", LD)):
    for name, w, trend in (("∇y", np.diff(y), "c"), ("∇²y (quá mức)", np.diff(y, 2), "n")):
        m = ARIMA(w, order=(0, 0, 1), trend=trend).fit(); th = m.maparams[0]
        rows.append({"bộ": c, "chuỗi": name, "n": len(w), "sd": w.std(ddof=1), "ρ̂(1)": A.sample_acf(w, 1)[1], "phương sai dài hạn Bartlett": long_run_var(w),
                     "θ̂ MA(1)": th, "SE(θ̂)": m.bse[list(m.param_names).index("ma.L1")]})
tab = save_table(pd.DataFrame(rows).set_index(["bộ", "chuỗi"]), "tab_b12_03_overdifferencing")

# Ước lượng Bartlett của chuỗi sai phân thừa W = ∇X theo K, so với giá trị đóng 2(γ̂_X(0) − γ̂_X(K+1))/(K+1)
rows_k = []
for c, y in (("NI", NI), ("DP", LD)):
    x = np.diff(y)
    for K in (3, 7, 15, 31, 63):
        gx = A.sample_autocov(x, K + 1)
        rows_k.append({"bộ": c, "K": K, "Bartlett của ∇y": long_run_var(x, K), "Bartlett của ∇²y": long_run_var(np.diff(y, 2), K),
                       r"giá trị đóng $2(\hat\gamma_X(0)-\hat\gamma_X(K+1))/(K+1)$": 2 * (gx[0] - gx[K + 1]) / (K + 1)})
tab_k = save_table(pd.DataFrame(rows_k).set_index(["bộ", "K"]), "tab_b12_03_bartlett_by_k")

In [ ]:
rng = S.child_rng(12, 3)
r1 = [A.sample_acf(np.diff(rng.normal(size=400)), 1)[1] for _ in range(1000)]
B, n = 5000, 100; e_rw, e_mean = [], []
for _ in range(B):
    e = rng.normal(size=n + 1); e_rw.append(e[n] - e[n - 1])                 # Y_t = μ + ε_t, dự báo bước ngẫu nhiên Ŷ = Y_n
    w = np.cumsum(rng.normal(size=n + 1)); e_mean.append(w[n] - w[:n].mean())  # bước ngẫu nhiên, dự báo bằng trung bình mẫu
tab_cost = save_table(pd.DataFrame({
    "giá trị mô phỏng": [np.mean(r1), np.var(e_rw, ddof=1), np.var(e_mean, ddof=1)],
    "lý thuyết": [-0.5, 2.0, (n - 1) * (2 * n - 1) / (6 * n) + 1],
    "MCSE": [np.std(r1, ddof=1) / np.sqrt(len(r1)), np.var(e_rw, ddof=1) * np.sqrt(2 / (B - 1)), np.var(e_mean, ddof=1) * np.sqrt(2 / (B - 1))]},
    index=["trung bình ρ̂(1) của ∇ε (nhiễu trắng)", "Var(Yₙ₊₁ − Yₙ) khi Y là nhiễu trắng (σ² = 1)", "Var(Yₙ₊₁ − Ȳₙ) khi Y là bước ngẫu nhiên, n = 100"]), "tab_b12_03_cost_simulation")

In [ ]:
# AP • Sau sai phân thường và mùa vụ, thử lấy thêm một sai phân thường.
ap_stationary = np.diff(TR[12:] - TR[:-12])
ap_extra_fit = ARIMA(np.diff(ap_stationary), order=(0, 0, 1), trend="n").fit()
print("AP: ACF(1) trước/sau, theta MA(1), môđun nghiệm",
      A.sample_acf(ap_stationary, 1)[1], A.sample_acf(np.diff(ap_stationary), 1)[1],
      ap_extra_fit.maparams[0], abs(ap_extra_fit.maroots[0]))

### Kết quả

In [ ]:
display(tab)
tab_cost

### Phân tích và diễn giải

*Đồng bộ từ báo cáo, mục 5.3(a).*

Với NI, sai phân một lần cho $\hat\rho(1)=-0.40$ và $\hat\theta=-0.765$, phù hợp với ARIMA$(0,1,1)$ ở mục 5.2; giá trị này khác $-0.733$ ở 5.2(a) vì ở đây MA(1) được khớp cho $\nabla$NI có hằng số, tức có độ trôi, còn ở 5.2(a) mô hình không có độ trôi; sai phân lần hai đưa $\hat\rho(1)$ xuống $-0.63$, $\hat\theta=-0.999$ sát biên với sai số chuẩn $2.1$, và độ lệch chuẩn tăng từ $168$ lên $283$: đây là sai phân thừa rõ ràng. Với DP, sai phân một lần cho gần nhiễu trắng ($\hat\rho(1)=-0.0004$); sai phân lần hai tạo đúng sai phân thừa của lý thuyết: $\hat\rho(1)=-0.486$ sát $-\tfrac12$, $\hat\theta=-0.9986$, độ lệch chuẩn tăng từ $0.0103$ lên $0.0146$, xấp xỉ $\sqrt2$ lần. Phương sai dài hạn ước lượng theo Bartlett giảm ở DP, từ $9.7\times10^{-5}$ xuống $2.7\times10^{-5}$, nhưng chưa về $0$, còn ở NI thì tăng từ $10252$ lên $15391$. Điều này đến từ chính ước lượng Bartlett chứ không mâu thuẫn với lý thuyết. Với $W=\nabla X$, tổng Bartlett $K$ độ trễ $\sum_{|h|\le K}\bigl(1-\tfrac{|h|}{K+1}\bigr)\gamma_W(h)$ bằng đúng $2\bigl(\gamma_X(0)-\gamma_X(K+1)\bigr)/(K+1)$, vì $\gamma_W(h)=2\gamma_X(h)-\gamma_X(h-1)-\gamma_X(h+1)$ và sai phân bậc hai của trọng số tam giác chỉ khác $0$ tại $h=0$ và $h=\pm(K+1)$. Đại lượng này giảm như $1/(K+1)$ chứ không bằng $0$ ở $K$ hữu hạn. Khi $X$ là nhiễu trắng, như $\nabla\log$ DAX, nó bằng $2\sigma^2/(K+1)=\sigma^2/4$ với $K=7$, khớp với mức $2.7\times10^{-5}\approx\hat\sigma^2/4$. Thay $\gamma_X$ bằng ước lượng mẫu, công thức cho $15221$ ở NI và $2.67\times10^{-5}$ ở DP, sát hai giá trị trong bảng. bảng `tab_b12_03_bartlett_by_k` cho thấy khi tăng $K$, ước lượng của chuỗi sai phân thừa giảm đúng theo $1/(K+1)$ về $0$. Riêng NI, $\hat\rho(1)=-0.63$ của $\nabla^2$NI nằm ngoài $[-\tfrac12,\tfrac12]$, miền cho phép của MA(1) theo 2.1.2(d). Lý do là $\nabla$NI vốn đã có $\hat\rho(1)=-0.40$, nên $\nabla^2$NI gần một MA(2) có nghiệm đơn vị chứ không phải MA(1); $\hat\theta=-0.999$ với sai số chuẩn $2.1$ vì vậy phản ánh cả việc khớp sai mô hình lẫn sai phân thừa. Mô phỏng xác nhận phần còn lại, đều trong dung sai $3$ MCSE: trung bình $\hat\rho(1)$ của nhiễu trắng đã sai phân là $-0.497$, so với lý thuyết là $-0.5$, lệch $2.6$ MCSE; dự báo nhiễu trắng bằng bước ngẫu nhiên có phương sai sai số $2.005$, so với lý thuyết là $2$ và gấp đôi mức tối ưu $1$, lệch $0.1$ MCSE; dự báo bước ngẫu nhiên bằng trung bình mẫu có phương sai $33.7$, so với lý thuyết là $33.8$ và gấp $33$ lần mức tối ưu, lệch $0.2$ MCSE.

Trên AP, sai phân thường bổ sung làm ACF bậc một giảm từ $-0.328$ xuống $-0.660$. MA(1) trên chuỗi mới cho $\hat\theta=-0.99995$, môđun nghiệm 1.00005. Nghiệm gần biên là dấu hiệu sai phân thừa; giá trị $-1/2$ chỉ là mốc cho trường hợp chuỗi trước sai phân là nhiễu trắng.

### Nhận xét

*Đồng bộ từ báo cáo, mục 5.3(a).*

Kết quả **phù hợp** với lý thuyết. Dấu hiệu chính của sai phân thừa ($\hat\rho(1)\to-0.5$, $\hat\theta\to-1$) và hai chi phí được xác nhận chính xác. Phương sai dài hạn ước lượng chưa về $0$ ở $K=3$ (NI) và $K=7$ (DP) đúng như giá trị $2(\gamma_X(0)-\gamma_X(K+1))/(K+1)$ của ước lượng Bartlett, và giảm về $0$ theo $1/(K+1)$ khi tăng $K$, nên đây không phải điểm lệch so với lý thuyết. Với NI cần lưu ý thêm: $\hat\rho(1)$ vượt $-\tfrac12$ cho thấy $\nabla^2$NI không phải MA(1) đơn thuần, nên ngoài sai phân thừa còn có sai mô hình.

## 12.4 Kiểm định nghiệm đơn vị: ADF và KPSS

*Bộ dữ liệu:* NI, AP, DP

### Nội dung lý thuyết

**Định lý 2.** Dickey–Fuller: dưới $H_0:\phi=1$, $\tau=(\hat\phi-1)/SE\to\{W(1)^2-1\}/\{2(\int W^2)^{1/2}\}$ (không chuẩn tắc, dịch về phía âm); điểm tới hạn 5%: $-1.95$ (không hệ số chặn),
$-2.86$ (có chặn), $-3.41$ (chặn và xu thế). ADF thêm các độ trễ $\nabla Y_{t-i}$. KPSS đảo giả thuyết không: $H_0$ dừng quanh mức/xu thế. ADF bác bỏ & KPSS không ⇒ I(0);
ngược lại ⇒ I(1); cùng bác bỏ hoặc cùng không bác bỏ ⇒ dữ liệu không khớp khuôn mẫu nào (thường có điểm gãy).

### Cách triển khai

**[D]** ADF (`c`, `ct`) và KPSS (`c`, `ct`) cho chuỗi gốc, ADF/KPSS (`c`) cho $\nabla y$ (AP: log huấn luyện). **[M]** dưới $H_0$ (bước ngẫu nhiên $n=200$, $B=2000$): phân phối của $\tau$ (5% phân vị, trung bình, tần suất < −1.645),
và công suất của ADF (AR(1) với $\phi=0.95,0.9,0.8$, $n=100,200$; iid + dịch mức 2 ở nửa sau).

### Mã lệnh Python

In [ ]:
def read(y, test, reg):
    if test == "ADF":
        s, p, lag, _, crit, _ = adfuller(y, regression=reg, autolag="AIC"); return s, p, crit["5%"], lag
    s, p, lag, crit = kpss(y, regression=reg, nlags="auto"); return s, p, crit["5%"], lag
rows = []
for c, y in SERIES.items():
    for tr_name, w, regs in (("y", y, ("c", "ct")), ("∇y", np.diff(y), ("c",))):
        for test in ("ADF", "KPSS"):
            for reg in regs:
                s, p, crit, extra = read(w, test, reg)
                rej = s < crit if test == "ADF" else s > crit
                rows.append({"bộ": c, "chuỗi": tr_name, "kiểm định": test, "hồi quy": {"c": "hằng số", "ct": "hằng số + xu thế"}[reg], "thống kê": s,
                             "p-value": (("≤ 0.01" if p <= 0.01 else "≥ 0.10" if p >= 0.1 else f"{p:.3f}") if test == "KPSS" else p),
                             "tới hạn 5%": crit, "bác bỏ H₀ (5%)": rej, "độ trễ": extra})
tab = save_table(pd.DataFrame(rows).set_index(["bộ", "chuỗi", "kiểm định", "hồi quy"]), "tab_b12_04_unit_root_tests")

In [ ]:
rng = S.child_rng(12, 4); B = 2000
tau = np.array([adfuller(S.simulate_random_walk(200, rng), maxlag=0, regression="c", autolag=None)[0] for _ in range(B)])
tab_null = save_table(pd.DataFrame({"mô phỏng": [np.quantile(tau, 0.05), tau.mean(), np.mean(tau < -2.86), np.mean(tau < -1.645)],
                                    "lý thuyết / bảng": [-2.86, -1.53, 0.05, np.nan]},
                                   index=["phân vị 5% của τ (có chặn, n = 200)", "trung bình τ", "tần suất τ < −2.86", "tần suất τ < −1.645 (bảng chuẩn: sai)"]), "tab_b12_04_null_distribution")
fig, ax = plt.subplots(figsize=(7.8, 3.9)); xs = np.linspace(-5, 3.5, 400)
ax.hist(tau, bins=60, density=True, color=BLUE, alpha=0.55, label="τ mô phỏng (B = 2000)"); ax.plot(xs, stats.norm.pdf(xs), color=ORANGE, label="N(0, 1)")
ax.axvline(-2.86, color=RED, ls="--", label="tới hạn 5% của Dickey–Fuller = −2.86"); ax.axvline(-1.645, color=GREY, ls=":", label="tới hạn 5% của N(0, 1) = −1.645")
ax.set(xlabel="τ", ylabel="mật độ", title="Phân phối của τ dưới H₀ (bước ngẫu nhiên): dịch về phía âm, không chuẩn tắc"); ax.legend(loc="upper right")
fig_path = save_figure(fig, "fig_b12_04_dickey_fuller_null")
rows = []; Bp = 1000

def rejection_rates(series):
    """Tỷ lệ bác bỏ ở mức 5% của DF không độ trễ và của ADF chọn độ trễ theo AIC (mặc định của adfuller, như mã mẫu Bài 12)."""
    df0 = np.mean([adfuller(y, maxlag=0, regression="c", autolag=None)[1] < 0.05 for y in series])
    aic = np.mean([adfuller(y, regression="c", autolag="AIC")[1] < 0.05 for y in series])
    return {"DF (không độ trễ)": df0, "ADF (độ trễ theo AIC)": aic}

for phi in (0.95, 0.9, 0.8):
    for n in (100, 200):
        rr = S.child_rng(12, 40, int(phi * 100), n)
        sims = [S.simulate_ar1(phi, n, rr) for _ in range(Bp)]
        rows.append({"mô hình đúng": f"AR(1), φ = {phi}", "n": n, **rejection_rates(sims)})
rr = S.child_rng(12, 41); shift = np.r_[np.zeros(50), 2 * np.ones(50)]
sims = [rr.normal(size=100) + shift for _ in range(Bp)]
rows.append({"mô hình đúng": "iid N(0,1) + dịch mức 2 ở nửa sau", "n": 100, **rejection_rates(sims)})
tab_power = save_table(pd.DataFrame(rows).set_index(["mô hình đúng", "n"]), "tab_b12_04_adf_power")

### Kết quả

In [ ]:
display(tab)
display(tab_null)
show_figure(fig_path)
tab_power

### Phân tích và diễn giải

*Đồng bộ từ báo cáo, mục 5.3(b).*

**NI.** ADF bác bỏ nghiệm đơn vị với $-4.05$ và $p=0.001$, *và* KPSS cũng bác bỏ tính dừng với $0.869>0.463$: đây là khuôn mẫu “cùng bác bỏ” mà lý thuyết gắn với điểm gãy, ở đây là lần đổi mức quanh 1899. Sau sai phân, ADF bác bỏ với $-4.68$ và KPSS không bác bỏ với $0.117$. Hai kết quả ủng hộ tính dừng của chuỗi sai phân.

**AP.** ADF không bác bỏ với $p=0.32$, và $p=0.80$ khi có xu thế, còn KPSS bác bỏ với $1.744>0.463$, nên chuỗi nghiêng về $I(1)$; nhưng KPSS có xu thế không bác bỏ với $0.061<0.146$, nên chưa loại trừ được mức dừng quanh xu thế. Sau sai phân bậc một, ADF vẫn không bác bỏ với $p=0.158$, nhiều khả năng do mùa vụ chưa được loại bỏ, vì ACF ở độ trễ 12 vẫn lớn như đã thấy ở mục 5.1(a).

**DP.** ADF không bác bỏ với $p=0.996$ và KPSS bác bỏ rõ với $5.75$, nên hai kiểm định thống nhất ủng hộ nghiệm đơn vị. Sau sai phân, ADF bác bỏ mạnh với $-43.1$ và KPSS không bác bỏ với $0.437<0.463$, nên sai phân bậc một là đủ.

**Mô phỏng.** Dưới $H_0$ với $n=200$, phân vị $5%$ của $\tau$ là $-2.79$ so với $-2.86$ trong bảng, trung bình là $-1.51$ so với $-1.53$, và tần suất $\tau<-2.86$ là $4.2%$ so với $5%$ lý thuyết, lệch $1.7$ lần sai số Monte Carlo. Nếu dùng giá trị tới hạn chuẩn $-1.645$, tần suất bác bỏ sai lên $45.7%$, hơn chín lần mức danh nghĩa. Công suất thấp khi $\phi$ gần $1$: với DF không độ trễ, $\phi=0.95$ chỉ cho $13.8%$ ở $n=100$ và $32.7%$ ở $n=200$; $\phi=0.9$ cho $33.1%$ và $88.6%$; $\phi=0.8$ cho $89.6%$ và $100%$. ADF chọn độ trễ theo AIC cho các mức tương tự, từ $16.7%$ đến $98.3%$, sát các giá trị $0.14$ đến $0.99$ của bài giảng. Khác biệt lớn nằm ở nhiễu iid cộng dịch mức: DF không độ trễ bác bỏ $100%$ số lần, còn ADF chọn độ trễ theo AIC chỉ bác bỏ $8%$, đúng bằng con số $0.08$ của bài giảng. Đây là hiện tượng Perron: bước dịch mức tạo tự tương quan dương kéo dài trong chuỗi, AIC bù bằng cách thêm độ trễ, và kiểm định gần như mất hết công suất dù chuỗi không có nghiệm đơn vị. Khi bắt buộc không có độ trễ, hồi quy DF chỉ thấy $\hat\rho(1)\approx0.5$, xa $1$, nên bác bỏ. Tỷ lệ $8%$ chỉ mô tả thiết kế mô phỏng trên. Với chuỗi NI thực tế, AIC chọn một độ trễ và ADF bác bỏ nghiệm đơn vị ($p=0.001$) dù chuỗi có đổi mức. Vì vậy khi nghi có dịch mức, kết quả ADF cần được đọc cùng KPSS và đồ thị của chuỗi, như đã làm với NI ở trên.

### Nhận xét

*Đồng bộ từ báo cáo, mục 5.3(b).*

Kết quả **phù hợp** với lý thuyết: phân phối của $\tau$ khớp bảng Dickey–Fuller và khác hẳn phân phối chuẩn; công suất thấp gần nghiệm đơn vị; trong mô phỏng có dịch mức, nhiễu dừng quanh từng mức trung bình và ADF chọn độ trễ theo AIC ít bác bỏ nghiệm đơn vị; và việc đọc ADF cùng KPSS giải thích đúng từng chuỗi, nổi bật là Nile. Với AP kết quả chưa dứt khoát vì mùa vụ và xu thế đan xen, cần kiểm định riêng cho mùa vụ.

## 12.5 Mô hình ARIMA(p, d, q) và độ trôi

*Bộ dữ liệu:* NI, AP, DP

### Nội dung lý thuyết

**Định nghĩa 2.** $\phi(B)\nabla^dY_t=c+\theta(B)\varepsilon_t$ với $\phi,\theta$ không nghiệm trong $|z|\le1$ và không nghiệm chung; trung bình của chuỗi sai phân $\mu_W=c/\phi(1)$.
$d=1$: $\mu_W$ là độ trôi, dự báo dài hạn là đường thẳng có hệ số góc $\mu_W$; thiếu độ trôi thì dự báo chân trời dài phẳng. ARIMA(0,1,0) là bước ngẫu nhiên. Trong statsmodels: `trend="t"` thêm độ trôi cho mô hình sai phân.

### Cách triển khai

**[D]** NI: ARIMA(0,1,1) không hằng số. DP: bước ngẫu nhiên có trôi `ARIMA(0,1,0, trend="t")` (so với $\hat\delta,\hat\sigma$ tính tay từ $\nabla y$) và ARIMA(1,1,0) có trôi; báo $\hat\mu_W=\hat c/\hat\phi(1)$,
SE, AIC. Hình: dự báo 100 ngày của DP khi có và không có độ trôi.

Với AP, ta khớp ARIMA$(0,1,1)$ có trôi trên 120 tháng log hành khách và kiểm tra phần dư ở độ trễ 12.

### Mã lệnh Python

In [ ]:
ni = ARIMA(NI, order=(0, 1, 1), trend="n").fit()
dp_rw = ARIMA(LD, order=(0, 1, 0), trend="t").fit(); dp_rw0 = ARIMA(LD, order=(0, 1, 0), trend="n").fit(); dp_ar = ARIMA(LD, order=(1, 1, 0), trend="t").fit()
d1 = np.diff(LD)
pl = lambda k: PLAB.get(k, k)
models = {"NI: ARIMA(0,1,1), không trôi": ni, "DP: ARIMA(0,1,0) + trôi": dp_rw, "DP: ARIMA(1,1,0) + trôi": dp_ar}
rows, stat_rows = [], []
for name, f in models.items():
    for k, v, se in zip(f.param_names, f.params, f.bse):
        rows.append({"mô hình": name, "tham số": pl(k), "ước lượng": v, "SE": se, "thống kê z": v / se})
    # ARIMA(trend="t") của statsmodels là hồi quy với sai số ARIMA: khi d = 1, hệ số x1 chính là μ_W = E∇Y (không phải c = φ(1)μ_W)
    mu_w = 0.0 if f is ni else f.params[0]
    stat_rows.append({"mô hình": name, "μ̂_W (hệ số x1)": mu_w, "AIC": f.aic, "BIC": f.bic, "số quan sát": int(f.nobs)})
tab = save_table(pd.DataFrame(rows).set_index(["mô hình", "tham số"]), "tab_b12_05_arima_params")
tab_stats = save_table(pd.DataFrame(stat_rows).set_index("mô hình"), "tab_b12_05_fit_stats")
tab_manual = save_table(pd.DataFrame({"tay (từ ∇y)": [d1.mean(), d1.var(ddof=1)], "ARIMA(0,1,0) + trôi": [dp_rw.params[0], sig2(dp_rw)]}, index=["δ̂ (độ trôi)", "σ̂²"]), "tab_b12_05_rw_manual_vs_arima")
h, n_show = 100, 150; fig, ax = plt.subplots(figsize=(8.5, 3.9)); xs_hist = np.arange(len(LD) - n_show + 1, len(LD) + 1); xs_f = np.arange(len(LD) + 1, len(LD) + h + 1)
ax.plot(xs_hist, LD[-n_show:], color="k", lw=1.0, label="log DAX quan sát")
for f, lab, col in ((dp_rw, "dự báo có độ trôi (trend='t')", BLUE), (dp_rw0, "dự báo không có độ trôi (trend='n')", ORANGE)):
    ax.plot(xs_f, f.get_forecast(h).predicted_mean, color=col, lw=1.8, label=lab)
ax.set(title="DP: dự báo 100 ngày giao dịch tiếp theo", xlabel="ngày giao dịch t", ylabel="log DAX"); ax.legend(loc="upper left")
fig_path = save_figure(fig, "fig_b12_05_drift_forecast")
print("AP: mô hình airline SARIMA(0,1,1)(0,1,1)₁₂ ở mục 12.10; NI không có trôi vì trung bình của ∇y ≈ 0 (đổi mức, không xu thế).")

In [ ]:
# AP • ARIMA không mùa vụ làm đối chứng trên 120 tháng huấn luyện.
ap_drift_fit = ARIMA(TR, order=(0, 1, 1), trend="t").fit()
ap_theta = float(ap_drift_fit.maparams[0])
print("AP: trôi, trung bình sai phân, theta, p Ljung–Box(12)",
      ap_drift_fit.params[0], np.diff(TR).mean(), ap_theta,
      A.ljung_box_manual(ap_drift_fit.resid[1:], 12, model_df=1)[1])

### Kết quả

In [ ]:
display(tab)
display(tab_stats)
display(tab_manual)
show_figure(fig_path)

### Phân tích và diễn giải

*Đồng bộ từ báo cáo, mục 5.2(a).*

Với NI, ARIMA$(0,1,1)$ cho $\hat\theta=-0.733$ với sai số chuẩn $0.074$ và $z=-9.9$, tức hằng số làm trơn mũ $\alpha=1+\hat\theta=0.267$ và AIC $1269.1$; mô hình không có trôi vì trung bình của chuỗi sai phân gần $0$, vì NI đổi mức chứ không có xu thế. Với DP, bước ngẫu nhiên có trôi cho độ trôi $6.47\times10^{-4}$ mỗi ngày với $z=2.66$ và $\hat\sigma^2=1.06\times10^{-4}$, rất gần giá trị tính tay là $6.52\times10^{-4}$ và $1.061\times10^{-4}$. Về lý thuyết, ước lượng hợp lý cực đại của độ trôi trong bước ngẫu nhiên bằng đúng trung bình của chuỗi sai phân; khoảng lệch $0.8%$ ở đây đến từ tối ưu số trên một tham số rất nhỏ, chỉ bằng khoảng $0.02$ sai số chuẩn $2.4\times10^{-4}$, và ARIMA$(1,1,0)$ cho đúng $6.52\times10^{-4}$. Lưu ý quy ước của `statsmodels`: với `trend="t"` và $d=1$, hệ số `x1` là $\mu_W=\mathrm{E}\nabla Y_t$ chứ không phải hằng số $c=\phi(1)\mu_W$ trong 2.2.2(a); hai đại lượng chỉ trùng nhau khi không có phần AR. Thêm hệ số AR không giúp gì: $\hat\phi=-0.0004$ với sai số chuẩn $0.018$, và AIC của bước ngẫu nhiên là $-11733$, thấp hơn $-11731$ của ARIMA$(1,1,0)$. Trên đồ thị, mô hình có trôi cho đường thẳng đi lên, cao hơn khoảng $0.065$ log-đơn vị sau $100$ ngày, còn mô hình bỏ trôi cho đường nằm ngang tại giá trị cuối, đúng như lý thuyết.

Ở AP, độ trôi ước lượng 0.00952 gần trung bình gia số 0.00926. Phần dư còn tự tương quan rất rõ ($p<10^{-18}$), nên mô hình diễn tả được độ trôi nhưng bỏ sót mùa vụ.

### Nhận xét

*Đồng bộ từ báo cáo, mục 5.2(a).*

Kết quả **phù hợp** với lý thuyết: độ trôi ước lượng rất gần tính tay, và việc có hay không có độ trôi làm dự báo dài hạn khác nhau đúng như lý thuyết. Độ trôi khác $0$ ở mức $5%$ với $z=2.66$ nhưng chỉ đáng kể khi dự báo xa.

## 12.6 Ảnh hưởng vĩnh viễn của cú sốc: trọng số Ψ

*Bộ dữ liệu:* NI, AP, DP

### Nội dung lý thuyết

**Định lý 3(a,b).** Với $d=1$, ảnh hưởng của $\varepsilon_t$ lên $Y_{t+k}$ là $\Psi_k=\sum_{j\le k}\psi_j$ (hệ số của $\Psi(z)=\theta(z)/\{\phi(z)(1-z)^d\}$) và
$\Psi_k\to\theta(1)/\phi(1)$: bằng 1 cho bước ngẫu nhiên, $1+\theta$ cho ARIMA(0,1,1); với $d=0$ ảnh hưởng tắt dần ($\Psi_k=\psi_k\to0$).

### Cách triển khai

**[D]** Từ mô hình đã khớp ở 12.5 tính $\Psi_k$ bằng `Psi_weights_arima` (đệ quy tay) và giới hạn $\theta(1)/\phi(1)$: NI (ARIMA(0,1,1)), DP (bước ngẫu nhiên và ARIMA(1,1,0));
đối chứng $d=0$: AR(1) $\phi=0.6$ (minh họa lý thuyết). AP mùa vụ: cú sốc cũng vĩnh viễn nhưng xem 12.10.

Để kiểm tra định lý một nghiệm đơn vị trên AP, ta dùng mô hình ARIMA không mùa vụ ở mục 5.2(a) làm đối chứng.

### Mã lệnh Python

In [ ]:
ni = ARIMA(NI, order=(0, 1, 1), trend="n").fit(); dp_ar = ARIMA(LD, order=(1, 1, 0), trend="t").fit()
th = ni.maparams[0]; ph = dp_ar.arparams[0]
models = {"NI ARIMA(0,1,1)": ([], [th], 1), "DP bước ngẫu nhiên": ([], [], 1), "DP ARIMA(1,1,0)": ([ph], [], 1), "đối chứng AR(1) φ=0.6, d=0": ([0.6], [], 0)}
ks = [0, 1, 2, 5, 10, 50, 100]; rows = {}; fig, ax = plt.subplots(figsize=(7.8, 3.9))
styles = {"NI ARIMA(0,1,1)": (BLUE, "-", 2.0), "DP bước ngẫu nhiên": (ORANGE, "-", 4.5), "DP ARIMA(1,1,0)": (GREEN, "--", 1.8), "đối chứng AR(1) φ=0.6, d=0": (RED, "-", 2.0)}
for name, (phi, theta, d) in models.items():
    Psi = F.Psi_weights_arima(phi, theta, d, 101); lim = (1 + sum(theta)) / (1 - sum(phi)) if d == 1 else 0.0
    rows[name] = {**{f"Ψ_{k}": Psi[k] for k in ks}, "giới hạn θ(1)/φ(1) (d = 1) hoặc 0": lim}; ax.plot(np.arange(40), Psi[:40], color=styles[name][0], ls=styles[name][1], lw=styles[name][2], label=name)
ax.set(xlabel="k (số kỳ sau cú sốc)", ylabel="Ψₖ", title="Ảnh hưởng của một cú sốc đơn vị lên Yₜ₊ₖ"); ax.legend(loc="center right")
fig_path = save_figure(fig, "fig_b12_06_psi_weights")
tab = save_table(pd.DataFrame(rows).T, "tab_b12_06_psi_weights")
print(f"NI: 1 + θ̂ = {1 + th:.4f} (α của làm trơn mũ, 12.7); DP ARIMA(1,1,0): 1/(1−φ̂) = {1 / (1 - ph):.4f}")

In [ ]:
# AP • Kiểm tra hệ số nhân trong mô hình có đúng một nghiệm đơn vị.
ap_weights = F.Psi_weights_arima([], [ap_theta], 1, 101)
print("AP: Psi(1), Psi(100), 1 + theta:", ap_weights[1], ap_weights[100], 1 + ap_theta)

### Kết quả

In [ ]:
show_figure(fig_path)
tab

### Phân tích và diễn giải

*Đồng bộ từ báo cáo, mục 6.1(a).*

Với NI, $\Psi_k=0.2666=1+\hat\theta$ từ $k=1$ đến $k=100$, trùng giới hạn $\theta(1)/\phi(1)$: nếu lưu lượng năm nay cao hơn dự báo $100$ đơn vị thì dự báo mọi năm sau được nâng khoảng $27$ đơn vị và giữ mãi, tức cú sốc không tắt nhưng chỉ được “nhớ” một phần. Với DP, bước ngẫu nhiên có $\Psi_k=1$, tức cú sốc được giữ trọn vẹn, ARIMA$(1,1,0)$ có $\Psi_k=0.9996=1/(1-\hat\phi)$ vì $\hat\phi\approx0$. Mô hình dừng đối chứng cho $\Psi_k=0.6^k$ giảm nhanh về $0$, với $0.078$ ở $k=5$ và $0.006$ ở $k=10$.

Ở mô hình đối chứng của AP, $\hat\theta=0.25849$ và $\Psi_1=\Psi_{100}=1.25849=1+\hat\theta$. Đây là đẳng thức của mô hình một nghiệm đơn vị. Mô hình airline còn có toán tử sai phân mùa vụ nên không dùng giới hạn này để diễn giải nó.

### Nhận xét

*Đồng bộ từ báo cáo, mục 6.1(a).*

Kết quả **phù hợp** với lý thuyết: các trọng số hội tụ đúng về hệ số nhân dài hạn, là $0.267$, $1$ và $0.9996$, ở các mô hình $d=1$ và về $0$ ở mô hình dừng. Mức “nhớ cú sốc” phụ thuộc tham số, chẳng hạn $27%$ ở NI và $100%$ ở DP, chứ không chỉ việc có $d=1$.

## 12.7 ARIMA(0,1,1) ≡ làm trơn mũ đơn ≡ mô hình mức cục bộ

*Bộ dữ liệu:* NI, AP, DP

### Nội dung lý thuyết

**Định lý 3(c).** Nếu $\nabla Y_t=\varepsilon_t+\theta\varepsilon_{t-1}$, $|\theta|<1$, $\alpha=1+\theta$: $\hat Y_{n+h|n}=\alpha Y_n+(1-\alpha)\hat Y_{n|n-1}=\alpha\sum_{j\ge0}(1-\alpha)^jY_{n-j}$ với mọi $h\ge1$.
**Ví dụ 2.2 (mức cục bộ).** $Y_t=\mu_t+\varepsilon_t$, $\mu_t=\mu_{t-1}+\eta_t$: $\nabla Y$ là MA(1); $\sigma_\varepsilon^2=-\theta\sigma^2$, $\sigma_\eta^2=(1+\theta)^2\sigma^2$, $\rho(1)=-1/(q+2)$, $q=\sigma^2_\eta/\sigma^2_\varepsilon$.

### Cách triển khai

**[D]** NI: $\alpha=1+\hat\theta$; đệ quy làm trơn tay (khởi tạo $y_1$) so với dự báo của ARIMA(0,1,1), với `SimpleExpSmoothing` ($\alpha$ cố định) và với $\alpha$ tối ưu; ước lượng
$\sigma^2_\varepsilon,\sigma^2_\eta$ từ $(\hat\theta,\hat\sigma^2)$ và trực tiếp từ `UnobservedComponents('llevel')`. DP chỉ là trường hợp $\theta=0$ ($\alpha=1$); AP có mùa vụ nên không phải mức cục bộ thuần.

Ta khớp SES trên mẫu huấn luyện của log AP và log DP, đối chiếu đệ quy với thư viện và kiểm tra phần dư.

### Mã lệnh Python

In [ ]:
ni = ARIMA(NI, order=(0, 1, 1), trend="n").fit()
th, s2_ = ni.maparams[0], sig2(ni); alpha = 1 + th
level = F.ses_recursion(NI, alpha); fc_arima = ni.get_forecast(1).predicted_mean[0]
ses_fixed = SimpleExpSmoothing(NI, initialization_method="known", initial_level=NI[0]).fit(smoothing_level=alpha, optimized=False)
ses_opt = SimpleExpSmoothing(NI, initialization_method="estimated").fit()
uc = UnobservedComponents(NI, "llevel").fit(disp=False)
eps_v, eta_v = F.local_level_variances(th, s2_)
tab_fc = save_table(pd.DataFrame({"dự báo 1971": [fc_arima, level[-1], ses_fixed.forecast(1)[0], ses_opt.forecast(1)[0]]},
    index=["ARIMA(0,1,1)", f"đệ quy tay, α = 1+θ̂ = {alpha:.4f}", "SimpleExpSmoothing (α cố định, ℓ₁ = y₁)", f"SimpleExpSmoothing (α̂ = {ses_opt.params['smoothing_level']:.4f} tối ưu)"]), "tab_b12_07_ses_forecasts")
ucp = dict(zip(uc.param_names, uc.params))
tab_var = save_table(pd.DataFrame({"σ²_ε (nhiễu quan sát)": [eps_v, ucp["sigma2.irregular"]], "σ²_η (nhiễu mức)": [eta_v, ucp["sigma2.level"]]},
    index=["từ (θ̂, σ̂²) của ARIMA(0,1,1)", "UnobservedComponents('llevel')"]).assign(**{"q = σ²_η/σ²_ε": lambda d: d["σ²_η (nhiễu mức)"] / d["σ²_ε (nhiễu quan sát)"]}), "tab_b12_07_local_level")
q = eta_v / eps_v; print(f"ρ(1) của ∇Y: θ̂/(1+θ̂²) = {th / (1 + th ** 2):.4f}; −1/(q+2) = {-1 / (q + 2):.4f}")
print(f"tổng trọng số α(1−α)^j (j < 100) = {sum(alpha * (1 - alpha) ** j for j in range(100)):.6f}")

In [ ]:
# AP và DP • SES trên chuỗi huấn luyện; kiểm tra cả trường hợp biên.
for ses_name, ses_train in (("AP", TR), ("DP", LD[:DP_TRAIN_N])):
    ses_model = SimpleExpSmoothing(ses_train, initialization_method="known", initial_level=ses_train[0])
    ses_est = ses_model.fit(optimized=True)
    alpha_est = float(ses_est.params["smoothing_level"])
    ses_manual = F.ses_recursion(ses_train, alpha_est)[-1]
    print(ses_name, "alpha, dự báo, sai khác hai cách, p Ljung–Box(12):",
          alpha_est, ses_manual, abs(ses_manual - ses_est.forecast(1)[0]),
          A.ljung_box_manual(ses_est.resid[1:], 12, model_df=1)[1])
print("DP: SES alpha = 1 khớp naive:", F.ses_recursion(LD[:DP_TRAIN_N], 1)[-1] - LD[DP_TRAIN_N-1])

### Kết quả

In [ ]:
display(tab_fc)
tab_var

### Phân tích và diễn giải

*Đồng bộ từ báo cáo, mục 6.1(b).*

Ba cách tính với cùng $\alpha=0.2666$ cho cùng dự báo $798.5$, nên ARIMA$(0,1,1)$ và làm trơn mũ đơn thật sự là một; khi `SimpleExpSmoothing` tự tối ưu, $\hat\alpha=0.2457$ cho $805.3$, khác chút vì tiêu chí và khởi tạo khác nhau. Phương sai mức cục bộ suy từ ARIMA là $15108$ và $1463.5$ với $q=0.0969$, gần ước lượng trực tiếp của `UnobservedComponents` là $15078$ và $1478.8$ với $q=0.0981$. Đẳng thức $\theta/(1+\theta^2)=-1/(q+2)$ đúng đồng nhất khi $q$ được suy từ $\theta$, nên phép kiểm có ý nghĩa là thay $q=0.0981$ ước lượng độc lập bằng `UnobservedComponents`: $-1/(q+2)=-0.4766$, sát $\hat\theta/(1+\hat\theta^2)=-0.4769$ của ARIMA. Ngoài ra tổng trọng số $\alpha(1-\alpha)^j$ bằng $1$.

Với AP và DP, hai cách tính dự báo một bước khớp tới độ chính xác máy. AP có $\hat\alpha\approx1$ nhưng phần dư còn mùa vụ ($p<10^{-22}$). DP có $\hat\alpha=0.9987$ và $p=0.172$; đặt $\alpha=1$ cho dự báo bằng đúng log giá cuối.

### Nhận xét

*Đồng bộ từ báo cáo, mục 6.1(b).*

Kết quả **phù hợp** với lý thuyết: dự báo và các đẳng thức khớp ở độ chính xác số học khi dùng cùng $\alpha$; sai khác nhỏ giữa hai ước lượng phương sai, $0.2%$ và $1%$, đến từ thuật toán ước lượng, không phải từ lý thuyết.

## 12.8 Dự báo h bước tối ưu MSE, phương sai sai số và khoảng dự báo

*Bộ dữ liệu:* NI, AP, DP

### Nội dung lý thuyết

**Định lý 4.** $\hat Y_{n+h|n}=E(Y_{n+h}|\mathcal F_n)$; $e_n(h)=\sum_{j<h}\Psi_j\varepsilon_{n+h-j}$ (MA(h−1)), $\operatorname{Var}e_n(h)=\sigma^2\sum_{j<h}\Psi_j^2$.
Trường hợp riêng: $h\sigma^2$ (ARIMA(0,1,0)); $\sigma^2\{1+(h-1)\alpha^2\}$ (ARIMA(0,1,1), $\alpha=1+\theta$); $\sigma^2(1-\phi^{2h})/(1-\phi^2)\to\gamma(0)$ (AR(1) nhân quả).
Khoảng $(1-a)$: $\hat Y_{n+h|n}\pm z_{1-a/2}\,\sigma(\sum_{j<h}\Psi_j^2)^{1/2}$ chính xác khi tham số đã biết; ước lượng tham số, chọn bậc làm khoảng chỉ xấp xỉ. $d=0$: độ rộng hội tụ $2z\sqrt{\gamma(0)}$; $d=1$ tăng như $\sqrt h$.

### Cách triển khai

**[D]** Phương sai sai số dự báo tính tay (Ψ + công thức đóng) so với `get_forecast().var_pred_mean`: NI ARIMA(0,1,1) ($h\le10$), DP bước ngẫu nhiên có trôi ($h\le250$), AP airline trên thang log ($h\le24$,
Ψ từ tích đa thức SARIMA). Đối chứng $d=0$: AR(1) $\phi=0.8$. **[M]** độ phủ khoảng 95% tại $h=1,5,10$: ARIMA(0,1,1) $\theta=-0.5,\sigma=1,n=100$ (thiết kế bài giảng, $B=2000$) và với $\hat\theta,\hat\sigma$ của NI ($n=100$, $B=1000$);
mỗi lần khớp lại mô hình đúng.

### Mã lệnh Python

In [ ]:
ni = ARIMA(NI, order=(0, 1, 1), trend="n").fit(); th, s2_ = ni.maparams[0], sig2(ni); alpha = 1 + th
H = 10; Psi = F.Psi_weights_arima([], [th], 1, H); hh = np.arange(1, H + 1)
var_ni = F.forecast_variance_manual(Psi, s2_, H); var_ni_closed = s2_ * (1 + (hh - 1) * alpha ** 2); var_ni_sm = ni.get_forecast(H).var_pred_mean
dp = ARIMA(LD, order=(0, 1, 0), trend="t").fit(); Hd = 250; hd = np.arange(1, Hd + 1)
var_dp = F.forecast_variance_manual(F.Psi_weights_arima([], [], 1, Hd), sig2(dp), Hd); var_dp_sm = dp.get_forecast(Hd).var_pred_mean
ap = airline(TR); pa = dict(zip(ap.param_names, ap.params))
ar_p, ma_p = F.sarima_polys(theta=[pa["ma.L1"]], Theta=[pa["ma.S.L12"]], d=1, D=1, s=12)
var_ap = F.forecast_variance_manual(F.psi_from_polys(ar_p, ma_p, 24), pa["sigma2"], 24); var_ap_sm = ap.get_forecast(24).var_pred_mean
rel = lambda a, b: float(np.max(np.abs(a / b - 1)))  # noqa: E731
tab = save_table(pd.DataFrame([
    {"bộ": "NI ARIMA(0,1,1)", "h tối đa": H, "SE tay (h=1)": np.sqrt(var_ni[0]), "SE tay (h=tối đa)": np.sqrt(var_ni[-1]), "SE statsmodels (h=tối đa)": np.sqrt(var_ni_sm[-1]),
     "max |tay/statsmodels − 1|": rel(var_ni, var_ni_sm), "max |công thức đóng/Ψ − 1|": rel(var_ni_closed, var_ni)},
    {"bộ": "DP bước ngẫu nhiên + trôi", "h tối đa": Hd, "SE tay (h=1)": np.sqrt(var_dp[0]), "SE tay (h=tối đa)": np.sqrt(var_dp[-1]), "SE statsmodels (h=tối đa)": np.sqrt(var_dp_sm[-1]),
     "max |tay/statsmodels − 1|": rel(var_dp, var_dp_sm), "max |công thức đóng/Ψ − 1|": rel(sig2(dp) * hd, var_dp)},
    {"bộ": "AP airline (log)", "h tối đa": 24, "SE tay (h=1)": np.sqrt(var_ap[0]), "SE tay (h=tối đa)": np.sqrt(var_ap[-1]), "SE statsmodels (h=tối đa)": np.sqrt(var_ap_sm[-1]),
     "max |tay/statsmodels − 1|": rel(var_ap, var_ap_sm), "max |công thức đóng/Ψ − 1|": np.nan}]).set_index("bộ"), "tab_b12_08_forecast_variance")
g = np.arange(1, 25); fig, ax = plt.subplots(figsize=(7.8, 3.9))
ax.plot(g, np.sqrt(g), color=RED, lw=1.8, label="bước ngẫu nhiên ARIMA(0,1,0): √h"); ax.plot(g, np.sqrt(1 + (g - 1) * 0.5 ** 2), color=BLUE, lw=1.8, label="ARIMA(0,1,1), α = 0.5")
ax.plot(g, np.sqrt((1 - 0.8 ** (2 * g)) / (1 - 0.8 ** 2)), color=GREEN, lw=1.8, label="AR(1), φ = 0.8 (d = 0)"); ax.axhline(np.sqrt(1 / (1 - 0.64)), color=GREY, ls=":", label="√γ(0) của AR(1)")
ax.set(xlabel="chân trời h", ylabel="SE của eₙ(h) với σ = 1", title="Độ rộng khoảng dự báo: d = 0 hội tụ, d = 1 tăng không giới hạn"); ax.legend(loc="upper left")
fig_path = save_figure(fig, "fig_b12_08_se_growth")
fc_ni = ni.get_forecast(20); ci_ni = fc_ni.conf_int(); yrs = DATA["NI"].year.to_numpy(); fyr = np.arange(yrs[-1] + 1, yrs[-1] + 21)
fig, ax = plt.subplots(figsize=(8.5, 3.9))
ax.plot(yrs, NI, color="k", lw=1.0, label="lưu lượng quan sát"); ax.plot(fyr, fc_ni.predicted_mean, color=BLUE, lw=2, label="dự báo ARIMA(0,1,1)")
ax.fill_between(fyr, ci_ni[:, 0], ci_ni[:, 1], color=BLUE, alpha=0.18, label="khoảng dự báo 95%")
ax.set(xlabel="năm", ylabel="lưu lượng (10⁸ m³)", title="NI: dự báo 20 năm (phẳng vì d = 1, không trôi; khoảng mở rộng vì Ψⱼ không tắt dần)"); ax.legend(loc="lower left")
fig_path_ni = save_figure(fig, "fig_b12_08_ni_forecast")

In [ ]:
def coverage_sim(theta, sigma, n, B, rng, hs=(1, 5, 10)):
    hits = dict.fromkeys(hs, 0)
    for _ in range(B):
        y = np.cumsum(S.simulate_arma([], [theta], n + 10, rng, sigma=sigma))
        ci = ARIMA(y[:n], order=(0, 1, 1), trend="n").fit().get_forecast(10).conf_int(alpha=0.05)
        for h in hs: hits[h] += ci[h - 1, 0] <= y[n + h - 1] <= ci[h - 1, 1]
    return {f"h = {h}": hits[h] / B for h in hs}, np.sqrt(0.95 * 0.05 / B)
cov1, mc1 = coverage_sim(-0.5, 1.0, 100, 2000, S.child_rng(12, 8))
cov2, mc2 = coverage_sim(th, np.sqrt(s2_), 100, 1000, S.child_rng(12, 81))
tab_cov = save_table(pd.DataFrame([{"thiết kế": "θ = −0.5, σ = 1, n = 100, B = 2000", **cov1, "MCSE ≈": mc1},
                                   {"thiết kế": f"θ = {th:.3f}, σ = {np.sqrt(s2_):.1f} (NI), n = 100, B = 1000".replace("-", "−"), **cov2, "MCSE ≈": mc2}]).set_index("thiết kế"), "tab_b12_08_coverage")

### Kết quả

In [ ]:
display(tab)
show_figure(fig_path)
show_figure(fig_path_ni)
tab_cov

### Phân tích và diễn giải

*Đồng bộ từ báo cáo, mục 6.2(a).*

Sai khác tương đối lớn nhất của phương sai dự báo giữa cách tính tay và `statsmodels` là $1.0084\times10^{-5}$ ở AP. Với NI, công thức đóng $\sigma^2\{1+(h-1)\alpha^2\}$ trùng tổng theo $\Psi_j$. Vì $d=1$, khoảng dự báo loe dần: ở NI sai số chuẩn tăng từ $143.5$ ở $h=1$ lên $183.8$ ở $h=10$, khoảng dự báo rộng ra từ $[517,1080]$ lên $[438,1159]$ trong khi đường dự báo nằm ngang ở $799$; ở AP, trên thang log, sai số chuẩn tăng từ $0.037$ lên $0.155$ khi $h=24$; ở DP nó bằng $\hat\sigma\sqrt h$, từ $0.0103$ lên $0.163$ ở $h=250$. Mô phỏng cho độ phủ $93.8%$, $94.4%$, $94.7%$ ở $h=1,5,10$ khi $\theta=-0.5$, và $93.4%$, $94.3%$, $94.4%$ với tham số của NI: đều thấp hơn $95%$ một chút, nhiều nhất ở $h=1$ với độ lệch $2.6$ và $2.3$ lần sai số Monte Carlo, rồi tiến dần về $95%$.

### Nhận xét

*Đồng bộ từ báo cáo, mục 6.2(a).*

Kết quả **phù hợp** với lý thuyết: phương sai sai số dự báo tính tay khớp thư viện, với sai khác tương đối lớn nhất xấp xỉ $10^{-5}$, và độ rộng khoảng tăng đúng theo bậc $d$. Độ phủ thấp hơn $95%$ một chút, trong dung sai $3$ MCSE, đúng với cảnh báo của lý thuyết: khi tham số phải ước lượng, khoảng tính bằng công thức hơi hẹp.

## 12.9 Dự báo trên thang log: trung vị, trung bình, khoảng chuyển đổi

*Bộ dữ liệu:* NI, AP, DP

### Nội dung lý thuyết

**Mệnh đề 2.** Nếu phân phối dự báo của $\log Y_{n+h}$ là $N(m,s^2)$: (a) khoảng $(e^L,e^U)$ là khoảng $(1-a)$ chính xác cho $Y_{n+h}$; (b) $e^m$ là **trung vị**, còn $E(Y_{n+h}|\mathcal F_n)=e^{m+s^2/2}>e^m$ là **trung bình**;
(c) báo cáo trung vị nếu đánh giá bằng sai số tuyệt đối, trung bình nếu bằng sai số bình phương hoặc khi cần cộng gộp. Hệ số $e^{s^2/2}$: $s=0.04$ → 0.08%, $s=0.3$ → 4.6%.

### Cách triển khai

**[D]** AP: airline (huấn luyện 120 tháng) dự báo 24 tháng 1959–1960: trung vị, trung bình, khoảng 95%, so với giá trị thực (đếm số điểm nằm trong khoảng). DP: bước ngẫu nhiên có trôi với $\hat\delta,\hat\sigma$ từ 1859
suất sinh lợi log, dự báo giá DAX sau 1, 20, 250 ngày giao dịch từ giá cuối 5473.72: trung vị, trung bình, khoảng bất đối xứng.

Ta khớp thêm ARIMA$(0,1,1)$ trên log lưu lượng của 80 năm đầu ở NI để kiểm tra phép đổi thang.

### Mã lệnh Python

In [ ]:
f = airline(TR); g = f.get_forecast(24); m, s = g.predicted_mean, np.sqrt(g.var_pred_mean); bt = F.back_transform_log(m, s); act = AP[120:]
sel = [0, 11, 23]
tab_ap = pd.DataFrame({"tháng": [DATA["AP"].month.iloc[120 + i] for i in sel], "s (thang log)": s[sel], "trung vị = exp(m)": bt["median"][sel], "trung bình = exp(m + s²/2)": bt["mean"][sel],
                       "cận dưới 95%": bt["lower"][sel], "cận trên 95%": bt["upper"][sel], "thực tế": act[sel], "exp(s²/2) − 1 (%)": 100 * (np.exp(s[sel] ** 2 / 2) - 1)}).set_index("tháng")
inside = int(np.sum((act >= bt["lower"]) & (act <= bt["upper"])))
tab_ap = save_table(tab_ap, "tab_b12_09_ap_backtransform")
yrs_all = 1949 + np.arange(144) / 12; fig, ax = plt.subplots(figsize=(8.5, 3.9))
ax.plot(yrs_all[:120], AP[:120], color="k", lw=1.0, label="huấn luyện (1949–1958)"); ax.plot(yrs_all[120:], act, color=GREEN, lw=1.5, marker="o", ms=3, label="thực tế 1959–1960")
ax.plot(yrs_all[120:], bt["median"], color=BLUE, lw=2, label="dự báo airline (trung vị)"); ax.fill_between(yrs_all[120:], bt["lower"], bt["upper"], color=BLUE, alpha=0.18, label="khoảng dự báo 95%")
ax.set(xlabel="năm", ylabel="số hành khách (nghìn người)", title="AP: dự báo mô hình airline 24 tháng trên thang gốc"); ax.legend(loc="upper left")
fig_path = save_figure(fig, "fig_b12_09_ap_forecast")
r = np.diff(LD); delta, sigma = r.mean(), r.std(ddof=1); hs = np.array([1, 20, 250])
bt_dp = F.back_transform_log(LD[-1] + delta * hs, sigma * np.sqrt(hs))
tab_dp = pd.DataFrame({"s = σ̂√h": sigma * np.sqrt(hs), "trung vị": bt_dp["median"], "trung bình": bt_dp["mean"], "cận dưới 95%": bt_dp["lower"], "cận trên 95%": bt_dp["upper"]}, index=pd.Index(hs, name="h (ngày)"))
tab_dp["trên − trung vị"] = tab_dp["cận trên 95%"] - tab_dp["trung vị"]; tab_dp["trung vị − dưới"] = tab_dp["trung vị"] - tab_dp["cận dưới 95%"]
tab_dp = save_table(tab_dp, "tab_b12_09_dp_forecast")
print(f"AP: {inside}/24 giá trị thực nằm trong khoảng 95%; DP: δ̂ = {delta:.6f}, σ̂ = {sigma:.6f}, giá cuối = {DAX[-1]:.2f}")
print("NI: không biến đổi log nên mục này không áp dụng (mức nước/lưu lượng dự báo trực tiếp trên thang gốc).")

In [ ]:
# NI • Kiểm tra phép đổi ngược trên một mô hình log phụ, khớp 80 năm đầu.
nile_log_fit = ARIMA(np.log(NI[:80]), order=(0, 1, 1), trend="n").fit()
nile_fc = nile_log_fit.get_forecast(1)
nile_m, nile_s = float(nile_fc.predicted_mean[0]), float(nile_fc.se_mean[0])
print("NI h=1: trung vị, trung bình, khoảng 95%, tỷ số",
      np.exp(nile_m), np.exp(nile_m + nile_s**2/2), np.exp(nile_fc.conf_int()[0]), np.exp(nile_s**2/2))

### Kết quả

In [ ]:
show_figure(fig_path)
display(tab_ap)
tab_dp

### Phân tích và diễn giải

*Đồng bộ từ báo cáo, mục 6.2(b).*

Với AP, trung bình lớn hơn trung vị đúng hệ số $e^{s^2/2}$: chênh $0.07%$ ở tháng đầu với $s=0.037$, $0.40%$ ở 12/1959 và $1.21%$ ở 12/1960 với $s=0.155$, tăng theo chân trời. Cả $24$ giá trị thực nằm trong khoảng $95%$, nhưng trung vị thấp hơn giá trị thực ở hầu hết các tháng của năm 1960, chẳng hạn $388$ so với $432$ ở 12/1960, nên mô hình hơi đánh giá thấp mức tăng. Với DP, trung vị và trung bình gần nhau ở $h=1$ với $5477.3$ và $5477.6$ nhưng khác $1.3%$ ở $h=250$ với $6442.8$ và $6528.9$; khoảng trên thang giá bất đối xứng, với nửa trên $2423$ và nửa dưới $1761$ ở $h=250$, đúng với hệ quả của phép đổi đơn điệu từ khoảng đối xứng trên thang log.

Ở mô hình log phụ cho NI, dự báo một năm trên thang lưu lượng có trung vị 858.7, trung bình 870.7 và khoảng 95% từ 619.1 đến 1191.0. Tỷ số 1.0140 khớp hệ số $e^{s^2/2}$. Mô hình chính của NI ở các bảng đánh giá vẫn dùng lưu lượng trên thang gốc.

### Nhận xét

*Đồng bộ từ báo cáo, mục 6.2(b).*

Kết quả **phù hợp** với lý thuyết: chênh lệch trung bình–trung vị khớp công thức $e^{s^2/2}$, khoảng chuyển đúng về thang gốc và bất đối xứng như lý thuyết nói. Cần nêu rõ đang báo cáo trung vị hay trung bình, vì ở chân trời dài như $h=250$ của DP chênh lệch đạt $1.3%$.

## 12.10 Sai phân mùa vụ, SARIMA và mô hình airline

*Bộ dữ liệu:* NI, AP, DP

### Nội dung lý thuyết

**Định nghĩa 4, Định lý 5.** $\nabla_s=1-B^s=(1-B)(1+B+\dots+B^{s-1})$ loại bỏ mẫu mùa vụ cố định và xu thế tuyến tính. SARIMA$(p,d,q)(P,D,Q)_s$: $\Phi(B^s)\phi(B)\nabla^d\nabla_s^DY_t=c+\Theta(B^s)\theta(B)\varepsilon_t$.
Mô hình airline $\nabla\nabla_sY_t=(1+\theta B)(1+\Theta B^s)\varepsilon_t$ là MA($s+1$): $\rho(1)=\frac{\theta}{1+\theta^2}$, $\rho(s)=\frac{\Theta}{1+\Theta^2}$, $\rho(s\pm1)=\rho(1)\rho(s)$, mọi độ trễ khác bằng 0;
khả nghịch ⇔ $|\theta|,|\Theta|<1$. Dự báo dài hạn $a+bh+m_h$ ($m_{h+s}=m_h$, $\sum m_j=0$). $\nabla\nabla_{12}$ mất 13 quan sát.

### Cách triển khai

**[D]** AP huấn luyện 120 tháng: độ lệch chuẩn của $\mathrm{tr},\nabla\mathrm{tr},\nabla_{12}\mathrm{tr},\nabla\nabla_{12}\mathrm{tr}$ (log); ACF của $\nabla\nabla_{12}$ log (dải $\pm1.96/\sqrt{131}$) tại độ trễ 1, 11, 12, 13 so với
giá trị lý thuyết tính từ $(\hat\theta,\hat\Theta)$ (công thức đóng và ACF của đa thức MA(13)). Khớp airline và dự báo 24 tháng. Đối chiếu ba cách mô hình hóa mùa vụ (Ví dụ 4.2): airline, 11 biến giả mùa vụ và $K$ cặp hạng Fourier
(sai số ARIMA(0,1,1)) theo AIC và RMSE trên tập kiểm định 1959–1960.

Với NI ta thử chu kỳ 12 năm; với DP ta thử chu kỳ 5 phiên trên mẫu huấn luyện. Ta so ACF ở một và hai chu kỳ và tỷ số phương sai sau khi sai phân mùa vụ thêm một lần.

### Mã lệnh Python

In [ ]:
d1, d12 = np.diff(TR), TR[12:] - TR[:-12]; dd = np.diff(d12)
tab_sd = save_table(pd.DataFrame({"n": [len(TR), len(d1), len(d12), len(dd)], "sd": [TR.std(ddof=1), d1.std(ddof=1), d12.std(ddof=1), dd.std(ddof=1)]},
                                 index=["tr = log y", "∇tr", "∇₁₂tr", "∇∇₁₂tr"]), "tab_b12_10_sd")
fit = airline(TR); par = dict(zip(fit.param_names, fit.params)); th_, Th_ = par["ma.L1"], par["ma.S.L12"]
r_hat = A.sample_acf(dd, 25)
ma_coef = np.polymul([1, th_], np.r_[1.0, np.zeros(11), Th_])[1:]       # (1 + θz)(1 + Θz¹²) → θ_1..θ_13
r_ma = R.theoretical_acf([], ma_coef, 25, terms=60)
r1, r12 = th_ / (1 + th_ ** 2), Th_ / (1 + Th_ ** 2)
closed = {1: r1, 11: r1 * r12, 12: r12, 13: r1 * r12}
tab_acf = save_table(pd.DataFrame({"ρ̂(h) của ∇∇₁₂": [r_hat[h] for h in closed], "ρ(h) công thức đóng": list(closed.values()),
                                   "ρ(h) từ MA(13)": [r_ma[h] for h in closed]}, index=pd.Index(list(closed), name="h")), "tab_b12_10_airline_acf")
others = [h for h in range(1, 26) if h not in closed]
print(f"θ̂ = {th_:.4f}, Θ̂ = {Th_:.4f}, σ̂² = {par['sigma2']:.5f}, AIC = {fit.aic:.1f}; dải ±1.96/√{len(dd)} = {1.96 / np.sqrt(len(dd)):.3f}; "
      f"max |ρ(h)| lý thuyết ở các độ trễ còn lại (h ≤ 25) = {np.abs(r_ma[others]).max():.2e}")
fig, ax = plt.subplots(figsize=(9.5, 4.0)); acf_bars(ax, r_hat, len(dd), "AP: ACF của ∇∇₁₂ log y và ACF lý thuyết của mô hình airline")
overlay_theory(ax, range(1, 26), r_ma[1:], "ACF lý thuyết của airline (θ̂, Θ̂)")
fig_path = save_figure(fig, "fig_b12_10_airline_acf")

In [ ]:
t_all = np.arange(144); month = np.eye(12)[t_all % 12][:, 1:]
fourier = lambda K: np.column_stack([fn(2 * np.pi * k * t_all / 12) for k in range(1, K + 1) for fn in (np.sin, np.cos)])
# n dòng = số quan sát còn lại sau sai phân (như Ví dụ 4.2): 120 − 13 với airline, 120 − 1 với các mô hình còn lại
rows = [{"mô hình": "airline", "k": len(fit.params), "n dòng": 120 - 13, "AIC": fit.aic, "RMSE giữ lại": E.rmse(AP[120:], np.exp(fit.get_forecast(24).predicted_mean))}]
# Ví dụ 4.2 khớp các mô hình biến giả / Fourier không có độ trôi; thêm bản có độ trôi (SARIMAX trend="c" với d = 1 là hằng số
# trong phương trình của ∇y, tức độ trôi) để tách tác động của xu thế khỏi cách mô hình hóa mùa vụ
for name, X in (("11 biến giả + ∇ + MA(1)", month), ("Fourier K = 2", fourier(2)), ("Fourier K = 3", fourier(3)), ("Fourier K = 4", fourier(4))):
  for tr_, suffix in (("n", ""), ("c", " + trôi")):
    try:
        mdl = SARIMAX(TR, exog=X[:120], order=(0, 1, 1), trend=tr_).fit(disp=0)
        fc = np.exp(mdl.get_forecast(24, exog=X[120:]).predicted_mean)
        rows.append({"mô hình": name + suffix, "k": len(mdl.params), "n dòng": 120 - 1, "AIC": mdl.aic, "RMSE giữ lại": E.rmse(AP[120:], fc)})
        if tr_ == "c": print(name, "độ trôi ước lượng:", round(float(mdl.params[0]), 5), "(trung bình ∇ log AP huấn luyện:", round(float(np.diff(TR).mean()), 5), ")")
    except Exception as exc:  # báo lỗi rõ thay vì bỏ qua im lặng
        rows.append({"mô hình": name, "k": np.nan, "n dòng": np.nan, "AIC": np.nan, "RMSE giữ lại": np.nan}); print("lỗi:", name, exc)
tab_models = save_table(pd.DataFrame(rows).set_index("mô hình"), "tab_b12_10_seasonality_models")
# Sàng lọc chu kỳ 12 quan sát ở NI và DP: ACF của chuỗi sai phân ở độ trễ 12 và 24 so với dải ±1.96/√n
rows_s = []
for c, w in (("AP (log, huấn luyện)", np.diff(TR)), ("NI", np.diff(NI)), ("DP (log)", np.diff(LD))):
    r = A.sample_acf(w, 24)
    rows_s.append({"chuỗi sai phân ∇y": c, "n": len(w), "ρ̂(12)": r[12], "ρ̂(24)": r[24], "dải ±": A.bartlett_band(len(w))})
tab_season = save_table(pd.DataFrame(rows_s).set_index("chuỗi sai phân ∇y"), "tab_b12_10_seasonal_lags")

In [ ]:
# Đối chứng mùa vụ: chu kỳ ứng viên theo đơn vị quan sát.
# NI ghi theo năm: thử chu kỳ 12 năm; DP: 5 phiên, chỉ là xấp xỉ tuần.
for cycle_code, cycle_y, cycle_s in (("NI", NI, 12), ("DP", LD[:DP_TRAIN_N], 5)):
    cycle_diff = np.diff(cycle_y)
    cycle_acf = A.sample_acf(cycle_diff, 2*cycle_s)
    cycle_extra = cycle_diff[cycle_s:] - cycle_diff[:-cycle_s]
    print(cycle_code, "s, ACF(s,2s), dải 95%, tỷ số phương sai sau/trước:",
          cycle_s, cycle_acf[[cycle_s, 2*cycle_s]], A.bartlett_band(len(cycle_diff)),
          cycle_extra.var(ddof=1)/cycle_diff.var(ddof=1))

### Kết quả

In [ ]:
display(tab_sd)
display(tab_acf)
show_figure(fig_path)
tab_models

### Phân tích và diễn giải

*Đồng bộ từ báo cáo, mục 5.2(b).*

Độ lệch chuẩn giảm đều qua từng phép sai phân, từ $0.392$ ở log AP xuống $0.106$ sau $\nabla$, $0.066$ sau $\nabla_{12}$ và $0.046$ sau $\nabla\nabla_{12}$, không có dấu hiệu sai phân thừa. ACF mẫu khớp ACF lý thuyết ở các độ trễ đặc trưng: $-0.328$ so với $-0.307$ ở độ trễ 1, $-0.406$ so với $-0.418$ ở độ trễ 12; ở độ trễ 11 và 13 là $0.076$ và $0.182$ so với $0.128$, trong phạm vi sai số mẫu khoảng $0.1$. Công thức đóng và khai triển MA(13) cho cùng giá trị. Ở các độ trễ còn lại lý thuyết cho $0$ và phần lớn ACF mẫu nằm trong dải $\pm0.189$; chỉ độ trễ 3 với $-0.22$ và độ trễ 23 với $0.23$ vượt dải, hợp lý khi xét $25$ độ trễ. Về dự báo, khi khớp đúng như Ví dụ 4.2, tức không có độ trôi, airline cho RMSE $43.2$ trên phần kiểm định, tốt hơn rõ so với $85.5$ của 11 biến giả và $74.8$ đến $82.0$ của Fourier. Nhưng các mô hình này chỉ lấy một lần sai phân và không có độ trôi, nên dự báo dài hạn nằm ngang, trong khi airline với $\nabla\nabla_{12}$ tự mang xu thế tuyến tính. Khi thêm độ trôi, được ước lượng khoảng $0.0093$ đến $0.0099$ mỗi tháng, sát trung bình $0.0093$ của $\nabla\log$ AP, RMSE giảm còn $33.1$ với 11 biến giả và từ $21.9$ đến $27.6$ với Fourier, tức tốt hơn airline. Như vậy khoảng cách ở Ví dụ 4.2 chủ yếu đến từ việc có hay không có xu thế chứ không phải từ cách mô hình hóa mùa vụ; trên 24 tháng kiểm định, mô hình mùa vụ cố định có độ trôi còn dự báo tốt hơn airline. AIC không so sánh trực tiếp được giữa các mô hình có cách sai phân khác nhau, nên AIC thấp hơn của mô hình biến giả, $-424.3$ hay $-434.0$ so với $-389.0$, không cho biết mô hình nào tốt hơn.

Ở NI, ACF tại 12 và 24 năm là $-0.044,-0.040$ (dải $\pm0.197$). Ở DP, ACF tại 5 và 10 phiên là $-0.011,-0.014$ (dải $\pm0.051$). Phép sai phân theo chu kỳ thử làm phương sai tăng 2.03 và 2.02 lần. Hai chuỗi chỉ xác nhận bước chẩn đoán; dữ liệu chưa ủng hộ việc dùng sai phân mùa vụ ở các chu kỳ này.

### Nhận xét

*Đồng bộ từ báo cáo, mục 5.2(b).*

Riêng AP **phù hợp** với lý thuyết: cấu trúc bốn độ trễ của ACF được xác nhận cả về giá trị lẫn vị trí, với một mô hình chỉ có hai tham số; hai độ trễ 3 và 23 vượt dải là ngoại lệ nhỏ. Về dự báo, airline vượt các mô hình mùa vụ không có độ trôi như trong Ví dụ 4.2, nhưng không vượt được khi các mô hình đó có độ trôi. Trên 24 tháng kiểm định, thứ hạng dự báo vì vậy phụ thuộc vào cách mô hình hóa xu thế.

NI và DP chỉ ủng hộ bước chẩn đoán; sai phân theo các chu kỳ ứng viên làm tăng phương sai.

## 12.11 Mốc chuẩn và thước đo độ chính xác

*Bộ dữ liệu:* NI, AP, DP

### Nội dung lý thuyết

**Định nghĩa 5.** Mốc chuẩn: trung bình $\bar y$; naive $y_n$; naive mùa vụ $y_{n+h-s(k+1)}$ ($k=\lfloor(h-1)/s\rfloor$); trôi $y_n+h(y_n-y_1)/(n-1)$. MAE, RMSE, MAPE $=\frac{100}m\sum|e_i/y_{n+i}|$,
MASE $=$ MAE$/Q$, $Q=\frac1{n-s}\sum_{t>s}|y_t-y_{t-s}|$ (MAE của naive mùa vụ một bước trong mẫu huấn luyện). MASE bất biến thang đo; MAPE bất đối xứng (Định lý 6(b,c)). Một con số không kèm chân trời, mốc chuẩn, thang đo thì không phải kết quả.

### Cách triển khai

**[D]** AP: tập kiểm định 24 tháng (1959–60), thang gốc: airline, naive mùa vụ, trôi, naive, trung bình. DP: huấn luyện 1500 ngày, kiểm định 360 ngày: (i) một bước theo log (%, RMSE/MAE) cho naive, trôi, ARIMA(1,1,0)
(tham số cố định từ mẫu huấn luyện, trạng thái cập nhật), (ii) dự báo đa bước từ gốc cố định 1500 trên thang giá. NI (bài giảng không chia sẵn tập kiểm định): **lựa chọn của nhóm** — huấn luyện 80 năm đầu (1871–1950), kiểm định 20 năm cuối, dự báo từ gốc cố định.

### Mã lệnh Python

In [ ]:
acc = lambda a, f, tr, s=1: {"RMSE": E.rmse(a, f), "MAE": E.mae(a, f), "MAPE (%)": E.mape(a, f), "MASE": E.mase(a, f, tr, s)}
# AP
tr_ap, te_ap = AP[:120], AP[120:]
air = airline(np.log(tr_ap)); g_air = air.get_forecast(24); f_air = np.exp(g_air.predicted_mean)
f_air_mean = np.exp(g_air.predicted_mean + g_air.var_pred_mean / 2)        # trung bình exp(m + s²/2), tối ưu cho RMSE
fc_ap = {"airline": f_air, "airline (trung bình)": f_air_mean, "naive mùa vụ": F.forecast_seasonal_naive(tr_ap, 24, 12), "trôi": F.forecast_drift(tr_ap, 24),
         "naive": F.forecast_naive(tr_ap, 24), "trung bình": F.forecast_mean(tr_ap, 24)}
tab_ap = save_table(pd.DataFrame({k: acc(te_ap, f, tr_ap, 12) for k, f in fc_ap.items()}).T, "tab_b12_11_ap_benchmarks")
# NI
tr_ni, te_ni = NI[:80], NI[80:]
ni_fit = ARIMA(tr_ni, order=(0, 1, 1), trend="n").fit()
fc_ni = {"ARIMA(0,1,1)": ni_fit.forecast(20), "naive": F.forecast_naive(tr_ni, 20), "trôi": F.forecast_drift(tr_ni, 20), "trung bình": F.forecast_mean(tr_ni, 20)}
tab_ni = save_table(pd.DataFrame({k: acc(te_ni, f, tr_ni, 1) for k, f in fc_ni.items()}).T, "tab_b12_11_ni_benchmarks")

In [ ]:
# DP: (i) một bước theo log, đơn vị % ; (ii) đa bước từ gốc cố định, thang giá
tr_ld, te_ld = LD[:DP_TRAIN_N], LD[DP_TRAIN_N:]; prev = LD[DP_TRAIN_N - 1:-1]; delta = (tr_ld[-1] - tr_ld[0]) / (DP_TRAIN_N - 1)
ar110 = ARIMA(tr_ld, order=(1, 1, 0), trend="n").fit()
one = {"naive": prev, "trôi": prev + delta, "ARIMA(1,1,0)": ar110.extend(te_ld).fittedvalues}
tab_dp1 = save_table(pd.DataFrame({k: {"RMSE (%)": 100 * E.rmse(te_ld, f), "MAE (%)": 100 * E.mae(te_ld, f)} for k, f in one.items()}).T, "tab_b12_11_dp_one_step")
H = len(te_ld); price_tr, price_te = DAX[:DP_TRAIN_N], DAX[DP_TRAIN_N:]
multi = {"naive": F.forecast_naive(price_tr, H), "trôi": F.forecast_drift(price_tr, H), "trung bình": F.forecast_mean(price_tr, H),
         "ARIMA(1,1,0) (trung vị)": np.exp(ar110.get_forecast(H).predicted_mean)}
tab_dp360 = save_table(pd.DataFrame({k: acc(price_te, f, price_tr, 1) for k, f in multi.items()}).T, "tab_b12_11_dp_multistep")

### Kết quả

In [ ]:
display(tab_ap)
display(tab_ni)
display(tab_dp1)
tab_dp360

### Phân tích và diễn giải

*Đồng bộ từ báo cáo, mục 6.3(a).*

**NI.** Ba phương pháp gần như ngang nhau, với RMSE $122.3$ cho trôi, $122.8$ cho ARIMA và $123.1$ cho naive, còn trung bình kém hơn với $133.3$. Thứ hạng đổi theo thước đo: theo RMSE và MAPE thì trôi tốt nhất, còn theo MAE thì naive tốt nhất với $102.0$ so với $104.3$ của ARIMA. Mọi MASE đều dưới $1$, từ $0.76$ đến $0.81$, nhưng chênh lệch giữa các phương pháp nhỏ.

**AP.** Airline vượt xa mọi mốc chuẩn ở cả bốn thước đo: RMSE $43.2$ so với $77.0$ của naive mùa vụ, tức tốt hơn $1.8$ lần, $115.7$ của trôi, $137.3$ của naive và $219.4$ của trung bình; MAPE $8.5%$ so với $15.5%$ của mốc tốt nhất. Dòng “airline” là trung vị $e^{m}$, đúng như Bài 12; dự báo trung bình $e^{m+s^2/2}$ cho RMSE $40.6$ và MAE $37.1$, tốt hơn trung vị ở cả hai thước đo, vì mô hình đánh giá thấp mức tăng năm 1960 (6.2(b)) nên dự báo cao hơn một chút có lợi cho cả hai. MASE $1.38>1$ nghĩa là sai số trung bình trên $24$ tháng lớn hơn sai số một bước của naive mùa vụ trong mẫu huấn luyện, điều này không mâu thuẫn vì dự báo xa khó hơn dự báo một bước.

**DP.** Ở dự báo một bước, ba phương pháp cho RMSE từ $1.435%$ đến $1.439%$, hầu như không phân biệt được, vì dự báo ARIMA$(1,1,0)$ gần với naive khi $\hat\phi\approx0$ và không có trôi. Ở dự báo đa bước từ gốc cố định, trôi tốt nhất với RMSE $1093.8$ và MAPE $17.6%$, hơn naive với $1332.5$ và $22.0%$, và hơn nhiều so với trung bình với $2546.4$. Các MASE rất lớn, từ $65$ đến $181$, vì mẫu số là sai số một bước còn dự báo đi tới $360$ ngày, nên MASE không dùng để so với $1$ ở đây.

### Nhận xét

*Đồng bộ từ báo cáo, mục 6.3(a).*

Kết quả **phù hợp** với lý thuyết: với AP mô hình vượt xa mốc chuẩn nên đáng dùng; với NI, RMSE của ARIMA, naive và trôi gần nhau khi dự báo từ một gốc cố định. Trên 360 ngày kiểm định của DP, RMSE một bước của ba phương pháp cũng gần nhau. Các thước đo xếp hạng khác nhau ở NI, và MASE cần thận trọng khi chân trời dự báo lệch xa mẫu số, đúng như lý thuyết cảnh báo.

## 12.12 Thẩm định trên gốc cuốn chiếu và vai trò của bước chọn mô hình

*Bộ dữ liệu:* NI, AP, DP

### Nội dung lý thuyết

**Định nghĩa 5(c).** Với các gốc $t=t_0,\dots,n-h$ khớp lại mô hình trên dữ liệu đến $t$, tính $e_t(h)=y_{t+h}-\hat y_{t+h|t}$ rồi lấy trung bình mất mát theo $t$ cho từng $h$. Không chia ngẫu nhiên; không chọn mô hình trên chính dữ liệu dùng báo cáo độ chính xác:
giữ một khối cuối chưa chạm tới hoặc **lồng bước chọn mô hình** trong từng gốc cuốn chiếu. Độ chính xác phụ thuộc chân trời $h$.

### Cách triển khai

**[D]** NI: gốc $t_0=80$, $h\le5$: naive, trung bình mở rộng, ARIMA(0,1,1) cố định, và ARIMA chọn trong $\{(0,1,0),(0,1,1),(1,1,0),(1,1,1)\}$ theo AIC **bên trong mỗi gốc** (bước chọn nằm trong thiết kế).
AP: gốc $t_0=108$ (36 gốc ở $h=1$), $h\le12$, airline khớp lại mỗi tháng so với naive mùa vụ; tái lập thiết kế bài giảng ($t=120..143$, $h=1$). DP: 360 ngày kiểm định, dự báo 1 bước:
ARIMA(1,1,0) với tham số cố định (khớp 1500 ngày) so với khớp lại mỗi 60 ngày; trôi cập nhật cùng lịch.

### Mã lệnh Python

In [ ]:
ff_naive = lambda tr, h: np.full(h, tr[-1]); ff_mean = lambda tr, h: np.full(h, tr.mean())
ff_011 = lambda tr, h: ARIMA(tr, order=(0, 1, 1), trend="n").fit().forecast(h)
cand = [(0, 1, 0), (0, 1, 1), (1, 1, 0), (1, 1, 1)]
ff_sel = lambda tr, h: min((ARIMA(tr, order=o, trend="n").fit() for o in cand), key=lambda f: f.aic).forecast(h)
H = 5; ro_ni = {k: E.rolling_origin_horizons(NI, ff, 80, H) for k, ff in (("naive", ff_naive), ("trung bình mở rộng", ff_mean), ("ARIMA(0,1,1) cố định", ff_011), ("ARIMA chọn theo AIC trong mỗi gốc", ff_sel))}
tab_ni = save_table(pd.DataFrame({k: {f"RMSE h={h}": E.rmse(*v[h]) for h in (1, 2, 3, 5)} | {"số gốc h=1": len(v[1][0])} for k, v in ro_ni.items()}).T, "tab_b12_12_ni_rolling")

In [ ]:
ff_air = lambda tr, h: np.exp(airline(np.log(tr)).get_forecast(h).predicted_mean)
ff_sn = lambda tr, h: F.forecast_seasonal_naive(tr, h, 12)
ro_ap = {"airline": E.rolling_origin_horizons(AP, ff_air, 108, 12), "naive mùa vụ": E.rolling_origin_horizons(AP, ff_sn, 108, 12)}
hs = list(range(1, 13)); curve = pd.DataFrame({k: [E.rmse(*v[h]) for h in hs] for k, v in ro_ap.items()}, index=pd.Index(hs, name="h"))
curve["số gốc"] = [len(ro_ap["airline"][h][0]) for h in hs]; tab_ap = save_table(curve, "tab_b12_12_ap_rolling_by_horizon")
a1, f1 = E.rolling_origin_forecast(AP, ff_air, 120, 1); s1 = AP[120:144] - AP[108:132]
tab_ap1 = save_table(pd.DataFrame({"RMSE 1 bước, gốc t = 120..143": [E.rmse(a1, f1), float(np.sqrt(np.mean(s1 ** 2)))]}, index=["airline", "naive mùa vụ"]), "tab_b12_12_ap_one_step")
fig, ax = plt.subplots(figsize=(7.8, 3.9))
ax.plot(curve.index, curve["airline"], "o-", color=BLUE, label="airline (khớp lại mỗi tháng)"); ax.plot(curve.index, curve["naive mùa vụ"], "s-", color=ORANGE, label="naive mùa vụ")
ax.set(xlabel="chân trời h (tháng)", ylabel="RMSE (nghìn người)", title="AP: RMSE gốc cuốn chiếu theo chân trời", ylim=(0, None)); ax.xaxis.set_major_locator(plt.MaxNLocator(integer=True)); ax.legend(loc="center right")
fig_path = save_figure(fig, "fig_b12_12_ap_rmse_by_horizon")

In [ ]:
n0, B = DP_TRAIN_N, 60
sched = {"ARIMA(1,1,0) tham số cố định": None, f"ARIMA(1,1,0) khớp lại mỗi {B} ngày": B}
fc = {k: np.empty(len(LD) - n0) for k in list(sched) + ["trôi cố định (δ̂ của 1500 ngày đầu)", f"trôi cập nhật mỗi {B} ngày", "naive"]}
for s0 in range(n0, len(LD), B):
    e0 = min(s0 + B, len(LD)); sl = slice(s0 - n0, e0 - n0)
    for k, every in sched.items():
        tr = LD[:s0] if every else LD[:n0]
        fc[k][sl] = ARIMA(tr, order=(1, 1, 0), trend="n").fit().extend(LD[s0:e0]).fittedvalues if every or s0 == n0 else fc[k][sl]
    fc["trôi cố định (δ̂ của 1500 ngày đầu)"][sl] = LD[s0 - 1:e0 - 1] + (LD[n0 - 1] - LD[0]) / (n0 - 1)
    fc[f"trôi cập nhật mỗi {B} ngày"][sl] = LD[s0 - 1:e0 - 1] + (LD[s0 - 1] - LD[0]) / (s0 - 1)
    fc["naive"][sl] = LD[s0 - 1:e0 - 1]
fc["ARIMA(1,1,0) tham số cố định"] = ARIMA(LD[:n0], order=(1, 1, 0), trend="n").fit().extend(LD[n0:]).fittedvalues
tab_dp = save_table(pd.DataFrame({k: {"RMSE (%)": 100 * E.rmse(LD[n0:], f), "MAE (%)": 100 * E.mae(LD[n0:], f)} for k, f in fc.items()}).T, "tab_b12_12_dp_rolling")

### Kết quả

In [ ]:
display(tab_ni)
display(tab_ap)
display(tab_ap1)
show_figure(fig_path)
tab_dp

### Phân tích và diễn giải

*Đồng bộ từ báo cáo, mục 6.3(b).*

Kết quả của NI khác với một điểm chia ở mục trước. Ở $h=1$, ARIMA$(0,1,1)$ với $126.9$ và ARIMA chọn theo AIC với $126.2$ tốt nhất, hơn trung bình mở rộng với $131.9$ và naive với $153.1$; nhưng từ $h=2$ trở đi trung bình mở rộng tốt hơn ARIMA, với $128.8$ so với $133.5$ ở $h=2$ và $127.5$ so với $140.6$ ở $h=5$, có lẽ vì sau lần đổi mức năm 1899 chuỗi dao động quanh một mức cố định nên trung bình ổn định hơn bước ngẫu nhiên. Bước chọn mô hình theo AIC ở mỗi gốc cải thiện không đáng kể, $126.2$ so với $126.9$. Mỗi chân trời chỉ dựa trên tối đa $20$ gốc ($20$, $19$, $18$ và $16$ gốc ở $h=1,2,3,5$) nên kết quả không chắc chắn. Với AP, airline có RMSE một bước $15.3$ khi khớp lại mỗi gốc từ $t=120$ đến $143$, so với $50.0$ của naive mùa vụ; theo chân trời, RMSE của airline chỉ tăng nhẹ từ $15.3$ ở $h=1$ lên $20.9$ ở $h=12$, còn của naive mùa vụ từ $42.0$ lên $49.0$, nên airline thắng ở mọi chân trời, dù số gốc giảm từ $36$ xuống $25$. Với DP, RMSE một bước là $1.439%$ khi tham số cố định, $1.440%$ khi khớp lại mỗi $60$ ngày và $1.435%$ với trôi cố định hoặc cập nhật, nên mọi phương án nằm trong $0.4%$ của nhau vì mô hình đã là bước ngẫu nhiên.

### Nhận xét

*Đồng bộ từ báo cáo, mục 6.3(b).*

Kết quả **phù hợp** với lý thuyết: gốc cuốn chiếu cho bức tranh đầy đủ hơn một lần chia: ở NI, kết luận “ARIMA tốt hơn trung bình” của gốc cố định đảo ngược từ chân trời 2 trở đi; ở AP, lợi thế của airline vững ở mọi chân trời. Hạn chế là số gốc ít, chỉ $20$ ở NI, và giảm dần khi chân trời tăng.

## 12.13 Hàm mất mát xác định phiếm hàm; kiểm định Diebold–Mariano

*Bộ dữ liệu:* NI, AP, DP

### Nội dung lý thuyết

**Định lý 6.** (a) $\arg\min_cE(Y-c)^2=EY$ và $\arg\min_cE|Y-c|$ là trung vị: RMSE nhắm tới trung bình, MAE nhắm tới trung vị; với dữ liệu lệch hai thước đo có thể xếp hạng khác nhau.
(d) Diebold–Mariano: $d_t=L(e_{1t})-L(e_{2t})$, $DM=\bar d/\sqrt{\hat v/m}\to N(0,1)$, $\hat v\approx\sum_{|k|<h}\hat\gamma_d(k)$ (Bài 11, Định lý 1); sai số $h$ bước là MA($h-1$) nên $\gamma_d(k)=0$ với $k\ge h$.
Kiểm định có thể hỏng với hai mô hình lồng nhau khớp trên cùng dữ liệu (Bài tập 10(4)).

### Cách triển khai

**[D]** AP: DM giữa airline và naive mùa vụ trên sai số một bước của gốc cuốn chiếu $t=120..143$ (hiệu bình phương, $t_{m-1}$ như bài giảng; thêm hiệu tuyệt đối); thứ hạng sáu dự báo ở tập giữ lại 24 tháng theo RMSE và theo MAE;
hàm mất mát và phiếm hàm: cực tiểu của tổng bình phương và tổng tuyệt đối trên lưới $c$ cho số hành khách (lệch phải) so với trung bình và trung vị. NI: DM giữa ARIMA(0,1,1) và naive trên gốc cuốn chiếu $t=80..99$.

Ở DP, ta so dự báo có trôi với naive trên cùng 360 gốc một bước. Độ trôi cố định từ 1.500 phiên đầu; chênh lệch mất mát được tính theo thứ tự có trôi trừ naive.

### Mã lệnh Python

In [ ]:
ff_air = lambda tr, h: np.exp(airline(np.log(tr)).get_forecast(h).predicted_mean)
a1, f1 = E.rolling_origin_forecast(AP, ff_air, 120, 1); e_air = a1 - f1; e_sn = AP[120:144] - AP[108:132]
dm_sq, p_sq = E.diebold_mariano(e_air ** 2 - e_sn ** 2); dm_ab, p_ab = E.diebold_mariano(np.abs(e_air) - np.abs(e_sn))
ff_011 = lambda tr, h: ARIMA(tr, order=(0, 1, 1), trend="n").fit().forecast(h)
a_n, f_n = E.rolling_origin_forecast(NI, ff_011, 80, 1); e_a, e_nv = a_n - f_n, NI[80:] - NI[79:-1]
dm_ni_sq, p_ni_sq = E.diebold_mariano(e_a ** 2 - e_nv ** 2); dm_ni_ab, p_ni_ab = E.diebold_mariano(np.abs(e_a) - np.abs(e_nv))
tab_dm = save_table(pd.DataFrame([
    {"so sánh": "AP: airline − naive mùa vụ", "hàm mất mát": "bình phương", "số gốc m": len(e_air), "DM": dm_sq, "p (t_{m−1})": p_sq},
    {"so sánh": "AP: airline − naive mùa vụ", "hàm mất mát": "tuyệt đối", "số gốc m": len(e_air), "DM": dm_ab, "p (t_{m−1})": p_ab},
    {"so sánh": "NI: ARIMA(0,1,1) − naive", "hàm mất mát": "bình phương", "số gốc m": len(e_a), "DM": dm_ni_sq, "p (t_{m−1})": p_ni_sq},
    {"so sánh": "NI: ARIMA(0,1,1) − naive", "hàm mất mát": "tuyệt đối", "số gốc m": len(e_a), "DM": dm_ni_ab, "p (t_{m−1})": p_ni_ab}]).set_index(["so sánh", "hàm mất mát"]), "tab_b12_13_diebold_mariano")

In [ ]:
tr_ap, te_ap = AP[:120], AP[120:]; g24 = airline(np.log(tr_ap)).get_forecast(24); f_air24 = np.exp(g24.predicted_mean)
fcs = {"airline": f_air24, "airline (trung bình)": np.exp(g24.predicted_mean + g24.var_pred_mean / 2), "naive mùa vụ": F.forecast_seasonal_naive(tr_ap, 24, 12), "trôi": F.forecast_drift(tr_ap, 24), "naive": F.forecast_naive(tr_ap, 24), "trung bình": F.forecast_mean(tr_ap, 24)}
rk = pd.DataFrame({"RMSE": {k: E.rmse(te_ap, f) for k, f in fcs.items()}, "MAE": {k: E.mae(te_ap, f) for k, f in fcs.items()}})
rk["hạng theo RMSE"] = rk["RMSE"].rank().astype(int); rk["hạng theo MAE"] = rk["MAE"].rank().astype(int); tab_rank = save_table(rk, "tab_b12_13_ranking")
grid = np.linspace(tr_ap.min(), tr_ap.max(), 4001)
mse = [np.mean((tr_ap - c) ** 2) for c in grid]; mae_ = [np.mean(np.abs(tr_ap - c)) for c in grid]
skew = stats.skew(tr_ap)
tab_fun = save_table(pd.DataFrame({"cực tiểu của hàm mất mát": [grid[np.argmin(mse)], grid[np.argmin(mae_)]], "đại lượng lý thuyết": [tr_ap.mean(), np.median(tr_ap)]},
                                  index=["tổng bình phương → trung bình", "tổng tuyệt đối → trung vị"]).assign(**{"độ lệch (skewness)": skew}), "tab_b12_13_loss_functional")

In [ ]:
# DP • Cùng 360 dự báo một bước, độ trôi cố định từ 1.500 ngày huấn luyện.
# Dấu âm nghĩa là mô hình có trôi có mất mát trung bình nhỏ hơn naive.
dp_previous = LD[DP_TRAIN_N-1:-1]
dp_actual = LD[DP_TRAIN_N:]
dp_delta = np.diff(LD[:DP_TRAIN_N]).mean()
dp_naive_error = 100 * (dp_actual - dp_previous)
dp_drift_error = 100 * (dp_actual - (dp_previous + dp_delta))
for loss_name, loss in (("bình phương", np.square), ("tuyệt đối", np.abs)):
    dp_loss_diff = loss(dp_drift_error) - loss(dp_naive_error)
    print(loss_name, "n, trung bình chênh lệch, DM, p:", len(dp_loss_diff),
          dp_loss_diff.mean(), *E.diebold_mariano(dp_loss_diff, h=1))

### Kết quả

In [ ]:
display(tab_dm)
display(tab_rank)
tab_fun

### Phân tích và diễn giải

*Đồng bộ từ báo cáo, mục 6.3(c).*

Với AP, điểm cực tiểu trên lưới của tổng bình phương là $246.0$, gần trung bình mẫu $245.9$. Với 120 quan sát, mọi giá trị trong đoạn $[233,234]$ đều cực tiểu hóa tổng sai lệch tuyệt đối, gồm cả điểm trên lưới $233.1$ và trung vị mẫu $233.5$. Chuỗi lệch phải với độ lệch $0.58$ nên hai đại lượng khác nhau, đúng như lý thuyết. Tuy vậy với sáu dự báo, thứ hạng theo RMSE và MAE giống hệt nhau, từ tốt đến kém là airline dạng trung bình, airline dạng trung vị, naive mùa vụ, trôi, naive và trung bình: chênh lệch giữa các phương pháp lớn hơn nhiều tác động của lựa chọn thước đo. Trung vị của phân phối dự báo cực tiểu hóa sai lệch tuyệt đối kỳ vọng theo phân phối ấy. Trên 24 tháng quan sát, dự báo trung bình của airline lại có MAE thấp hơn vì điều chỉnh lên giúp giảm mức dự báo thiếu trong năm 1960. Kiểm định Diebold–Mariano bác bỏ rất mạnh việc airline và naive mùa vụ ngang nhau với $DM=-7.49$ cho hàm bình phương và $-8.785$ cho hàm tuyệt đối, $p<0.0001$. Trên $20$ gốc cuốn chiếu của NI, RMSE một bước là $126.9$ với ARIMA$(0,1,1)$ và $153.1$ với naive, giảm khoảng $17%$ (bảng `tab_b12_12_ni_rolling`). Kiểm định DM cho $p=0.101$ theo mất mát bình phương và $p=0.182$ theo mất mát tuyệt đối. Ở mức $5%$, bằng chứng chưa đủ để kết luận ARIMA dự báo tốt hơn. Cần đọc kết quả này thận trọng: naive là ARIMA$(0,1,1)$ với $\theta=0$, nên hai mô hình lồng nhau, trường hợp mà 2.3.3(c) đã cảnh báo; khi đó phân phối của $DM$ dưới giả thuyết không không còn xấp xỉ chuẩn, và các hiệu chỉnh như của Clark và West nên được dùng nếu cần kết luận chặt.

Trên DP, thống kê DM cho mất mát bình phương là $-1.415$ ($p=0.158$), cho mất mát tuyệt đối là $-0.653$ ($p=0.514$). Cả hai chênh lệch nghiêng nhẹ về mô hình có trôi, nhưng chưa đủ bằng chứng về lợi thế dự báo ở mức 5%.

### Nhận xét

*Đồng bộ từ báo cáo, mục 6.3(c).*

Kết quả **phù hợp** với lý thuyết: tính chất phiếm hàm được xác nhận bằng số, và kiểm định Diebold–Mariano cho thấy lợi thế dự báo một bước của airline so với naive mùa vụ ở AP. Với NI, kết quả trên 20 gốc còn bất định. Trong dữ liệu này thứ hạng theo RMSE và MAE không đổi; lý thuyết chỉ nói hai thước đo *có thể* xếp hạng khác nhau, không bảo đảm điều đó xảy ra.

## 12.14 Checklist báo cáo dự báo ARIMA

*Bộ dữ liệu:* NI, AP, DP

### Nội dung lý thuyết

**Checklist (trang 66).** (1) Vẽ đồ thị, tách khối kiểm định → (2) biến đổi (log, Box–Cox) → (3) chọn $d,D$ (đồ thị, ACF, kiểm định) → (4) xác định $(p,q),(P,Q)$, khớp ML → (5) phần dư trắng? vượt mốc chuẩn?
Báo cáo: biến đổi và $d,D$ kèm bằng chứng và kiểm tra sai phân quá mức; mô hình khớp với SE, $\hat\sigma^2$, nhân quả và khả nghịch không triệt tiêu gần đúng; ACF phần dư, $Q_{LB}(H)$ với $H-p-q-P-Q$ bậc tự do,
ACF của bình phương phần dư; dự báo kèm khoảng và nêu rõ trung bình hay trung vị; độ chính xác so với mốc chuẩn tại chân trời có ý nghĩa, trên gốc cuốn chiếu.

### Cách triển khai

**[D]** Mỗi bộ một cột trong bảng checklist: NI ARIMA(0,1,1) khớp toàn bộ 100 năm (như bài giảng), AP airline (huấn luyện 120 tháng), DP bước ngẫu nhiên có trôi (huấn luyện 1500 ngày). Các số về độ chính xác lấy từ cùng thiết kế với 12.11–12.12
(NI: 20 năm cuối; AP: 24 tháng giữ lại; DP: 1 bước trên 360 ngày). Hình: ACF phần dư và ACF phần dư bình phương.

### Mã lệnh Python

In [ ]:
def unit_root(y):
    a0, a1 = adfuller(y, regression="c")[1], adfuller(np.diff(y), regression="c")[1]
    fp = lambda q: "<0.0001" if q < 1e-4 else f"{q:.3g}"
    k0, k1 = kpss(y, regression="c", nlags="auto")[0], kpss(np.diff(y), regression="c", nlags="auto")[0]
    return f"ADF p-value: {fp(a0)} (y), {fp(a1)} (∇y); KPSS η: {k0:.2f} (y), {k1:.2f} (∇y), tới hạn 5% = 0.463"
models = {"NI": ARIMA(NI, order=(0, 1, 1), trend="n").fit(), "AP": airline(TR), "DP": ARIMA(LD[:DP_TRAIN_N], order=(0, 1, 0), trend="t").fit()}
spec = {"NI": dict(H=12, df=1, k=1, y=NI), "AP": dict(H=24, df=2, k=13, y=TR), "DP": dict(H=10, df=0, k=1, y=LD[:DP_TRAIN_N])}
ap_sd = lambda v: pd.Series(v).groupby(np.arange(len(v)) // 12).std(ddof=1)
out = {c: {} for c in models}; fig, axes = plt.subplots(3, 2, figsize=(11.5, 8.8))
for i, (c, f) in enumerate(models.items()):
    sp = spec[c]; e = F.zero_diffuse_residuals(f.resid, sp["k"]); roots_ar, roots_ma = np.abs(f.arroots), np.abs(f.maroots)
    q, p = lb(e, sp["H"], sp["df"]); q2, p2 = lb(e ** 2, sp["H"], 0)
    out[c]["1. Biến đổi"] = {"NI": "không", "AP": f"log (sd trong năm của mức (số hành khách): {ap_sd(AP).min():.1f}–{ap_sd(AP).max():.1f}; của log: {ap_sd(LAP).min():.3f}–{ap_sd(LAP).max():.3f})",
                             "DP": "log giá"}[c]
    out[c]["2. d, D và bằng chứng"] = {"NI": "d = 1, D = 0; ", "AP": "d = 1, D = 1; ", "DP": "d = 1, D = 0; "}[c] + unit_root(sp["y"] if c != "AP" else TR[12:] - TR[:-12]) + (" (AP: kiểm định trên ∇₁₂)" if c == "AP" else "")
    w = np.diff(sp["y"], 2) if c != "AP" else np.diff(TR[12:] - TR[:-12])
    out[c]["3. Kiểm tra sai phân quá mức"] = f"ρ̂(1) của chuỗi đã sai phân (tối đa) = {A.sample_acf(np.diff(sp['y']) if c != 'AP' else w, 1)[1]:.3f}; ρ̂(1) sau khi sai phân thêm một lần = {A.sample_acf(np.diff(np.diff(sp['y']) if c != 'AP' else w), 1)[1]:.3f}"
    out[c]["4. Mô hình khớp (ML)"] = "; ".join(f"{PLAB.get(n, n)} = {sci(v, 4)} (SE {sci(se, 2)})" for n, v, se in zip(f.param_names, f.params, f.bse))
    out[c]["5. Nhân quả / khả nghịch"] = f"{('min|nghiệm AR| = ' + format(roots_ar.min(), '.3g')) if len(roots_ar) else 'không có phần AR'}; {('min|nghiệm MA| = ' + format(roots_ma.min(), '.3g')) if len(roots_ma) else 'không có phần MA'}"
    e_cut = np.asarray(f.resid, dtype=float)[sp["k"]:]                     # kiểm tra độ nhạy: bỏ hẳn k phần dư khởi tạo thay vì đặt bằng 0
    qc, pc = lb(e_cut, sp["H"], sp["df"]); qc2, pc2 = lb(e_cut ** 2, sp["H"], 0)
    out[c]["6. Phần dư"] = (f"Q_LB({sp['H']}) = {q:.2f}, df = {sp['H'] - sp['df']}, p = {p:.3f}; phần dư²: Q = {q2:.2f}, p = {sci(p2)}"
                          f" (bỏ {sp['k']} phần dư đầu: p = {pc:.3f}; phần dư²: p = {sci(pc2)})")
    acf_bars(axes[i, 0], A.sample_acf(e, 24), len(e), f"{c}: ACF của phần dư"); acf_bars(axes[i, 1], A.sample_acf(e ** 2, 24), len(e), f"{c}: ACF của phần dư²"); panel_label(axes[i, 0], "(" + "abc"[i] + ")")
fig.tight_layout(); fig_path = save_figure(fig, "fig_b12_14_residual_acf")
# dự báo và so với mốc chuẩn
fc_ni = models["NI"].get_forecast(10); ci = fc_ni.conf_int(); out["NI"]["7. Dự báo + khoảng 95% (thang gốc)"] = f"phân phối dự báo chuẩn trên thang gốc, trung bình = trung vị; h=1: {fc_ni.predicted_mean[0]:.0f} [{ci[0, 0]:.0f}, {ci[0, 1]:.0f}]; h=10: {fc_ni.predicted_mean[9]:.0f} [{ci[9, 0]:.0f}, {ci[9, 1]:.0f}]"
g = models["AP"].get_forecast(24); bt = F.back_transform_log(g.predicted_mean, np.sqrt(g.var_pred_mean))
out["AP"]["7. Dự báo + khoảng 95% (thang gốc)"] = f"gốc 12/1958; trung vị/trung bình (1/1959): {bt['median'][0]:.1f}/{bt['mean'][0]:.1f} [{bt['lower'][0]:.0f}, {bt['upper'][0]:.0f}]; (12/1960): {bt['median'][23]:.1f}/{bt['mean'][23]:.1f} [{bt['lower'][23]:.0f}, {bt['upper'][23]:.0f}]"
gd = models["DP"].get_forecast(20); sd = np.sqrt(gd.var_pred_mean[[0, 19]]); b2 = F.back_transform_log(gd.predicted_mean[[0, 19]], sd)
out["DP"]["7. Dự báo + khoảng 95% (thang gốc)"] = f"gốc t = {DP_TRAIN_N} (cuối tập huấn luyện); h=1: trung vị {b2['median'][0]:.0f}, trung bình {b2['mean'][0]:.0f} [{b2['lower'][0]:.0f}, {b2['upper'][0]:.0f}]; h=20: trung vị {b2['median'][1]:.0f}, trung bình {b2['mean'][1]:.0f} [{b2['lower'][1]:.0f}, {b2['upper'][1]:.0f}]"
ni_tr = ARIMA(NI[:80], order=(0, 1, 1), trend="n").fit().forecast(20); sn = F.forecast_seasonal_naive(AP[:120], 24, 12)
af = np.exp(airline(np.log(AP[:120])).get_forecast(24).predicted_mean)
out["NI"]["8. So với mốc chuẩn"] = f"RMSE 20 năm cuối: ARIMA {E.rmse(NI[80:], ni_tr):.1f} vs naive {E.rmse(NI[80:], F.forecast_naive(NI[:80], 20)):.1f}"
out["AP"]["8. So với mốc chuẩn"] = f"RMSE 24 tháng: airline {E.rmse(AP[120:], af):.1f} vs naive mùa vụ {E.rmse(AP[120:], sn):.1f}; MASE {E.mase(AP[120:], af, AP[:120], 12):.2f}"
tl, te = LD[:DP_TRAIN_N], LD[DP_TRAIN_N:]
out["DP"]["8. So với mốc chuẩn"] = f"RMSE 1 bước (%): trôi {100 * E.rmse(te, LD[DP_TRAIN_N - 1:-1] + (tl[-1] - tl[0]) / (DP_TRAIN_N - 1)):.3f} vs naive {100 * E.rmse(te, LD[DP_TRAIN_N - 1:-1]):.3f}"
tab = save_table(pd.DataFrame(out), "tab_b12_14_checklist")

### Kết quả

In [ ]:
show_figure(fig_path)
pd.set_option('display.max_colwidth', None)
tab

### Phân tích và diễn giải

*Đồng bộ từ báo cáo, mục 6.3(d).*

**NI.** Không biến đổi, $d=1$ và $D=0$: ở chuỗi sai phân ADF bác bỏ nghiệm đơn vị và KPSS không bác bỏ, còn ở chuỗi gốc ADF cũng bác bỏ trong chuỗi có đổi mức, như ở mục 5.3(b). $\hat\rho(1)$ từ $-0.40$ xuống $-0.63$ khi sai phân thêm một lần nên $d=1$ là đủ. $\hat\theta=-0.733$ khả nghịch với môđun nghiệm MA là $1.36$, và kiểm định Ljung–Box trên phần dư cho $Q_{LB}(12)=13.6$, $p=0.26$, còn phần dư bình phương có $p=0.84$. Dự báo là $799$ với khoảng $[517,1080]$ ở $h=1$ và $[438,1159]$ ở $h=10$, và RMSE là $122.8$ trên $20$ năm cuối, ngang naive với $123.1$.

**AP.** Biến đổi log hợp lý vì độ lệch chuẩn trong từng năm của số hành khách dao động từ $13.7$ đến $77.7$, khác nhau hơn $5$ lần, còn của log chỉ từ $0.108$ đến $0.160$; $d=1$ và $D=1$. Airline có $\hat\theta=-0.343$ và $\hat\Theta=-0.541$, khả nghịch rõ ràng: $|\hat\Theta|$ cách biên $1$ khoảng $4.6$ sai số chuẩn. Môđun nghiệm MA mùa vụ bằng $|\hat\Theta|^{-1/12}=1.05$ trông gần $1$ chỉ vì căn bậc $12$, không phải dấu hiệu sát biên. Kiểm định trên phần dư cho $Q_{LB}(24)=19.8$, $p=0.60$, còn phần dư bình phương ở ranh giới với $p=0.066$. Kết quả này tính như Bài 12, tức đặt $13$ phần dư khởi tạo bằng $0$; nếu bỏ hẳn $13$ phần dư đó thì $p=0.71$ cho phần dư và $p=0.11$ cho phần dư bình phương, nên dấu hiệu ở ranh giới phần nào đến từ cách xử lý phần dư khởi tạo. Ở NI và DP, cách xử lý này hầu như không đổi kết quả. RMSE là $43.2$ so với $77.0$ của naive mùa vụ.

**DP.** Lấy log, $d=1$ và $D=0$; sai phân thêm một lần đưa $\hat\rho(1)$ từ $-0.004$ xuống $-0.490$, tức sai phân thừa, nên $d=1$ là đủ. Kiểm định trên phần dư cho $p=0.40$ nhưng phần dư bình phương còn tự tương quan rất mạnh với $p=5.6\times10^{-8}$, nên khoảng dự báo dựa trên phương sai không đổi chưa đáng tin và cần mô hình cho phương sai như GARCH. RMSE một bước của trôi là $1.435%$, chỉ ngang naive với $1.439%$.

### Nhận xét

*Đồng bộ từ báo cáo, mục 6.3(d).*

Kết quả **phù hợp** với lý thuyết: cả tám phần thực hiện được cho cả ba bộ dữ liệu và cho một bức tranh nhất quán, đồng thời chỉ ra đúng những điểm cần thận trọng: phần dư bình phương của AP ở ranh giới, tùy cách xử lý phần dư khởi tạo, và cụm biến động còn lại của DP. Ở NI và DP, RMSE của các phương pháp gần nhau trong các phép so ở hàng cuối bảng. Ở AP, airline có RMSE thấp hơn rõ rệt so với naive mùa vụ.

## ⋆ Mở rộng (tùy chọn): hồi quy giả tạo và đồng tích hợp log DAX ~ log SMI

*Bộ dữ liệu:* DAX và SMI (phần mở rộng, không thuộc 6 bộ chính)

### Nội dung lý thuyết

Hai chuỗi I(1) $X_t,Y_t$ đồng tích hợp nếu $Y_t-\beta X_t$ là I(0) (Engle–Granger, 1987). $R^2$ cao không phải là đồng tích hợp: hồi quy log DAX theo log SMI (1991–1998) cho $\hat\beta=0.820$, $R^2=0.965$ nhưng
kiểm định Engle–Granger trên phần dư có $p=0.28$ (bài giảng, trang ⋆). *Phần mở rộng:* SMI chỉ dùng ở đây và phải được ghi chú là mở rộng.

### Cách triển khai

OLS $\log\mathrm{DAX}\sim\log\mathrm{SMI}$ rồi kiểm định Engle–Granger (`statsmodels.tsa.stattools.coint`). Chỉ đọc cột SMI từ file gốc `data/raw/EuStockMarkets.csv`.

### Mã lệnh Python

In [ ]:
from as3179ts.data import load_raw
raw = load_raw("EuStockMarkets"); x, y = np.log(raw["SMI"].to_numpy()), np.log(raw["DAX"].to_numpy())
ols = sm.OLS(y, sm.add_constant(x)).fit(); eg = coint(y, x)
tab_ext = save_table(pd.DataFrame({"giá trị": [ols.params[1], ols.rsquared, eg[0], eg[1]]}, index=["β̂", "R²", "thống kê Engle–Granger", "p-value Engle–Granger"]), "tab_b12_ext_cointegration")

### Kết quả

In [ ]:
tab_ext

### Phân tích và diễn giải

*Đồng bộ từ báo cáo, mục 6.3(d).*

**NI.** Không biến đổi, $d=1$ và $D=0$: ở chuỗi sai phân ADF bác bỏ nghiệm đơn vị và KPSS không bác bỏ, còn ở chuỗi gốc ADF cũng bác bỏ trong chuỗi có đổi mức, như ở mục 5.3(b). $\hat\rho(1)$ từ $-0.40$ xuống $-0.63$ khi sai phân thêm một lần nên $d=1$ là đủ. $\hat\theta=-0.733$ khả nghịch với môđun nghiệm MA là $1.36$, và kiểm định Ljung–Box trên phần dư cho $Q_{LB}(12)=13.6$, $p=0.26$, còn phần dư bình phương có $p=0.84$. Dự báo là $799$ với khoảng $[517,1080]$ ở $h=1$ và $[438,1159]$ ở $h=10$, và RMSE là $122.8$ trên $20$ năm cuối, ngang naive với $123.1$.

**AP.** Biến đổi log hợp lý vì độ lệch chuẩn trong từng năm của số hành khách dao động từ $13.7$ đến $77.7$, khác nhau hơn $5$ lần, còn của log chỉ từ $0.108$ đến $0.160$; $d=1$ và $D=1$. Airline có $\hat\theta=-0.343$ và $\hat\Theta=-0.541$, khả nghịch rõ ràng: $|\hat\Theta|$ cách biên $1$ khoảng $4.6$ sai số chuẩn. Môđun nghiệm MA mùa vụ bằng $|\hat\Theta|^{-1/12}=1.05$ trông gần $1$ chỉ vì căn bậc $12$, không phải dấu hiệu sát biên. Kiểm định trên phần dư cho $Q_{LB}(24)=19.8$, $p=0.60$, còn phần dư bình phương ở ranh giới với $p=0.066$. Kết quả này tính như Bài 12, tức đặt $13$ phần dư khởi tạo bằng $0$; nếu bỏ hẳn $13$ phần dư đó thì $p=0.71$ cho phần dư và $p=0.11$ cho phần dư bình phương, nên dấu hiệu ở ranh giới phần nào đến từ cách xử lý phần dư khởi tạo. Ở NI và DP, cách xử lý này hầu như không đổi kết quả. RMSE là $43.2$ so với $77.0$ của naive mùa vụ.

**DP.** Lấy log, $d=1$ và $D=0$; sai phân thêm một lần đưa $\hat\rho(1)$ từ $-0.004$ xuống $-0.490$, tức sai phân thừa, nên $d=1$ là đủ. Kiểm định trên phần dư cho $p=0.40$ nhưng phần dư bình phương còn tự tương quan rất mạnh với $p=5.6\times10^{-8}$, nên khoảng dự báo dựa trên phương sai không đổi chưa đáng tin và cần mô hình cho phương sai như GARCH. RMSE một bước của trôi là $1.435%$, chỉ ngang naive với $1.439%$.

### Nhận xét

*Đồng bộ từ báo cáo, mục 6.3(d).*

Kết quả **phù hợp** với lý thuyết: cả tám phần thực hiện được cho cả ba bộ dữ liệu và cho một bức tranh nhất quán, đồng thời chỉ ra đúng những điểm cần thận trọng: phần dư bình phương của AP ở ranh giới, tùy cách xử lý phần dư khởi tạo, và cụm biến động còn lại của DP. Ở NI và DP, RMSE của các phương pháp gần nhau trong các phép so ở hàng cuối bảng. Ở AP, airline có RMSE thấp hơn rõ rệt so với naive mùa vụ.

## Bảng tổng hợp: ID × bộ dữ liệu → phù hợp lý thuyết?

Nguyên liệu cho Mục 4.6 của báo cáo (Bài 12). Cột cuối lấy từ Bảng 7.1 của báo cáo (P: phù hợp, M: phù hợp một phần, K: không phù hợp).

| ID | Nội dung | Bộ | Phù hợp lý thuyết? |
|---|---|---|---|
| 12.1 | Xu thế tất định và xu thế ngẫu nhiên; I(d) | NI, AP, DP | NI: M, AP: M, DP: P (mục 5.1(a)) |
| 12.2 | Bước ngẫu nhiên | NI, AP, DP | NI: K, AP: K, DP: P (mục 5.1(b)) |
| 12.3 | Sai phân sửa được gì và làm hỏng gì: sai phân quá mức | NI, AP, DP | NI: P, AP: P, DP: P (mục 5.3(a)) |
| 12.4 | Kiểm định nghiệm đơn vị: ADF và KPSS | NI, AP, DP | NI: P, AP: M, DP: P (mục 5.3(b)) |
| 12.5 | Mô hình ARIMA(p, d, q) và độ trôi | NI, AP, DP | NI: P, AP: M, DP: P (mục 5.2(a)) |
| 12.6 | Ảnh hưởng vĩnh viễn của cú sốc: trọng số Ψ | NI, AP, DP | NI: P, AP: P, DP: P (mục 6.1(a)) |
| 12.7 | ARIMA(0,1,1) ≡ làm trơn mũ đơn ≡ mô hình mức cục bộ | NI, AP, DP | NI: P, AP: M, DP: P (mục 6.1(b)) |
| 12.8 | Dự báo h bước tối ưu MSE, phương sai sai số và khoảng dự báo | NI, AP, DP | NI: P, AP: P, DP: P (mục 6.2(a)) |
| 12.9 | Dự báo trên thang log: trung vị, trung bình, khoảng chuyển đổi | NI, AP, DP | NI: P, AP: P, DP: P (mục 6.2(b)) |
| 12.10 | Sai phân mùa vụ, SARIMA và mô hình airline | NI, AP, DP | NI: M, AP: P, DP: M (mục 5.2(b)) |
| 12.11 | Mốc chuẩn và thước đo độ chính xác | NI, AP, DP | NI: P, AP: P, DP: P (mục 6.3(a)) |
| 12.12 | Thẩm định trên gốc cuốn chiếu và vai trò của bước chọn mô hình | NI, AP, DP | NI: P, AP: P, DP: P (mục 6.3(b)) |
| 12.13 | Hàm mất mát xác định phiếm hàm; kiểm định Diebold–Mariano | NI, AP, DP | NI: P, AP: P, DP: P (mục 6.3(c)) |
| 12.14 | Checklist báo cáo dự báo ARIMA | NI, AP, DP | NI: P, AP: P, DP: P (mục 6.3(d)) |
| ⋆ | Mở rộng (tùy chọn): hồi quy giả tạo và đồng tích hợp log DAX ~ log SMI | DAX và SMI (phần mở rộng, không thuộc 6 bộ chính) | phù hợp với cảnh báo của lý thuyết: $R^2=0.965$ nhưng Engle–Granger $p=0.28$ (mục 5.1(b)) |